<img src="img/copertina.png" width="100%">

---


## 1 — Il Problema

Nei sistemi di sospensione veicolare tradizionali (passivi), la progettazione dei componenti meccanici (molle e ammortizzatori) richiede un **trade-off continuo tra due obiettivi fisici in contrasto**:

* **Comfort di marcia (Ride Comfort):** Isolamento della cabina dalle irregolarità del manto stradale per minimizzare le accelerazioni verticali percepite dai passeggeri ($\ddot{z}_s(t)$).
* **Tenuta di strada (Road Holding):** Mantenimento del contatto costante tra pneumatico e asfalto per garantire stabilità e sicurezza di guida, legato alle variazioni di carico dinamico percepito dal gruppo ruota ($z_u(t) - z_r(t)$) che dal suo punto di vista vede una forzante di eccitazione di tipo spostamento provocata dal profilo e dalle asperità del manto stradale.

I sistemi di **Sospensione Attiva** consentono di superare questo compromesso applicando forze variabili tramite attuatori idraulici o elettrici, ma introducono un elevato consumo energetico e complessità hardware. D'altra parte, i sistemi **Semi-Attivi** regolano unicamente lo smorzamento (es. ammortizzatori magnetorheologici), risultando energeticamente efficienti ma vincolati da limiti di dissipazione passiva.

**La Sfida del Machine Learning:**  
La sintesi di un controllo ottimo in tempo reale per un'**architettura ibrida (Attivo/Semi-Attivo)** richiede la risoluzione di problemi di ottimizzazione vincolata complessi e non lineari. Approcci tradizionali come LQR (Kalman, 1960) / MPC (Richalet et al., 1978; Cutler e Ramaker, 1980) presentano limitazioni o elevati costi computazionali a bordo veicolo, oltre al fatto che necessitano di un numero di sensori non trascurabili (e anche se si costruissero osservatori alla Luenberger per osservare grandezze non misurate direttamente, non è nemmeno detto che il sistema sia completamente osservabile per la struttura matematica intrinseca, ed utilizzare filtri di Kalman non è detto che possa aiutare nella dinamica se il modello e i dati sono malposti).  

L'obiettivo di questo progetto è quindi quello di utilizzare un approccio ML con **Reti Neurali (Learned Optimal Control)** per approssimare la politica di controllo ottimale del sistema ibrido attivo/semi-attivo, garantendo un'esecuzione real-time ad alte prestazioni utilizzando pochi sensori di bordo e valutando nell'insieme anche policy rigenerative che possono permettere di recuperare parte dell'energia per alimentare autonomamente il sistema di sospensioni durante i tragitti.


--- 



### 1.1 Genesi del Progetto e Limitazioni del Primo Approccio (1 GDL Quarter-Car semplificato)
Nelle prime fasi dello studio, si è valutata una formulazione semplificata del modello fisico dell'auto basata su un **sistema a un singolo grado di libertà (1 GDL)** e si era proceduto a costruire tutta l'interfaccia neurale con questo sistema.  

L'idea fondamentale prevedeva l'imposizione del fattore di smorzamento $\xi$ variabile in funzione della sola velocità di traslazione del veicolo in direzione orizzontale $v_x(t)$, tale che $\xi =f(v_x(t))$, accoppiato a un controllo puramente proporzionale $k_p$ per la forza di attuazione $F_a$ tale che $F_a \propto k_p \dot{z}_s(t)$, cioè ad una velocità verticale dell'auto (come vedremo l'attuatore deve in qualche modo entrare nella dinamica insieme allo smorzatore che di fatto lavora esclusivamente in velocità e non in accelerazione).  

Costruiamo quindi un modello riconducibile ad oscillatore smorzato semplice che si muove verticalmente (in direzione definita $z$) con l'aggiunta di un'ulteriore forza verticale premente sulla massa (che rappresenta la cassa del quarto del veicolo dove vi sono i passegeri) data dall'attuatore $F_{a}(t)$ ed una forza di eccitazione di tipo spostamento: essa non eccita direttamente la massa veicolare, bensì viene applicata all'estremo più basso del sistema sospensioni, situato ad una certa quota $z$ rispetto la posizione della massa.


---



#### Dinamica del Sistema e Bilancio delle Forze
Il bilancio delle forze per la massa sospesa $m_s$ tiene conto dell'elemento elastico passivo, dello smorzamento semi-attivo variabile $c(t)$ e della forza attiva dell'attuatore $F_a(t)$.  
Considerando un riferimento nullo per la velocità assoluta della massa tale che $\dot z_{rif}(t)=0$ (ovvero avendo come obbiettivo che la massa sia ferma), e considerando un controllo puramente proporzionale, l'attuatore opera in logica *Skyhook* (Karnopp, Crosby e Harwood, 1974) con guadagno $k_p$:

$$\begin{aligned}
F_a(t) &= k_p \dot e(t)=k_p(\dot z_{rif}(t)-\dot z_s(t))= -k_p \dot{z}_s(t) \\[8pt]
m_s \ddot{z}_s(t) &= -k \big(z_s(t) - z_u(t)\big) - c(t) \big(\dot{z}_s(t) - \dot{z}_u(t)\big) + F_a(t) \\[10pt]
m_s \ddot{z}_s(t) &+ \big(c(t) + k_p\big) \dot{z}_s(t) + k z_s(t) = c(t) \dot{z}_u(t) + k z_u(t)
\end{aligned}$$

---



#### Parametrizzazione e Forma Normalizzata
Per generalizzare il comportamento del sistema, si divide l'equazione per la massa $m_s$ e si introducono i parametri canonici di pulsazione propria $\omega_n$ e smorzamento adimensionale $\xi(t)$:

$$\begin{aligned}
\omega_n &= \sqrt{\frac{k}{m_s}} \qquad \implies \qquad \omega_n^2 = \frac{k}{m_s} \\[10pt]
\xi(t) &= \frac{c(t)}{2 m_s \omega_n} \qquad \implies \qquad \frac{c(t)}{m_s} = 2 \xi(t) \omega_n
\end{aligned}$$

Sostituendo queste definizioni nell'equazione di bilancio, si perviene alla **forma finale normalizzata**, che evidenzia il contributo dinamico dell'attuatore attivo al primo membro:

$$\ddot{z}_s(t) + \left( 2 \xi(t) \omega_n + \frac{k_p}{m_s} \right) \dot{z}_s(t) + \omega_n^2 z_s(t) = 2 \xi(t) \omega_n \dot{z}_u(t) + \omega_n^2 z_u(t)$$

Inoltre, definendo il guadagno dell'attuatore in termini di smorzamento attivo adimensionale equivalente $\xi_p$:

$$\xi_p = \frac{k_p}{2 m_s \omega_n} \qquad \implies \qquad \frac{k_p}{m_s} = 2 \xi_p \omega_n$$

L'equazione del moto assume la sua espressione più compatta, in cui la componente semi-attiva ($\xi$) e quella attiva equivalente ($\xi_p$) risultano sommate direttamente:

$$\ddot{z}_s(t) + 2 \big(\xi(t) + \xi_p\big) \omega_n \dot{z}_s(t) + \omega_n^2 z_s(t) = 2 \xi(t) \omega_n \dot{z}_u(t) + \omega_n^2 z_u(t)$$


Il termine a secondo membro $2 \xi(t) \omega_n \dot{z}_u(t) + \omega_n^2 z_u(t)$ è la forzante esterna di tipo spostamento e rappresenta l'effetto del manto stradale che inietta nel sistema delle sospensioni l'eccitazione alla base del sistema stesso.

Va precisato che il termine con $\dot{z}_u(t)$ non costituisce un'eccitazione indipendente di tipo velocità: è semplicemente la derivata temporale dello stesso ingresso cinematico $z_u(t)$, ed entra nell'equazione con quel coefficiente perché l'accoppiamento smorzatore-massa è per natura sensibile alla velocità relativa (termine dissipativo), mentre l'accoppiamento molla-massa è sensibile alla posizione relativa (termine elastico): sono due canali distinti attraverso cui **la stessa eccitazione cinematica** entra nel sistema, non due eccitazioni fisicamente separate. Per questo, nei testi classici di vibrazioni meccaniche (Den Hartog, 1956; Rao, 2011), nel capitolo dedicato alla *support/base motion excitation*, l'intero secondo membro viene chiamato collettivamente "eccitazione di spostamento" (o *cinematica*): il criterio di classificazione è la natura fisica della grandezza prescritta dall'esterno — una posizione — non la derivata che compare letteralmente in ciascun termine dell'equazione.

Un'eccitazione di **tipo velocità**, per contrasto, si avrebbe se fosse $\dot{z}_u(t)$ stesso — e non $z_u(t)$ — la grandezza imposta indipendentemente dall'esterno (per esempio in sistemi dove si impone direttamente una velocità di trascinamento alla base, come un nastro trasportatore o un attuatore idraulico controllato in velocità): in quel caso il termine con $z_u(t)$ nell'equazione non sarebbe più disponibile come tale (andrebbe ottenuto per integrazione, introducendo un grado di libertà o una condizione iniziale aggiuntiva), e sarebbe la velocità, non lo spostamento, la grandezza da cui tutto il resto discende per derivazione o integrazione.


---



#### Leggi di Controllo e Schedulazione Morbida
Come termine di paragone e baseline iniziale si è utilizzata la **regola della trasmissibilità per frequenze $\frac{\omega}{\omega_n} > \sqrt{2}$**, la quale stabilisce il confine oltre cui uno smorzamento elevato peggiora l'isolamento del veicolo, dove con $\omega$ e $\omega_n$ si identificano rispettivamente la pulsazione della forzante di tipo spostamento che eccita la base del modello quarter-car (dipende dal profilo stradale, aka dalla geometria spaziale della strada) e la pulsazione naturale del sistema massa-molla-smorzatore, che è univocamente determinata dalle sole proprietà dell'auto.  

Di base quindi, l'attuatore ha l'incarico di esercitare una funzione di buffering. Va notato che questo è **opposto** alla prassi automotive comune, dove gli ammortizzatori a smorzamento variabile in commercio tendono a *irrigidirsi* alle alte velocità per massimizzare la tenuta di strada: qui invece la regola della trasmissibilità impone di *ammorbidire* lo smorzamento passivo oltre r = √2 e di scaricare il compito di tenuta sull'attuatore attivo, perché è quest'ultimo — e non il damper — a dover fornire il contributo che stabilizza la ruota in quel regime, fornendo una quota parte di forza al sistema.

Per un sistema a 1 GDL eccitato dalla base, dopo essere passati nel dominio della frequenza ed aver creato la risposta in frequenza $Tr(ir) = \frac{\text{spostamento}_{\text{verticale}}(ir)}{\text{forzante}_{\text{spostamento}}(ir)}$, ovvero il rapporto tra lo spostamento verticale della massa auto e la forzante di tipo spostamento derivata dalla strada, tutte le curve di trasmissibilità — qualunque sia lo smorzamento, detto $r = \frac{\omega}{\omega_n}$ — si incrociano in $r = \sqrt{2}$ (risultato classico della teoria dell'isolamento vibrazionale, si veda Den Hartog, 1956; Rao, 2011).

Allora tutte le famiglie di funzioni parametriche al variare di $\xi(v(t))=\xi(t)$ valgono esattamente:

$$ \left| \frac{\text{spostamento}_{\text{verticale}}(ir)}{\text{forzante}_{\text{spostamento}}(ir)} \right|_{r = \sqrt{2}} = 1 $$

Ovvero, sull'asse delle frequenze normalizzate $r$, a sinistra di quel punto ($\frac{\omega}{\omega_n} < \sqrt{2}$) aumentare lo smorzamento **riduce** il modulo della trasmissibilità, mentre alla sua destra invece **aumenta** ($\frac{\omega}{\omega_n} > \sqrt{2}$).  
Questo significa semplicemente che se la vettura viaggia in direzione $x$ a bassa velocità traslazionale, lo smorzamento deve aumentare per diminuire l'ampiezza delle oscillazioni trasmesse al comparto passegeri mentre se l'auto viaggia ad alta velocità , lo smorzamento deve diminuire in modo tale che il comparto passegeri si trovi in frequenza nello spot detto di *isolamento frequenziale*: l'ampiezza delle oscillazioni trasmesse diminuisce al diminuire di $\xi$.  

Considerando che lo smorzamento e la forza dell'attuatore sono parametri di progetto fisicamente finiti — per cui il sistema può richiedere un intervento dinamico solo fino alla saturazione dei rispettivi limiti fisici —, la legge di controllo e la relativa schedulazione destinate all'apprendimento della rete neurale sono state modellate analiticamente come segue:


Detto $r$ il rapporto delle frequenze e definita $s$ la variabile funzionale di smussamento:

$$\begin{aligned}
r &= \frac{\omega}{\omega_n} \\[8pt]
s &= f(r) = k \cdot \left( \frac{r}{\sqrt{2}} - 1 \right) \qquad \left( \text{detta } k \text{ la pendenza attorno a } |\text{Tr}(i\omega)| = 1, \text{ abbiamo scelto } k=4 \right)
\end{aligned}$$

$$\begin{cases}
\xi(r) = \xi_{\min} + \dfrac{\xi_{\max} - \xi_{\min}}{1 + e^{+s}} & \text{(scende con } r\text{)} \\[12pt]
k_p(r) = k_{p,\text{basso}} + \dfrac{k_{p,\text{alto}} - k_{p,\text{basso}}}{1 + e^{-s}} & \text{(sale con } r\text{)}
\end{cases}$$


L'architettura iniziale dunque seguiva una **legge duale** guidata dalla teoria classica della trasmissibilità nel **dominio della frequenza**:

* **A basse frequenze (basse velocità):** Smorzamento $\xi$ elevato per contenere le ampiezze di oscillazione e forza di attuazione $F_a$ ridotta.
* **Ad alte frequenze (alte velocità):** Smorzamento $\xi$ ridotto per isolare il telaio dalle vibrazioni e forza di attuazione $F_a$ elevata per compensare la perdita di stabilità.

Nello specifico, si è adottata una legge di transizione morbida (sigmoidale) per modellare correttamente le non linearità reali del sistema e garantire una risposta dinamica continua. L'impiego di leggi di commutazione lineari o a gradino risulterebbe infatti fisicamente inconsistente, in quanto introdurrebbe discontinuità irrealistiche nell'erogazione di forza e nella variazione dello smorzamento (che comporterebbe dinamiche impulsive sia nella dinamica sia nella cinematica del gruppo sospensione, peggiorando per assurdo le prestazioni e il comfort).

La legge accoppiata è sigmoidale simmetrica dove le due sigmoidi sono complementari: dove il damper viene reso morbido per isolare dinamicamente, l'attuatore prende il suo posto per tenere ferma la cassa; gli estremi di $\xi$ non sono scelti né dati a caso, ma sono i limiti fisici del damper ottenibili usando coefficienti di smorzamento massimo e minimo attuabile da progetto (nelle sezioni successive verranno mostrati i dati).

Usando dataset di registrazioni stradali reali, la frequenza della forzante si stima **dalla velocità** di traslazione orizzontale dell'automobile, $\omega = \frac{2\pi v}{\lambda_c}$ dove $\lambda_c = f(x(t))$ ($\lambda_c = 6.75\text{ m}$ nei casi stimati) è la lunghezza d'onda del profilo stradale media che si era calcolata a partire da dati grezzi, cioè $r = \sqrt{2} \cdot \frac{v}{v_{\text{cross}}}$ con $v_{\text{cross}} = 7.14\text{ m/s} \approx 26\text{ km/h}$ stimate (erroneamente, come vedremo) usando una FFT sul moto di traslazione orizzontale.  

Si deduce quindi che non si è usata $a_z(t)$ accelerazione verticale per ricostruire il profilo stradale dal moto orizzontale (di base il moto globale dell'auto può essere visto come un moto composto o una traiettoria curva piana $\gamma (t)=(x(t),y(t))$ con $y(t)=f(x(t))$ tale che $\gamma (t)=(x(t),y(x(t)))$), perché lo spettro di $a_z(t)$, nel dominio delle frequenze, è cosiddetto *vehicle-shaped*: esso (lo spettro) è dominato in modulo dalle frequenze naturali del veicolo che nascono dalla sua geometria e dalle sue proprietà meccaniche intrinseche; conoscendo massa, rigidezza e smorzamento dell'automobile si calcolano le pulsazioni proprie ($1.5\text{ Hz}$ e $11.9\text{ Hz}$) quindi e non dalla strada.  

Deconvolvere la funzione di trasferimento di $a_z(t)$ a partire da una finestra temporale della durata di un viaggio si è rivelato mal condizionato e per strade ISO restituisce comunque l'informazione già contenuta in $v_x(t)$ (velocità orizzontale dell'auto) sulla pulsazione della frequenza dominante della forzante di tipo spostamento (cioè $\omega$).


---

### 1.2 Reti Neurali ed implementazioni ML  

**Disclaimer:**  
Abbiamo scelto di utlizzare le librerie di PyTorch perché è moderatamente semplice da usare per implementare e risolvere il problema proposto in Python e gestisce in modo naturale anche processi complicati (come il ri-addestramento continuo richiesto dalla teoria DAgger che useremo in futuro). Può contare su un'enorme community e su una notevole documentazione, che permettono quindi di trovare supporto o hints per costruire le funzioni necessarie.  



#### 1.2.1 **Rete CNN a due teste - 1GDL**
Inizialmente, dato che la vettura richiede due leve di controllo differenti, si era pensato di costruire un'unica rete di tipo CNN con due teste (multi-head output) la quale permette di estrarre le feature della strada tramite $a_z(t)$ accelerazione verticale dell'auto e della velocità di traslazione orizzontale $v_x(t)$ una sola volta, facendo apprendere alla rete una rappresentazione condivisa per coordinare smorzatore e attuatore nello stesso istante.  

Sebbene la rete convoluzionale 1D (vedi la definizione di `SuspensionCNN1D`) sia stata impiegata nel codice ed evidenzi un livello di adattamento apparentemente elevato nei test, dal punto di vista architetturale e teorico nel machine learning applicato ai sistemi dinamici non è la scelta ottimale, per diverse ragioni strutturali:  

1. Le CNN elaborano le finestre temporali come blocchi statici ed indipendenti. Nei sistemi dinamici e nelle equazioni differenziali (come il modello a 1 e 2 GDL controllato da RK4), lo stato al tempo $t$ dipende direttamente dallo stato accumulato al tempo $t-1$ (posizioni, velocità della cassa e della ruota).  
Una CNN non possiede una "memoria interna" o uno stato nascosto persistente: ogni finestra viene valutata "da zero", ignorando la continuità dello stato fisico reale tra un frame e l'altro. 

2. I filtri convoluzionali standard (come quelli usati da pytorch per il modulo `nn.Conv1d`) guardano l'intera finestra di input simultaneamente (sia il passato che il futuro *interno alla finestra*, non il futuro reale: la finestra \[t-seq\_len, t\] resta comunque fatta di soli campioni già acquisiti). Il problema non è quindi una fuga di dati futuri, ma la mancanza di struttura causale rigorosa: ogni posizione interna alla finestra viene elaborata guardando anche i campioni successivi ad essa (non solo quelli precedenti), il che è incoerente con un controllore che in produzione dovrebbe poter aggiornare la stima un campione alla volta man mano che arrivano i dati dai sensori, non ricalcolando l'intera finestra da zero ad ogni passo.  

3. La CNN ha un campo ricettivo limitato e rigido determinato dalla dimensione dei kernel e dai layer di pooling (nel nostro codice seq_len = 100 campioni, pari a 1 secondo ricavato direttamente dalla frequenza di campionamento disponibile come dato nella descrizione del dataset).   
Se il veicolo viaggia a velocità molto basse, la frequenza della strada cala e 1 secondo può non essere sufficiente per catturare un intero ciclo di oscillazione. Al contrario, ad alte velocità, 1 secondo contiene fin troppi cicli, rendendo la dimensione fissa della finestra inefficiente e rigida.   

***Vediamo ora il codice proposto per risolvere il problema iniziale ad 1 GDL.*** 
 
---



Caricamento degli header e delle librerie necessarie, inoltre subito dopo gli import, troviamo l'impostazione del seed che serve a garantire la riproducibilità dell'esperimento:  

* **Risultati identici:** Le reti neurali inizializzano i pesi e mescolano i dati usando generatori di numeri pseudocasuali. Fissare il seed a un numero (es. 42) fa sì che Python, NumPy e PyTorch generino sempre la stessa identica sequenza di numeri ogni volta che si esegue il codice.  

* **Confronti scientifici reali:** Se modifichiamo l'architettura della rete e la loss migliora, si ha la certezza che sia merito delle modifiche ai parametri delle reti (ad esempio learning rate, weight loss ecc ecc.) e non di un'inizializzazione "fortunata" dei pesi casuali.


In [ ]:
import os
import glob
import pickle
import time
import warnings
import kagglehub
from dataclasses import dataclass

import numpy as np
import scipy.signal as signal
from sklearn.metrics import r2_score, mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

---

Costruzioni delle classi, dei parametri fisici dell'auto e della legge che governa la trasmissibilità

In [ ]:
@dataclass
class QuarterCar2DOF:
    """Parametri sospensione anteriore Peugeot 207 (classe ISO B/C)."""
    m_s: float = 260.0        # Massa sospesa (cassa)          [kg]
    m_u: float = 38.0         # Massa non sospesa (ruota)      [kg]
    k_s: float = 23000.0      # Rigidezza molla                [N/m]
    k_t: float = 190000.0     # Rigidezza pneumatico           [N/m]
    c_min: float = 800.0      # Smorzamento min (damper)       [Ns/m]
    c_max: float = 3200.0     # Smorzamento max (damper)       [Ns/m]
    c_nom: float = 1500.0     # Smorzamento passivo (baseline) [Ns/m]

    @property
    def wn(self) -> float:
        """Pulsazione naturale della cassa: sqrt(k_s/m_s)."""
        return np.sqrt(self.k_s / self.m_s)

    @property
    def c_crit(self) -> float:
        """Smorzamento critico della cassa: 2*sqrt(k_s*m_s) = 2*m_s*wn."""
        return 2.0 * np.sqrt(self.k_s * self.m_s)

    def c_to_xi(self, c):
        return c / self.c_crit

    def xi_to_c(self, xi):
        return xi * self.c_crit


@dataclass
class Config:
    fs: float = 100.0         # Frequenza di controllo/campionamento  [Hz]
    dt: float = 0.01          # Passo temporale                        [s]
    n_sub: int = 2            # Sotto-passi RK4 per stabilità
    seq_len: int = 100        # Finestra di memoria per la CNN (1.0 s > periodo cassa 0.67 s)
    freq_win: int = 100       # Finestra per la stima di ω istantanea (1.0 s)

    F_max: float = 1500.0     # Saturazione forza attuatore   [N]  (NO forza infinita)
    road_rms: float = 0.010   # RMS strada nominale prima della calibrazione [m]
    calibra_ampiezza: bool = True  # Calibra z_r per traccia -> forze in Newton fisici

    # --- Scheduling adattivo (regola √2·ωn) sul rapporto r = ω/ωn ---
    # ω è la frequenza di eccitazione stimata istante per istante (strada + velocità).
    sched_steep: float = 4.0  # Ripidità della transizione attorno a r = √2
    xi_hi: float = 0.60       # ξ per ω < √2·ωn : comfort/controllo
    xi_lo: float = 0.18       # ξ per ω > √2·ωn : isolamento
    kp_lo: float = 200.0      # Guadagno attuatore per ω < √2·ωn   [Ns/m]
    kp_hi: float = 2500.0     # Guadagno attuatore per ω > √2·ωn   [Ns/m]
    r_clip: float = 5.0       # Clip di r = ω/ωn per robustezza
    v_cross: float = 13.0     # Solo per riferimento nei grafici (ω=√2ωn ~ 47 km/h)

    w_F: float = 8.0          # Peso della loss su F_att (ξ è facile e domina)
    epochs: int = 15
    save_fig: bool = True

    # Animazione confronto CON vs SENZA intervento
    show_anim: bool = True
    anim_strada_demo: bool = True # True: strada dimostrativa con dossi (effetto ben visibile)
                                  # False: usa la traccia reale di validazione
    anim_seconds: float = 8.0     # durata simulata da animare [s]
    anim_v: float = 7.0           # velocità di avanzamento nella demo [m/s]
    anim_gain: float = 6.0        # ingrandimento verticale del movimento (resa grafica)
    anim_step: int = 2            # sotto-campionamento frame
    anim_fps: int = 24            # fps del video (più basso = scorrimento più lento)
    anim_mostra: bool = True      # apre la finestra interattiva Python (plt.show())
    anim_save_video: bool = True  # salva anche il video MP4 (richiede ffmpeg)
    anim_video_file: str = "confronto_sospensione.mp4"


CAR = QuarterCar2DOF()
CFG = Config()

XI_MIN = CAR.c_to_xi(CAR.c_min)   # ~0.164
XI_MAX = CAR.c_to_xi(CAR.c_max)   # ~0.654


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


SQRT2 = np.sqrt(2.0)


def xi_of_r(r, cfg: Config):
    """ξ alto per r=ω/ωn < √2, basso per r > √2. Transizione al crossover √2."""
    z = cfg.sched_steep * (r / SQRT2 - 1.0)
    return cfg.xi_lo + (cfg.xi_hi - cfg.xi_lo) * _sigmoid(-z)


def kp_of_r(r, cfg: Config):
    """Guadagno attuatore basso per r<√2, alto per r>√2 (buffer skyhook)."""
    z = cfg.sched_steep * (r / SQRT2 - 1.0)
    return cfg.kp_lo + (cfg.kp_hi - cfg.kp_lo) * _sigmoid(z)


def stima_omega(az, cfg: Config):
    """
    Frequenza di eccitazione istantanea ω(t) [rad/s] = frequenza media (Rice) di a_z:
        ω = RMS_finestra(a_z) / RMS_finestra(v_z),   v_z = integrale singolo di a_z.
    Per a_z = A·sin(ωt) restituisce esattamente ω. Usa UNA sola integrazione (v_z,
    passa-alto) -> nessun bias verso le basse frequenze. Scala-invariante (rapporto):
    non dipende dall'ampiezza, cresce con velocità E rugosità -> ξ adattivo alla strada.
    """
    fs = cfg.fs
    nyq = 0.5 * fs
    b, a = signal.butter(2, 0.3 / nyq, btype="highpass")
    vz = signal.filtfilt(b, a, np.cumsum(az) / fs)     # velocità (una integrazione)
    w = cfg.freq_win
    ker = np.ones(w) / w
    mrms = lambda s: np.sqrt(np.convolve(s * s, ker, mode="same") + 1e-12)
    omega = mrms(az) / (mrms(vz) + 1e-9)
    wn = np.sqrt(CAR.k_s / CAR.m_s)
    return np.clip(omega, 0.15 * wn, cfg.r_clip * wn)


---

Costruzione delle funzioni atte alla ricostruzione del profilo stradale a partire dei dati scaricati dal dataset: un DC-offset remover per rimuovere la componente continua del segnale dall'accelerazione (la gravità g a 9.81 m/s^2) che se non trattata durante l'integrazione per ottenere lo spostamento fa scoppiare il risultato dell'integrale e la funzione d'integrazione stessa

In [ ]:
def rimuovi_dc_offset(az, fs=100.0, fc=0.1):
    nyq = 0.5 * fs
    b, a = signal.butter(2, fc / nyq, btype="highpass")
    return signal.filtfilt(b, a, az)


def ricostruisci_strada(az, cfg: Config):
    """Doppia integrazione anti-windup di a_z -> profilo strada z_r(t), z_r'(t)."""
    fs = cfg.fs
    nyq = 0.5 * fs
    b, a = signal.butter(2, 0.3 / nyq, btype="highpass")
    vz = signal.filtfilt(b, a, np.cumsum(az) / fs)
    zr = signal.filtfilt(b, a, np.cumsum(vz) / fs)
    rms = np.sqrt(np.mean(zr ** 2)) + 1e-9
    zr = zr * (cfg.road_rms / rms)
    zr_dot = np.gradient(zr, cfg.dt)
    return zr.astype(float), zr_dot.astype(float)

---

Definizione del modello 1 GDL analitico e costruzione del risolutore RK4 per risolvere il sistema stesso e le funzioni di controllo definite successivamente

In [ ]:
# Stato x = [ z_s-z_u , zs' , z_u-z_r , zu' ]
def _deriv(x, c, F_att, zr_dot, car):
    ms, mu, ks, kt = car.m_s, car.m_u, car.k_s, car.k_t
    x1, x2, x3, x4 = x
    v_rel = x2 - x4
    dx1 = x2 - x4
    dx2 = (-ks * x1 - c * v_rel + F_att) / ms
    dx3 = x4 - zr_dot
    dx4 = (ks * x1 + c * v_rel - F_att - kt * x3) / mu
    return np.array([dx1, dx2, dx3, dx4])


def _rk4_step(x, c, F_att, zr0, zr1, car, h):
    zm = 0.5 * (zr0 + zr1)
    k1 = _deriv(x,            c, F_att, zr0, car)
    k2 = _deriv(x + 0.5*h*k1, c, F_att, zm,  car)
    k3 = _deriv(x + 0.5*h*k2, c, F_att, zm,  car)
    k4 = _deriv(x + h*k3,     c, F_att, zr1, car)
    return x + (h / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

---

**Generatore dei Target Fisici** (`PhysicsAdaptiveTargetGenerator` / Ground Truth)

Funge da insegnante fisico del sistema: applica le equazioni della dinamica del veicolo per calcolare esattamente quali dovrebbero essere lo smorzamento ($\xi$) e la forza dell'attuatore ($F_{\text{att}}$) ideali istante per istante, creando i dati di riferimento su cui la rete neurale verrà poi addestrata.  

1. **Calibrazione dell'ampiezza strada (`_passive_rms` e scala)**
   - La funzione `_passive_rms` simula una sospensione passiva tradizionale (con smorzamento fisso $c_{\text{nom}}$) e ne calcola l'accelerazione media (RMS).
   - Il generatore confronta l'accelerazione reale misurata dai sensori con quella del modello e scala il profilo della strada $z_r$. Questo passaggio serve a garantire che la forza generata dall'attuatore sia espressa in Newton fisici reali.

2. **Calcolo dei parametri adattivi (Regola della $\sqrt{2}$)**
   - Stima la frequenza di sollecitazione istantanea della strada $\omega(t)$ e il rapporto $r = \omega / \omega_n$.
   - In base a $r$, calcola lo smorzamento ottimale $\xi(t)$ e il guadagno dell'attuatore $k_p(t)$ secondo la regola della trasmissibilità.

3. **Simulazione dinamica parallela (Integratore RK4)**
   - Nel ciclo `for`, integra passo-passo le equazioni differenziali del moto facendo avanzare due simulazioni in parallelo:
     - **Ramo Controllato ($x$)**: Applica lo smorzamento variabile $c(t)$ e la forza dell'attuatore Skyhook $F_{\text{att}} = -k_p \cdot \dot{z}_s$ (limitata dalla saturazione $\pm F_{\text{max}}$).
     - **Ramo Passivo ($xp$)**: Mantiene lo smorzamento fisso $c_{\text{nom}}$ senza attuatore, usato come *baseline* di confronto per valutare il miglioramento del comfort.

4. **Output dei dati**
   - Restituisce un dizionario contenente i vettori target `xi` e `F` (i valori che la CNN dovrà imparare a prevedere), oltre alle accelerazioni della cassa risultanti (`az_opt` e `az_pas`) e al profilo strada ricostruito.


In [ ]:
def _passive_rms(zr_dot, car, cfg):
    """RMS dell'accelerazione cassa del modello PASSIVO (c_nom) -> per calibrazione."""
    N = len(zr_dot)
    xp = np.zeros(4)
    h = cfg.dt / cfg.n_sub
    acc = 0.0
    for k in range(N):
        acc += (-car.k_s * xp[0] - car.c_nom * (xp[1] - xp[3])) ** 2
        zr0 = zr_dot[k]
        zr1 = zr_dot[k + 1] if k + 1 < N else zr_dot[k]
        for _ in range(cfg.n_sub):
            xp = _rk4_step(xp, car.c_nom, 0.0, zr0, zr1, car, h)
    return np.sqrt(acc / N) / car.m_s


class PhysicsAdaptiveTargetGenerator:
    def __init__(self, car: QuarterCar2DOF, cfg: Config):
        self.car = car
        self.cfg = cfg

    def generate(self, az_clean, v_interp):
        cfg, car = self.cfg, self.car
        N = len(az_clean)
        zr, zr_dot = ricostruisci_strada(az_clean, cfg)

        # --- Calibrazione ampiezza strada: il modello passivo deve riprodurre
        #     l'RMS dell'a_z MISURATO -> forze attuatore in Newton fisici ---
        if cfg.calibra_ampiezza:
            rms_meas = np.sqrt(np.mean(az_clean ** 2))
            rms_pas0 = _passive_rms(zr_dot, car, cfg)
            factor = np.clip(rms_meas / (rms_pas0 + 1e-9), 0.2, 20.0)
            zr = zr * factor
            zr_dot = zr_dot * factor

        # --- Frequenza di eccitazione istantanea -> r = ω/ωn (strada + velocità) ---
        omega = stima_omega(az_clean, cfg)       # da a_z (scala-invariante)
        r_arr = omega / car.wn
        xi_arr = xi_of_r(r_arr, cfg)
        kp_arr = kp_of_r(r_arr, cfg)
        c_arr = car.xi_to_c(xi_arr)

        xi_t = xi_arr.astype(np.float32)
        kp_t = kp_arr.astype(np.float32)
        F_t = np.zeros(N, dtype=np.float32)
        az_opt = np.zeros(N, dtype=np.float32)
        az_pas = np.zeros(N, dtype=np.float32)

        x = np.zeros(4)      # ramo controllato
        xp = np.zeros(4)     # ramo passivo
        h = cfg.dt / cfg.n_sub

        for k in range(N):
            c = c_arr[k]
            kp = kp_arr[k]
            zs_dot = x[1]                        # velocità assoluta cassa
            v_rel = x[1] - x[3]
            F_att = np.clip(-kp * zs_dot, -cfg.F_max, cfg.F_max)   # skyhook, forza finita

            F_t[k] = F_att
            az_opt[k] = (-car.k_s * x[0] - c * v_rel + F_att) / car.m_s
            az_pas[k] = (-car.k_s * xp[0] - car.c_nom * (xp[1] - xp[3])) / car.m_s

            zr0 = zr_dot[k]
            zr1 = zr_dot[k + 1] if k + 1 < N else zr_dot[k]
            for _ in range(cfg.n_sub):
                x = _rk4_step(x, c, F_att, zr0, zr1, car, h)
                xp = _rk4_step(xp, car.c_nom, 0.0, zr0, zr1, car, h)

        return dict(xi=xi_t, F=F_t, kp=kp_t, r=r_arr.astype(np.float32),
                    az_opt=az_opt, az_pas=az_pas, zr=zr, N=N)

---

**Caricamento Dataset e Ricostruzione Target (RK4)**

1. **`carica_tracce()`**
   - **Download e Sincronizzazione**: Recupera i dati reali dal dataset Kaggle (`road-quality-dataset`), rimuove il bias di gravità da $a_z$ e interpola la velocità GPS $v$ sulla frequenza dell'IMU.
   - **Split Dati**: Mantiene 2 tracce per l'addestramento e riserva la 3ª traccia (mai vista) per la validazione finale.
   - **Fallback Sintetico**: Se Kaggle non è disponibile, genera in automatico 3 tracce sintetiche a velocità differenti (8 m/s, 15 m/s, 26 m/s).

2. **`costruisci_finestre()`**
   - **Target Fisici**: Esegue la simulazione RK4 su ogni traccia per calcolare i valori ottimali di $\xi$, $F_{\text{att}}$ e del rapporto $r$.
   - **Sliding Window**: Taglia l'accelerazione $a_z$ in vettori di 100 campioni (1,0 s) per l'input della CNN.
   - **Filtro Velocità**: Scarta i punti sotto i 2,0 m/s per evitare il rumore di calcolo a veicolo fermo.


In [ ]:
def carica_tracce():
    try:
        import kagglehub
        cartella = kagglehub.dataset_download("nickkotarelas/road-quality-dataset")
        files = sorted(glob.glob(os.path.join(cartella, "**", "*.pkl"), recursive=True))
    except Exception:
        files = []

    tr, va = [], []

    if len(files) >= 3:
        files_tr, files_va = files[:-1], [files[-1]]
        print(f"  [ok] {len(files)} tracce reali -> {len(files_tr)} train / 1 validazione ({os.path.basename(files_va[0])})")
        for grp, dst in ((files_tr, tr), (files_va, va)):
            for pkl in grp:
                with open(pkl, "rb") as f:
                    d = pickle.load(f)
                imu = d["imu"]
                t_imu = np.array(imu["time"]["rel"], dtype=float)
                medie = {kk: abs(np.mean(vv)) for kk, vv in imu["accel"].items()}
                az = rimuovi_dc_offset(np.array(imu["accel"][max(medie, key=medie.get)], dtype=float))
                t_gps = np.array(d["gps"]["time"]["rel"], dtype=float)
                v_gps = np.array(d["gps"]["speed"], dtype=float)
                v = np.interp(t_imu, t_gps, v_gps)
                dst.append((az, v))
    else:
        print("  [!] Dataset reale non disponibile: genero 3 tracce sintetiche distinte (2 train / 1 val).")
        for i in range(3):
            rng = np.random.default_rng(SEED + i)
            N = 9000
            t = np.linspace(0, 90, N)
            base_v = [8.0, 15.0, 26.0][i]     # tracce a regimi di velocità diversi
            v = np.clip(base_v + 7.0 * np.sin(2*np.pi*(0.01 + 0.004*i)*t) + rng.normal(0, 0.2, N), 3.0, 33.0)
            f_road = 0.6 + 0.9 * v / 10.0     # freq. strada cresce con la velocità
            az = np.sin(2*np.pi*f_road*t) * (0.7 + 0.05*v) + rng.normal(0, 0.15, N)
            (tr if i < 2 else va).append((az, v))

    return tr, va


def costruisci_finestre(tracce, gen, cfg: Config):
    X_az, X_v, Y_xi, Y_F, X_r = [], [], [], [], []
    comfort_stats = []
    for az, v in tracce:
        out = gen.generate(az, v)
        comfort_stats.append((np.sqrt(np.mean(out["az_pas"]**2)),
                              np.sqrt(np.mean(out["az_opt"]**2))))
        for i in range(cfg.seq_len, out["N"]):
            if v[i] < 2.0:
                continue
            X_az.append(az[i-cfg.seq_len:i])
            X_v.append(v[i])
            Y_xi.append(out["xi"][i])
            Y_F.append(out["F"][i])
            X_r.append(out["r"][i])
    return (np.array(X_az, dtype=np.float32), np.array(X_v, dtype=np.float32),
            np.array(Y_xi, dtype=np.float32), np.array(Y_F, dtype=np.float32),
            np.array(X_r, dtype=np.float32), comfort_stats)

---

**Architettura CNN 1D a Due Teste** (`SuspensionDataset` e `SuspensionCNN1D`)

- **`SuspensionDataset`**: Convertitore PyTorch che impacchetta i vettori NumPy (finestre temporali di accelerazione $X_{az}$, velocità $X_v$, target $\xi$ e target $F_{\text{att}}$) in *Tensor* formattati correttamente per l'elaborazione a batch.
- **Estrazione Caratteristiche (`self.conv`)**: Processa la sequenza temporale di accelerazione $a_z$ tramite due strati di convoluzione 1D, *Batch Normalization*, attivazioni *ReLU* e riduzione dimensionale (*Pooling*).
- **Fusione dei Dati (`self.fc`)**: Appiattisce le caratteristiche temporali estratte e vi concatena la velocità scalare del veicolo $x_v$, passandole a uno strato *Fully Connected* con *Dropout* per prevenire l'overfitting.
- **Le Due Teste di Output**:
  - **`head_xi` (Smorzamento $\xi$)**: Usa la funzione di attivazione *Sigmoid* per vincolare rigorosamente l'output nell'intervallo fisicamente valido $[0, 1]$.
  - **`head_F` (Forza Attuatore $F_{\text{att}}$)**: Usa la *Tanh* per limitare la forza erogata nell'intervallo $[-1, 1]$, garantendo che la forza predetta non superi mai i limiti fisici dell'attuatore ($\pm F_{\text{max}}$).


**Dashboard Risultati** (`salva_figura`)

- **In alto a sinistra ($\xi$ nel tempo)**: Confronta lo smorzamento adattivo ideale (target fisico) con quello predetto dalla CNN.
- **In alto a destra ($F_{\text{att}}$ nel tempo)**: Confronta la forza dell'attuatore Skyhook target con quella predetta dalla CNN, mostrando le linee di saturazione $\pm F_{\text{max}}$.
- **In basso a sinistra (Regola della $\sqrt{2}$)**: Grafico chiave della dinamica del veicolo. Mostra la risposta del sistema al variare del rapporto di frequenza $r = \omega/\omega_n$:
  - Smorzamento alto per la risonanza ($r < \sqrt{2}$).
  - Smorzamento basso per l'isolamento acustico/vibrazionale ($r > \sqrt{2}$).
- **In basso a destra (Confronto Comfort RMS)**: Istogramma comparativo tra l'accelerazione della cassa della sospensione passiva standard e quella adattiva/attiva gestita dalla rete, calcolando la percentuale di abbattimento delle vibrazioni.

---

In [ ]:
class SuspensionDataset(Dataset):
    def __init__(self, X_az, X_v, Y_xi, Y_F):
        self.X_az = torch.tensor(X_az, dtype=torch.float32).unsqueeze(1)
        self.X_v = torch.tensor(X_v, dtype=torch.float32).unsqueeze(1)
        self.Y_xi = torch.tensor(Y_xi, dtype=torch.float32).unsqueeze(1)
        self.Y_F = torch.tensor(Y_F, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.Y_xi)

    def __getitem__(self, i):
        return self.X_az[i], self.X_v[i], self.Y_xi[i], self.Y_F[i]


class SuspensionCNN1D(nn.Module):
    """
    Input : [ finestra a_z (seq_len campioni, 1 s) , velocità v ]
    Output: [ xi_norm in (0,1) via Sigmoid ,  F_norm in (-1,1) via Tanh ]
      xi   -> [xi_min, xi_max]
      F_att-> [-F_max, F_max]   (forza intrinsecamente FINITA anche in inferenza)
    """
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(8),
        )
        self.fc = nn.Sequential(nn.Linear(64*8 + 1, 128), nn.ReLU(), nn.Dropout(0.2))
        self.head_xi = nn.Sequential(nn.Linear(128, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
        self.head_F = nn.Sequential(nn.Linear(128, 32), nn.ReLU(), nn.Linear(32, 1), nn.Tanh())

    def forward(self, x_az, x_v):
        feat = self.conv(x_az).view(x_az.size(0), -1)
        emb = self.fc(torch.cat((feat, x_v), dim=1))
        return self.head_xi(emb), self.head_F(emb)

---

**Simulazione Dinamica del Veicolo e Animazione 2D** — confronto visivo tra la sospensione attiva/adattiva e quella passiva.

1. **Simulazione Fisica (`_simula_rami`, `simula_traiettorie`, `simula_demo`)**
   - **Simulazione a Due Rami (`_simula_rami`)**: Calcola in parallelo tramite integrazione RK4 la risposta di due veicoli identici sulla stessa strada:
     - **Ramo Controllato**: applica lo smorzamento $\xi$ adattivo e la forza dell'attuatore Skyhook $F_{\text{att}}$ (limitata a $\pm F_{\text{max}}$).
     - **Ramo Passivo**: usa lo smorzamento fisso standard ($c_{\text{nom}}$).
   - **Strada Reale vs Strada Demo**: `simula_traiettorie` ricostruisce il profilo della strada dalle misurazioni reali ($a_z, v$). `simula_demo` genera invece una strada sintetica con dossi tarati alla frequenza di risonanza ($r \approx \sqrt{2}$) per evidenziare al massimo l'efficacia del controllo.

2. **Rendering Geometrico (`_crea_pannello`, `_spring_zig`)**
   - Costruisce la grafica 2D in scala reale degli elementi del quarter-car: pneumatico, mozzo, cassa del veicolo, ammortizzatore e molla con il classico disegno a zig-zag.

3. **Animazione Video e Telemetria (`anima_confronto`)**
   - **Confronto Visivo**: Genera un'animazione a due pannelli verticali:
     - **Pannello Superiore (Attivo)**: la cassa rimane ferma e isolata sulla linea di riferimento verde mentre la ruota segue la strada.
     - **Pannello Inferiore (Passivo)**: la cassa subisce i sobbalzi trasmessi dai dossi.
   - **Telemetria Live**: Mostra a schermo in tempo reale il tempo $t$, lo smorzamento $\xi$, la forza erogata dall'attuatore $F_{\text{att}}$ e l'accelerazione RMS $a_z$.
   - **Output**: Apre una finestra interattiva su Matplotlib e permette di esportare il filmato in formato MP4 (se è installato `ffmpeg`).


In [ ]:
# =====================================================================
# 7. FIGURA RISULTATI
# =====================================================================
def salva_figura(xi_true, xi_pred, F_true, F_pred, r_va, rms_pas, rms_opt, path):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception as e:
        print(f"    [i] matplotlib non disponibile, salto la figura ({e})")
        return
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    fig.suptitle("Sospensione 2 GDL Peugeot 207 — ξ adattivo alla strada + attuatore skyhook (regola √2·ωn)",
                 fontsize=13, fontweight="bold")

    n = min(800, len(xi_true))
    tt = np.arange(n) * CFG.dt
    ax[0, 0].plot(tt, xi_true[:n], "r-", lw=1.0, label="ξ target (fisico)")
    ax[0, 0].plot(tt, xi_pred[:n], "g-", lw=1.0, alpha=0.8, label="ξ CNN")
    ax[0, 0].set_title("ξ(t): smorzamento adattivo"); ax[0, 0].set_xlabel("t [s]"); ax[0, 0].set_ylabel("ξ")
    ax[0, 0].legend(fontsize=8); ax[0, 0].grid(alpha=0.3)

    ax[0, 1].plot(tt, F_true[:n], "r-", lw=1.0, label="F_att target")
    ax[0, 1].plot(tt, F_pred[:n], "b-", lw=1.0, alpha=0.8, label="F_att CNN")
    ax[0, 1].axhline(CFG.F_max, ls=":", c="k"); ax[0, 1].axhline(-CFG.F_max, ls=":", c="k")
    ax[0, 1].set_title("F_att(t): attuatore skyhook (saturato ±F_max)")
    ax[0, 1].set_xlabel("t [s]"); ax[0, 1].set_ylabel("F [N]")
    ax[0, 1].legend(fontsize=8); ax[0, 1].grid(alpha=0.3)

    # ξ e kp in funzione di r = ω/ωn : il grafico chiave della regola √2
    rr = np.linspace(0.3, 4.0, 200)
    axL = ax[1, 0]
    sub = np.random.default_rng(0).choice(len(r_va), size=min(3000, len(r_va)), replace=False)
    axL.scatter(r_va[sub], xi_true[sub], s=3, alpha=0.15, c="orange", label="target (strada reale)")
    axL.plot(rr, xi_of_r(rr, CFG), "-", c="darkred", lw=2, label="legge ξ(r)")
    axL.axvline(SQRT2, ls="--", c="k", alpha=0.7); axL.text(SQRT2 + 0.05, 0.57, "r = √2", fontsize=9)
    axL.text(0.5, 0.57, "ξ alto\n(comfort)", fontsize=8, color="darkred")
    axL.text(3.0, 0.21, "ξ basso\n(isolamento)", fontsize=8, color="darkred")
    axL.set_xlabel("r = ω/ωn"); axL.set_ylabel("ξ", color="darkred"); axL.grid(alpha=0.3)
    axL.legend(fontsize=7, loc="center right")
    axR = axL.twinx()
    axR.plot(rr, kp_of_r(rr, CFG), "-", c="teal", lw=2)
    axR.set_ylabel("kp attuatore [Ns/m]", color="teal")
    axL.set_title("ξ e kp vs frequenza di eccitazione r = ω/ωn")

    ax[1, 1].bar(["passiva\n(c_nom)", "adattiva\n(ξ+attuatore)"], [rms_pas, rms_opt],
                 color=["gray", "seagreen"])
    ax[1, 1].set_title(f"RMS accel. cassa (comfort)  —  {100*(1-rms_opt/rms_pas):.0f}% meglio")
    ax[1, 1].set_ylabel("RMS a_z [m/s²]"); ax[1, 1].grid(alpha=0.3, axis="y")

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(path, dpi=120)
    print(f"    [ok] Figura salvata: {os.path.basename(path)}")

# =====================================================================
# 7b. ANIMAZIONE CONFRONTO: CON vs SENZA INTERVENTO
# =====================================================================
def _simula_rami(zr, zr_dot, xi_arr, kp_arr, v_slice, car: QuarterCar2DOF, cfg: Config):
    """
    Integra i DUE rami (controllato: ξ adattivo + attuatore; passivo: c_nom) sullo
    STESSO profilo strada e registra le traiettorie ASSOLUTE z_s, z_u.
    Stato x = [z_s-z_u, zs', z_u-z_r, zu'] -> z_u = x3 + z_r, z_s = x1 + z_u.
    """
    N = len(zr)
    c_arr = car.xi_to_c(xi_arr)
    zs_c = np.zeros(N); zu_c = np.zeros(N); zs_p = np.zeros(N); zu_p = np.zeros(N)
    F = np.zeros(N); az_c = np.zeros(N); az_p = np.zeros(N)
    x = np.zeros(4); xp = np.zeros(4)
    h = cfg.dt / cfg.n_sub
    for k in range(N):
        c = c_arr[k]; kp = kp_arr[k]
        zu_c[k] = x[2] + zr[k]; zs_c[k] = x[0] + zu_c[k]
        zu_p[k] = xp[2] + zr[k]; zs_p[k] = xp[0] + zu_p[k]
        F_att = np.clip(-kp * x[1], -cfg.F_max, cfg.F_max); F[k] = F_att
        az_c[k] = (-car.k_s * x[0] - c * (x[1] - x[3]) + F_att) / car.m_s
        az_p[k] = (-car.k_s * xp[0] - car.c_nom * (xp[1] - xp[3])) / car.m_s
        zr0 = zr_dot[k]; zr1 = zr_dot[k + 1] if k + 1 < N else zr_dot[k]
        for _ in range(cfg.n_sub):
            x = _rk4_step(x, c, F_att, zr0, zr1, car, h)
            xp = _rk4_step(xp, car.c_nom, 0.0, zr0, zr1, car, h)
    x_pos = np.cumsum(np.maximum(v_slice, 0.0) * cfg.dt)
    return dict(zr=zr, zs_c=zs_c, zu_c=zu_c, zs_p=zs_p, zu_p=zu_p, F=F,
                xi=xi_arr.astype(np.float32), az_c=az_c, az_p=az_p, x_pos=x_pos, N=N)


def simula_traiettorie(az_slice, v_slice, car: QuarterCar2DOF, cfg: Config):
    """Traiettorie per l'animazione a partire dalla traccia reale (a_z + v)."""
    zr, zr_dot = ricostruisci_strada(az_slice, cfg)
    if cfg.calibra_ampiezza:
        rms_meas = np.sqrt(np.mean(az_slice ** 2))
        factor = np.clip(rms_meas / (_passive_rms(zr_dot, car, cfg) + 1e-9), 0.2, 20.0)
        zr = zr * factor; zr_dot = zr_dot * factor
    r_arr = stima_omega(az_slice, cfg) / car.wn
    return _simula_rami(zr, zr_dot, xi_of_r(r_arr, cfg), kp_of_r(r_arr, cfg), v_slice, car, cfg)


def simula_demo(car: QuarterCar2DOF, cfg: Config):
    """
    Strada DIMOSTRATIVA: fondo liscio con dossi periodici a velocità costante, con
    contenuto attorno alla risonanza (r~√2) dove l'effetto è massimo. La ruota segue
    i dossi mentre la cassa attiva resta ferma e quella passiva sobbalza.
    """
    N = int(cfg.anim_seconds * cfg.fs)
    t = np.arange(N) * cfg.dt
    v0 = cfg.anim_v
    v = np.full(N, v0)
    x_pos = np.cumsum(v * cfg.dt)
    L = v0 / 1.6                       # spaziatura dossi -> ~1.6 Hz (banda risonanza)
    zr = np.zeros(N)
    for xb in np.arange(4.0, x_pos[-1], L):
        zr += 0.035 * np.exp(-((x_pos - xb) ** 2) / (2 * 0.45 ** 2))
    zr += 0.002 * np.sin(2 * np.pi * 2.5 * t)     # texture leggera
    zr_dot = np.gradient(zr, cfg.dt)
    # ω dal profilo strada (RMS_win(zr') / RMS_win(zr))
    w = cfg.freq_win; ker = np.ones(w) / w
    mrms = lambda s: np.sqrt(np.convolve(s * s, ker, mode="same") + 1e-12)
    omega = np.clip(mrms(zr_dot) / (mrms(zr) + 1e-9), 0.15 * car.wn, cfg.r_clip * car.wn)
    r_arr = omega / car.wn
    return _simula_rami(zr, zr_dot, xi_of_r(r_arr, cfg), kp_of_r(r_arr, cfg), v, car, cfg)


# --- Geometria (metri) del quarter-car disegnato, scala realistica ---
_R_W = 0.33          # raggio ruota
_HUB = 0.13          # raggio mozzo
_BODY_W = 1.9        # larghezza cassa
_BODY_H = 0.55       # altezza cassa
_STRUT = 0.42        # luce ruota-cassa a riposo
_BODY_BASE = 2 * _R_W + _STRUT   # quota base cassa a riposo


def _spring_zig(xc, y0, y1, coils=6, width=0.14):
    ys = np.linspace(y0, y1, 2 * coils + 2)
    xs = np.full_like(ys, xc)
    xs[1:-1:2] = xc - width / 2
    xs[2:-1:2] = xc + width / 2
    return xs, ys


def _crea_pannello(ax, colore, titolo):
    from matplotlib.patches import Circle, Rectangle, Polygon
    asfalto = Polygon([[0, 0]], closed=True, fc="0.78", ec="0.35", lw=1.0, zorder=1)
    road, = ax.plot([], [], "-", c="0.15", lw=2.5, zorder=2)
    tire = Circle((0, 0), _R_W, fc="0.15", ec="black", lw=1.5, zorder=5)
    hub = Circle((0, 0), _HUB, fc="0.7", ec="black", lw=1.0, zorder=6)
    body = Rectangle((0, 0), _BODY_W, _BODY_H, fc=colore, ec="black", lw=1.8, alpha=0.92, zorder=5)
    spring, = ax.plot([], [], "-", c="crimson", lw=2.0, zorder=4)
    damper, = ax.plot([], [], "-", c="royalblue", lw=5.0, solid_capstyle="butt", zorder=4)
    ref = ax.axhline(_BODY_BASE + _BODY_H / 2, ls=":", c="green", alpha=0.5, zorder=3)
    txt = ax.text(0.015, 0.96, "", transform=ax.transAxes, fontsize=10, va="top", zorder=8,
                  bbox=dict(boxstyle="round", fc="white", ec="0.6", alpha=0.85))
    ax.add_patch(asfalto); ax.add_patch(tire); ax.add_patch(hub); ax.add_patch(body)
    ax.set_title(titolo, fontsize=12, fontweight="bold")
    ax.set_ylabel("quota [m]")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, ls="--", alpha=0.25)
    return dict(asfalto=asfalto, road=road, tire=tire, hub=hub, body=body,
               spring=spring, damper=damper, ref=ref, txt=txt)


def anima_confronto(traj, cfg: Config, path_video=None, mostra=True):
    """
    Due pannelli impilati: SOPRA con sospensione adattiva + attuatore, SOTTO passiva.
    La ruota (tangente all'asfalto) segue i dossi; con l'intervento la cassa resta
    incollata alla linea verde di riferimento, senza intervento sobbalza.
    Apre una finestra interattiva (plt.show) e salva il video MP4 (serve ffmpeg).
    """
    try:
        import matplotlib
        if not mostra:
            matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import matplotlib.animation as animation
    except Exception as e:
        print(f"    [i] matplotlib/animazione non disponibile ({e})")
        return

    g = cfg.anim_gain
    W = 4.5                        # semi-finestra strada visibile [m]
    step = max(1, cfg.anim_step)
    frames = range(0, traj["N"], step)
    dt = cfg.dt

    x_pos = traj["x_pos"]
    road_y = traj["zr"] * g                        # profilo asfalto amplificato
    body_yc = _BODY_BASE + traj["zs_c"] * g        # base cassa (con intervento)
    body_yp = _BODY_BASE + traj["zs_p"] * g        # base cassa (passivo)

    fig, (axT, axB) = plt.subplots(2, 1, figsize=(12, 8.2), sharex=True)
    fig.suptitle("Peugeot 207 sulla stessa strada con dossi — sospensione adattiva vs passiva",
                 fontsize=13, fontweight="bold")
    A = _crea_pannello(axT, "seagreen", "CON intervento  (ξ adattivo + attuatore skyhook)")
    B = _crea_pannello(axB, "0.55", "SENZA intervento  (passivo c_nom)")
    for ax in (axT, axB):
        ax.set_ylim(-0.3, _BODY_BASE + _BODY_H + 0.5)
    axB.set_xlabel("posizione strada [m]")

    ybot = -0.3

    def set_panel(P, i, body_y):
        xc = x_pos[i]
        mask = (x_pos >= xc - W) & (x_pos <= xc + W)
        rx = x_pos[mask]; ry = road_y[mask]
        P["road"].set_data(rx, ry)
        # asfalto pieno sotto la strada
        verts = np.column_stack([np.r_[rx, rx[-1], rx[0]], np.r_[ry, ybot, ybot]])
        P["asfalto"].set_xy(verts)
        # ruota tangente all'asfalto: centro = quota strada locale + raggio
        z_road_here = road_y[i]
        cyr = z_road_here + _R_W
        P["tire"].center = (xc, cyr); P["hub"].center = (xc, cyr)
        # cassa
        P["body"].set_xy((xc - _BODY_W / 2, body_y[i]))
        # molla (sx) e ammortizzatore/attuatore (dx) tra sommità ruota e base cassa
        y_top_wheel = cyr + _R_W
        sx, sy = _spring_zig(xc - 0.28, y_top_wheel, body_y[i])
        P["spring"].set_data(sx, sy)
        P["damper"].set_data([xc + 0.28, xc + 0.28], [y_top_wheel, body_y[i]])

    def update(i):
        set_panel(A, i, body_yc)
        set_panel(B, i, body_yp)
        rms_c = np.sqrt(np.mean(traj["az_c"][:i+1] ** 2))
        rms_p = np.sqrt(np.mean(traj["az_p"][:i+1] ** 2))
        A["txt"].set_text(f"t = {i*dt:4.1f} s     ξ = {traj['xi'][i]:.2f}     "
                          f"F_att = {traj['F'][i]:+5.0f} N     RMS a_z = {rms_c:.2f} m/s²")
        B["txt"].set_text(f"t = {i*dt:4.1f} s     c = c_nom fisso"
                          f"                    RMS a_z = {rms_p:.2f} m/s²")
        xc = x_pos[i]
        axT.set_xlim(xc - W, xc + W); axB.set_xlim(xc - W, xc + W)
        return (A["asfalto"], A["road"], A["tire"], A["hub"], A["body"], A["spring"], A["damper"], A["txt"],
                B["asfalto"], B["road"], B["tire"], B["hub"], B["body"], B["spring"], B["damper"], B["txt"])

    anim = animation.FuncAnimation(fig, update, frames=frames, interval=1000/cfg.anim_fps,
                                   blit=False, repeat=False)

    # 1) salva il VIDEO MP4 (serve ffmpeg installato)
    if path_video and cfg.anim_save_video:
        try:
            from matplotlib.animation import FFMpegWriter
            anim.save(path_video, writer=FFMpegWriter(fps=cfg.anim_fps, bitrate=2600),
                      dpi=120)
            print(f"    [ok] Video salvato: {os.path.basename(path_video)}")
        except Exception as e:
            print(f"    [i] MP4 non salvato (installa ffmpeg: 'brew install ffmpeg'). Dettaglio: {e}")

    # 2) apre la finestra interattiva Python
    if mostra:
        plt.show()
    else:
        plt.close(fig)
    return anim

---

**Pipeline Finale ed Esecuzione** (`main`):

Coordinamento in sequenza della preparazione dei dati, dell'addestramento della CNN 1D, della validazione metrica e della generazione finale dei report grafici e video.

1. **Preparazione Dati & Normalizzazione**
   - Carica le tracce (2 train, 1 validation).
   - Applica la scala $[0, 1]$ a $\xi$ (Sigmoid) e $[-1, 1]$ a $F_{\text{att}}$ (Tanh).
   - Impacchetta il tutto nei `DataLoader` di PyTorch con batch size pari a 64.

2. **Addestramento Multi-Task**
   - Allena la rete con l'ottimizzatore **AdamW** usando una loss pesata $\text{MSE}(\xi) + w_F \cdot \text{MSE}(F_{\text{att}})$ (dando maggior peso alla forza dell'attuatore).
   - Monitora le prestazioni sulla traccia di validazione mai vista e salva il miglior modello su disco (`.pt`).

3. **Valutazione Metrica**
   - Calcola i coefficienti di determinazione $R^2$ e gli errori MAE sulle predizioni di $\xi$ e $F_{\text{att}}$.
   - Misura la percentuale di abbattimento dell'accelerazione RMS della cassa rispetto alla sospensione passiva.

4. **Analisi della Regola $\sqrt{2}$**
   - Suddivide i risultati di validazione in intervalli del rapporto di frequenza $r = \omega/\omega_n$.
   - Stampa una tabella a console che mostra come variano velocità media, $\xi$, smorzamento $c$, guadagno $k_p$ e forza $F_{\text{att}}$ a seconda del regime stradale.

5. **Render e Output**
   - Invoca `salva_figura` per generare la dashboard `.png` a 4 riquadri.
   - Se abilitato, avvia la simulazione 2D con `anima_confronto` esportando l'animazione video.

Notare:

**Definizione della Loss**

Questa formulazione è una **Funzione di Loss Multi-Task a combinazione lineare statica** (o *scalarizzazione lineare a peso fisso*), come definito nella letteratura classica del Multi-Task Learning (Caruana, 1997, *"Multitask Learning"*, Springer):

$$\text{Loss} = \text{Loss}_{\xi} + w_F \cdot \text{Loss}_{F_{\text{att}}}$$

* **Multi-Task:** Combina due target fisici differenti ($\xi$ e $F_{\text{att}}$) in un unico scalare per guidare la retropropagazione di un'unica rete condivisa.
* **Pesatura Statica:** Il coefficiente $w_F$ è un iperparametro rigido e prefissato che non cambia durante le epoche.
* **Assenza di Bilanciamento Dinamico:** Manca di meccanismi di calibrazione automatica (come *Uncertainty Weighting* o *GradNorm*) -in accordo con la rassegna sullo stato dell'arte del Deep MTL (Ruder, 2017)-, lasciando che la differente scala degli errori e la magnitudo dei gradienti entrino in diretta competizione (*allo stato delle cose non ci eravamo ancora accorti che cosa potesse succedere non avere un bilanciamento dinamico*).


In [ ]:
def main():
    t0 = time.time()
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.path.expanduser("~/Desktop")
    model_path = os.path.join(base_dir, "suspension_adaptive_cnn.pt")
    fig_path = os.path.join(base_dir, "risultati_sospensione.png")

    print("=" * 94)
    print(" SOSPENSIONE 2 GDL (PEUGEOT 207) — ξ e kp ADATTIVI ALLA STRADA+VELOCITÀ (regola √2·ωn)")
    print(f" ωn = {CAR.wn:.2f} rad/s ({CAR.wn/(2*np.pi):.2f} Hz)   |   √2·ωn = {np.sqrt(2)*CAR.wn:.2f} rad/s "
          f"({np.sqrt(2)*CAR.wn/(2*np.pi):.2f} Hz)")
    print(f" Crossover a r = ω/ωn = √2 (ω stimata istante per istante da strada+velocità)")
    print(f" ξ ∈ [{CFG.xi_lo:.2f}, {CFG.xi_hi:.2f}]   |   F_att saturata a ±{CFG.F_max:.0f} N (no forza infinita)"
          f"   |   calibrazione ampiezza: {CFG.calibra_ampiezza}")
    print("=" * 94)

    gen = PhysicsAdaptiveTargetGenerator(CAR, CFG)

    print("\n[1] Caricamento tracce e generazione target (integrazione ODE 2 GDL)...")
    tracce_tr, tracce_va = carica_tracce()
    Xaz_tr, Xv_tr, Yxi_tr, YF_tr, Xr_tr, cs_tr = costruisci_finestre(tracce_tr, gen, CFG)
    Xaz_va, Xv_va, Yxi_va, YF_va, Xr_va, cs_va = costruisci_finestre(tracce_va, gen, CFG)
    print(f"    Campioni TRAIN : {len(Xaz_tr)}  (2 tracce)")
    print(f"    Campioni VALID : {len(Xaz_va)}  (3ª traccia, mai vista)")

    def norm_xi(x): return (x - XI_MIN) / (XI_MAX - XI_MIN)
    def denorm_xi(x): return XI_MIN + x * (XI_MAX - XI_MIN)
    def norm_F(x): return x / CFG.F_max
    def denorm_F(x): return x * CFG.F_max

    tr_ds = SuspensionDataset(Xaz_tr, Xv_tr, norm_xi(Yxi_tr), norm_F(YF_tr))
    va_ds = SuspensionDataset(Xaz_va, Xv_va, norm_xi(Yxi_va), norm_F(YF_va))
    tr_loader = DataLoader(tr_ds, batch_size=64, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=64, shuffle=False)

    print("\n[2] Addestramento CNN 1D (target: ξ adattivo + forza attuatore)...")
    model = SuspensionCNN1D()
    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.MSELoss()

    best = float("inf")
    for epoch in range(1, CFG.epochs + 1):
        model.train(); tl = 0.0
        for az, v, yxi, yF in tr_loader:
            opt.zero_grad()
            pxi, pF = model(az, v)
            loss = crit(pxi, yxi) + CFG.w_F * crit(pF, yF)   # peso maggiore su F_att
            loss.backward(); opt.step()
            tl += loss.item() * len(az)
        tl /= len(tr_ds)

        model.eval(); vl = 0.0
        with torch.no_grad():
            for az, v, yxi, yF in va_loader:
                pxi, pF = model(az, v)
                vl += (crit(pxi, yxi) + CFG.w_F * crit(pF, yF)).item() * len(az)
        vl /= len(va_ds)
        if vl < best:
            best = vl
            torch.save(model.state_dict(), model_path)
        if epoch % 5 == 0 or epoch == 1:
            print(f"    Epoca {epoch:02d}/{CFG.epochs} | Loss Train {tl:.5f} | Loss Val {vl:.5f}")

    model.load_state_dict(torch.load(model_path)); model.eval()
    with torch.no_grad():
        pxi, pF = model(torch.tensor(Xaz_va).unsqueeze(1), torch.tensor(Xv_va).unsqueeze(1))
    xi_pred = denorm_xi(pxi.numpy().flatten())
    F_pred = denorm_F(pF.numpy().flatten())
    xi_true, F_true = Yxi_va, YF_va

    print("\n" + "=" * 94)
    print(" VALIDAZIONE SU TRACCIA ISOLATA (3ª, MAI VISTA)")
    print("=" * 94)
    print(f" ξ(t)     -> R²: {r2_score(xi_true, xi_pred):.4f} | MAE: {mean_absolute_error(xi_true, xi_pred):.4f}")
    print(f" F_att(t) -> R²: {r2_score(F_true, F_pred):.4f} | MAE: {mean_absolute_error(F_true, F_pred):.1f} N")
    print(f" |F_att| max previsto dalla CNN: {np.abs(F_pred).max():.1f} N   (limite {CFG.F_max:.0f} N)")

    rms_pas = np.mean([p for p, o in (cs_tr + cs_va)])
    rms_opt = np.mean([o for p, o in (cs_tr + cs_va)])
    print(f"\n Comfort  RMS accel cassa  passiva: {rms_pas:.3f} m/s²  ->  adattiva: {rms_opt:.3f} m/s²"
          f"  ({100*(1-rms_opt/rms_pas):.1f}% miglioramento)")

    print("\n" + "-" * 82)
    print(" REGOLA √2: ξ e kp in funzione del rapporto r = ω/ωn (target sul validation)")
    print(f" (ω stimata da strada+velocità; crossover a r = √2 ≈ {SQRT2:.3f})")
    print("-" * 82)
    r_bins = [0.0, 1.0, SQRT2, 2.0, 10.0]
    r_labels = ["r<1 (ω<<√2ωn)", "1<r<√2 (comfort)", "√2<r<2 (isolam.)", "r>2 (ω>>√2ωn)"]
    print(f"{'Range r':<18} | {'v med [km/h]':<12} | {'ξ medio':<9} | {'c [Ns/m]':<10} | {'kp [Ns/m]':<10} | {'|F_att| [N]'}")
    print("-" * 82)
    for i in range(len(r_bins) - 1):
        m = (Xr_va >= r_bins[i]) & (Xr_va < r_bins[i+1])
        if np.any(m):
            xim = xi_true[m].mean()
            print(f"{r_labels[i]:<18} | {Xv_va[m].mean()*3.6:<12.1f} | {xim:<9.3f} | {CAR.xi_to_c(xim):<10.0f} | "
                  f"{kp_of_r(Xr_va[m].mean(), CFG):<10.0f} | {np.abs(F_true[m]).mean():.0f}")

    if CFG.save_fig:
        salva_figura(xi_true, xi_pred, F_true, F_pred, Xr_va, rms_pas, rms_opt, fig_path)

    # --- Animazione confronto CON vs SENZA intervento ---
    if CFG.show_anim:
        print("\n[3] Animazione confronto (con vs senza sospensione attiva)...")
        video_path = os.path.join(base_dir, CFG.anim_video_file)
        if CFG.anim_strada_demo or not tracce_va:
            traj = simula_demo(CAR, CFG)
        else:
            az_v, v_v = tracce_va[0]
            n_anim = min(len(az_v), int(CFG.anim_seconds * CFG.fs))
            traj = simula_traiettorie(az_v[:n_anim], v_v[:n_anim], CAR, CFG)
        anima_confronto(traj, CFG, path_video=video_path, mostra=CFG.anim_mostra)

    print("=" * 94)
    print(f"Completato in {time.time()-t0:.1f} s")


if __name__ == "__main__":
    main()


---

**Nota sull'output:** le celle di questa sezione sono eseguibili end-to-end così come sono (non richiedono librerie esterne al notebook). L'output viene comunque riportato già calcolato qui sotto per velocizzare la lettura, evitando di dover rieseguire l'intera pipeline (download del dataset, training, inferenza) solo per consultare i risultati.

Output a terminale:

<pre style="font-size:11px; line-height:1.2; overflow-x:auto;">
==============================================================================================
 SOSPENSIONE 2 GDL (PEUGEOT 207) — ξ e kp ADATTIVI ALLA STRADA+VELOCITÀ (regola √2·ωn)
 ωn = 9.41 rad/s (1.50 Hz)   |   √2·ωn = 13.30 rad/s (2.12 Hz)
 Crossover a r = ω/ωn = √2 (ω stimata istante per istante da strada+velocità)
 ξ ∈ [0.18, 0.60]   |   F_att saturata a ±1500 N (no forza infinita)   |   calibrazione ampiezza: True
==============================================================================================

[1] Caricamento tracce e generazione target (integrazione ODE 2 GDL)...
  [ok] 3 tracce reali -> 2 train / 1 validazione (thessaloniki_19_11_2023.pkl)
    Campioni TRAIN : 360308  (2 tracce)
    Campioni VALID : 157412  (3ª traccia, mai vista)

[2] Addestramento CNN 1D (target: ξ adattivo + forza attuatore)...
    Epoca 01/15 | Loss Train 0.02598 | Loss Val 0.03335
    Epoca 05/15 | Loss Train 0.02068 | Loss Val 0.03450
    Epoca 10/15 | Loss Train 0.01790 | Loss Val 0.03660
    Epoca 15/15 | Loss Train 0.01611 | Loss Val 0.03772

==============================================================================================
 VALIDAZIONE SU TRACCIA ISOLATA (3ª, MAI VISTA)
==============================================================================================
 ξ(t)     -> R²: 0.4738 | MAE: 0.0657
 F_att(t) -> R²: 0.6103 | MAE: 22.8 N
 |F_att| max previsto dalla CNN: 290.8 N   (limite 1500 N)

 Comfort  RMS accel cassa  passiva: 0.323 m/s²  ->  adattiva: 0.195 m/s²  (39.6% miglioramento)

----------------------------------------------------------------------------------
 REGOLA √2: ξ e kp in funzione del rapporto r = ω/ωn (target sul validation)
 (ω stimata da strada+velocità; crossover a r = √2 ≈ 1.414)
----------------------------------------------------------------------------------
Range r            | v med [km/h] | ξ medio   | c [Ns/m]   | kp [Ns/m]  | |F_att| [N]
----------------------------------------------------------------------------------
r<1 (ω<<√2ωn)      | 34.8         | 0.528     | 2581       | 584        | 35
1<r<√2 (comfort)   | 32.7         | 0.446     | 2180       | 1036       | 38
√2<r<2 (isolam.)   | 30.8         | 0.312     | 1524       | 1799       | 40
r>2 (ω>>√2ωn)      | 30.6         | 0.206     | 1006       | 2416       | 32
    [ok] Figura salvata: risultati_sospensione.png
</pre>


<div style="text-align:left;">
    <img src="img_CNN2Teste/risultati_sospensione - ReLU.png"
         style="width:50%; height:auto; display:block; margin:0;">
</div>

**Analisi Critica dei Risultati e tuning degli iperparametri**

L'analisi dell'output evidenzia due criticità principali nel comportamento della rete:

* **Inversione della Loss (Overfitting):**
  Mentre la `Loss(Training)` scende con continuità ($0.02598 \rightarrow 0.01611$), la `Loss(Validation)` comincia a salire già a partire dall'epoca 5 ($0.03335 \rightarrow 0.03772$). Questo divergere chiaro indica che il modello sta andando in overfitting sulla traccia di addestramento, perdendo capacità di generalizzazione.

* **Metriche di Accuratezza Sovrastimate/Basse:**
  Il coefficiente di determinazione per lo smorzamento adattivo $\xi(t)$ si ferma a $R^2 = 0.4738$. La rete spiega meno del 50% della varianza del target su una strada mai vista.

**Interventi Applicati:**
Per risolvere il degrado della loss di validazione e incrementare l'$R^2$, si è proceduto a una fase di ottimizzazione (Ablation Study) agendo su due fronti:

1. **Funzioni di Attivazione:** Sostituzione attivazioni differenti nei layer interni della rete neurale (provando ad utilizzare *GeLU*, Hendrycks e Gimpel 2016, al posto di *ReLU* dentro la rete CNN) per prevenire il fenomeno dei "neuroni morti" e migliorare la propagazione del gradiente verso i branch finali (ovvero i branch `Sigmoid` e `Tanh`).
2. **Learning Rate & Regolarizzazione:** Tuning del learning rate della funzione di ottimizzazione *AdamW* (riduzione del valore iniziale unito a un incremento del *Weight Decay* per contrastare la salita della `Loss(Validation)`): dopo vari run la combinazione lr=1e-3, weight_decay=1e-4 per una funzione optimizer di tipo AdamW si è rivelata lo sweet spot.


#### 1.2.2 **Reti TCN separate, parallelizzazione dei processi tramite GPU/CPU e necessità di aumentare il DataSet - 2GDL "Parziale"**   

**Evoluzione dell'Architettura e Strategia di Ottimizzazione**

Per superare i limiti dell'approccio multi-task e garantire prestazioni ottimali sul veicolo, la pipeline di modellazione viene riorganizzata secondo tre direttrici fondamentali:

* **Reti TCN Separate:**
  La sostituzione della CNN 1D multi-head con due **Temporal Convolutional Networks (TCN) indipendenti** risponde a una duplice esigenza:
  1. *Eliminazione del Conflitto:* Disaccoppia completamente l'ottimizzazione di $\xi$ e $F_{\text{att}}$, annullando la competizione tra gradienti.
  2. *Memoria Temporale Estesa:* Le TCN, grazie alle convoluzioni dilatate e causali, catturano con maggiore precisione le dipendenze temporali a lungo termine e le dinamiche transitorie delle sospensioni rispetto alle tradizionali CNN 1D.

* **Parallelizzazione Hardware (GPU/CPU):**
  La totale indipendenza delle due reti apre in linea di principio alla possibilità di addestrarle in **parallelo** (thread separati su CPU o stream distinti su GPU), cosa non praticabile con un'unica rete multi-head. Nella pipeline finale le due reti vengono comunque addestrate in sequenza sullo stesso device: il guadagno reale non è quindi un minor tempo di calcolo, ma la possibilità di ottimizzarle separatamente senza che l'una condizioni i gradienti dell'altra.

* **Espansione e Data Augmentation del Dataset:**
  La persistente divaricazione tra `Loss(Training)` e `Loss(Validation)` evidenzia che due sole tracce di addestramento non coprono la varietà statistica dei profili stradali reali. Aumentare il volume del dataset (con nuove tracce e tecniche di data augmentation sulle frequenze) è indispensabile per migliorare la capacità di generalizzazione e innalzare l'$R^2$ su strade mai viste.  
  Verrà ampiamente trattato più avanti come è stato l'approccio del Data Augmentation per aumentare il numero di tracce per training e validazione: in soldoni si sono utilizzate tabelle standard ISO facilmente reperibili in rete che contengono come deve essere il contenuto spettrale delle accelerazioni verticali e le velocità di traslazione per normative su strada; pertanto integrando le accelerazioni verticali e sapendo che i moti verticali in realtà dipendono dalla traslazione orizzontale (il problema dell'auto è tutti gli effetti un'onda nello spazio e nel tempo), si possono costruire nel dominio temporale funzioni armoniche/polinomiali finestrate di larghezza $T=[T_0,T_1]$ (s) ed $L=[0,L]$ (m) che rappresentano strade sintetiche (a cui si può sovrapporre della granatura geometrica stocastica che rasenta la rugosità della strada anch'essa normata).


* **Passaggio ad un modello 2GDL parziale - Aggiunta della massa della ruota a terra:**

  * **Modellazione della Tenuta di Strada (Road Holding):**
    L'inserimento di $m_u$ e della rigidezza del pneumatico ($k_t$) ha permesso di monitorare la deflessione dinamica della gomma. Senza la massa della ruota, il modello non avrebbe potuto quantificare la variazione del carico verticale a terra, rendendo impossibile valutare se l'azione dell'attuatore compromettesse la sicurezza del veicolo.

  * **Doppia Frequenza di Risonanza (*Wheel Hop*):**
    Il modello non considera più soltanto il moto di cassa (*body bounce*, $\omega_n \approx 9.41 \text{ rad/s} \approx 1.50 \text{ Hz}$), ma introduce il secondo modo di vibrare ad alta frequenza legato al gruppo ruota (*wheel hop*, tipicamente $10\text{--}12 \text{ Hz}$). Questo ha costretto l'algoritmo di ML a gestire sollecitazioni spettrali molto più complesse e vicine alla realtà.

  * **Vincolo Realistico sull'Attuatore ($F_{\text{att}}$):**
    L'accoppiamento dinamico tra $m_s$ (massa sospesa) e $m_u$ ha mostrato come un'eccessiva forza dell'attuatore per abbattere l'accelerazione del telaio rischiasse di far staccare la ruota dall'asfalto. L'aggiunta della massa ruota è stata quindi la premessa fondamentale per imporre il limite fisico di $1500 \text{ N}$ sulla forza.

**DISCLAIMER:** 

Per rispondere alla naturale domanda **"Ma perché non avete semplicemente cambiato DataSet prendendone uno con più dettagli"**, la risposta è che dopo aver fatto una ricerca meticolosa di DataSet alternativi , sia su KaggleHub sia su altri supporti, non si è trovato nulla di consistente che avesse una frequenza di campionamento abbastanza alta ($f_s \geq 100$ Hz) all'infuori di quello utilizzato qui: tutti i data set trovati usavano come accelerometri semplici smartphone con una $f_s < 10$ Hz e jittering elevatissimo; ciò significa che per il teorema di Shannon-Nyquist non saremmo stati in grado nemmeno di risolvere le frequenze proprie dell'auto (che sono tipicamente intorno proprio ai 10Hz) oltre a portare in banda rumore numerico ed essere soggiogati all'aliasing, rendendo di fatto inutile l'implementazione.

**Senza dovizia di completezza vengono riportate a sommi capi le modifiche sostanziali rispetto alla versione a due teste nei punti salienti, non verranno riportate librerie custom che devono essere importate perché come vedremo anche questo modello verrà abbandonato e sono più importanti i risultati che non il codice.**

---

> Costruzione delle nuove classi separate per le reti di tipo TCN

In [ ]:
class ReteXi(nn.Module):
    """TCN 1D con dilatazioni crescenti (1,2,4,8) per un ampio contesto temporale."""

    def __init__(self, seq_len, att=nn.ReLU, hidden=128):
        super().__init__()
        self.tcn = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1, dilation=1), nn.BatchNorm1d(32), att(),
            nn.Conv1d(32, 32, 3, padding=2, dilation=2), nn.BatchNorm1d(32), att(),
            nn.Conv1d(32, 64, 3, padding=4, dilation=4), nn.BatchNorm1d(64), att(),
            nn.Conv1d(64, 64, 3, padding=8, dilation=8), nn.BatchNorm1d(64), att(),
        )
        with torch.no_grad():                         # dimensione flatten dalla finestra fissa
            self.tcn.eval()
            n_flat = self.tcn(torch.zeros(1, 1, seq_len)).flatten(1).size(1)
            self.tcn.train()
        self.fc = nn.Sequential(nn.Linear(n_flat + 1, hidden), att(), nn.Dropout(0.2))
        self.testa = nn.Sequential(nn.Linear(hidden, 32), att(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, az_norm, v_norm):
        """az_norm: (B,1,seq) normalizzata ; v_norm: (B,1). Uscita: xi_norm in (0,1)."""
        feat = self.tcn(az_norm).flatten(1)
        return self.testa(self.fc(torch.cat((feat, v_norm), dim=1)))


class ReteForza(nn.Module):
    """CNN 1D per stimare la forza attuatore (regressione)."""

    def __init__(self, seq_len, att=nn.GELU, hidden=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, 5, padding=2), nn.BatchNorm1d(32), att(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), att(), nn.MaxPool1d(2),
        )
        with torch.no_grad():
            self.conv.eval()
            n_flat = self.conv(torch.zeros(1, 1, seq_len)).flatten(1).size(1)
            self.conv.train()
        self.fc = nn.Sequential(nn.Linear(n_flat + 1, hidden), att(), nn.Dropout(0.2))
        self.testa = nn.Sequential(nn.Linear(hidden, 32), att(), nn.Linear(32, 1), nn.Tanh())

    def forward(self, az_norm, v_norm):
        """Uscita: forza_norm in (-1,1)."""
        feat = self.conv(az_norm).flatten(1)
        return self.testa(self.fc(torch.cat((feat, v_norm), dim=1)))

> Generazione stradale: profili sintetici basati sulle tabelle dello standard **ISO 8608:2016** (estratto PDF consultabile su [standards.iteh.ai](https://cdn.standards.iteh.ai/samples/71202/05b2151f255b44928f80acb897fc0c2c/ISO-8608-2016.pdf)) in tandem con uno script open-source trovato online e modificato per le nostre esigenze.
> Verranno generate 6 strade sintetiche extra da aggiungere alle due strade reali contenute nel dataset.


In [ ]:
# Gd(n0) [m^3] per classe ISO 8608 (A liscia -> E molto sconnessa), n0 = 0.1 cicli/m
_GD0 = {"A": 16e-6, "B": 64e-6, "C": 256e-6, "D": 1024e-6, "E": 4096e-6}


def _profilo_iso8608(classe, x, rng, n_min=0.011, n_max=2.83, n_freq=300):
    """Profilo strada z_r(x) [m] per la classe data, sintesi a somma di sinusoidi (ISO 8608)."""
    n = np.linspace(n_min, n_max, n_freq)          # frequenze spaziali [cicli/m]
    dn = n[1] - n[0]
    Gd = _GD0[classe] * (n / 0.1) ** (-2)            # densita' spettrale
    amp = np.sqrt(2.0 * Gd * dn)                     # ampiezza di ogni sinusoide
    phi = rng.uniform(0, 2 * np.pi, n_freq)
    # z_r(x) = somma_i amp_i * cos(2*pi*n_i*x + phi_i)   (matrice x-per-n)
    return (amp[None, :] * np.cos(2 * np.pi * np.outer(x, n) + phi[None, :])).sum(axis=1)


def _aggiungi_buche(zr, x, rng, n_buche, prof_min=0.02, prof_max=0.08, largh=0.4):
    """Buche/dossi isolati: avvallamenti gaussiani a posizioni casuali (profondita' variabile)."""
    for _ in range(n_buche):
        xb = rng.uniform(x[0] + 5, x[-1] - 5)
        prof = rng.uniform(prof_min, prof_max) * rng.choice([-1.0, 1.0])  # buca (giu') o dosso (su')
        zr = zr + prof * np.exp(-((x - xb) ** 2) / (2 * largh ** 2))
    return zr


def _salita_discesa(x, rng):
    """Ondulazioni lente del terreno (salite/discese): lunghezza d'onda 40..150 m, ampiezza cm-dm.
    NB: la componente quasi-statica viene poi tolta dal passa-alto; conta la sua VARIAZIONE."""
    lam = rng.uniform(40.0, 150.0)                  # lunghezza d'onda collina [m]
    amp = rng.uniform(0.05, 0.30)                   # dislivello [m]
    fase = rng.uniform(0, 2 * np.pi)
    return amp * np.sin(2 * np.pi * x / lam + fase)


def genera_tracce_sintetiche(auto, cfg):
    """Restituisce una lista di tracce (a_z, v) sintetiche da aggiungere al TRAINING."""
    rng = np.random.default_rng(1234)
    classi = ["B", "C", "C", "D", "D", "E"]         # da abbastanza liscio a molto sconnesso
    velocita = [8.0, 14.0, 20.0, 25.0, 16.0, 11.0]  # m/s (diverse -> copre r bassi e alti)
    N = int(cfg.aug_secondi * cfg.freq_campion)
    tracce = []
    for i in range(cfg.n_tracce_sintetiche):
        classe = classi[i % len(classi)]
        v0 = velocita[i % len(velocita)]
        v = np.full(N, v0) + rng.normal(0, 0.3, N)   # velocita' quasi costante con piccole variazioni
        v = np.clip(v, 3.0, 33.0)
        x = np.cumsum(v * cfg.passo_t)               # avanzamento longitudinale [m]
        zr = _profilo_iso8608(classe, x, rng)
        zr = zr + _salita_discesa(x, rng)            # salite/discese
        zr = _aggiungi_buche(zr, x, rng, n_buche=rng.integers(3, 8))
        zr_dot = np.gradient(zr, cfg.passo_t)
        az_pulita = accel_passiva_serie(zr_dot, auto, cfg)   # a_z del modello passivo
        # rumore del sensore realistico: gaussiano + qualche picco (li togliera' il despiking)
        az = az_pulita + rng.normal(0, 0.05, N)
        for _ in range(rng.integers(2, 6)):
            az[rng.integers(N)] += rng.uniform(2, 6) * rng.choice([-1.0, 1.0])
        tracce.append((az.astype(float), v.astype(float)))
        print(f"      strada sintetica {i+1}: classe {classe}, v0={v0*3.6:.0f} km/h, "
              f"RMS a_z={np.sqrt(np.mean(az_pulita**2)):.2f} m/s^2")
    return tracce

> Modifica della pipeline: due reti separate, loss e optimizer totalmente indipendenti, training su tracce reali e sintetiche.  
> Come si nota vengono richieste funzioni e librerie custom che non sono presenti nel documento pertanto il file non è eseguibile all'interno del file jupyter ed è una scelta precisa perché si vuole dimostrare altre problematiche come detto poc'anzi.

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import r2_score, mean_absolute_error

from config import Auto, Config
from hardware import rileva_hardware
from dati import carica_tracce, costruisci_finestre, Normalizzatore
from controllo import GeneratoreEtichette, kp_da_r, RADQ2
from reti import crea_reti, xi_a_norm, norm_a_xi, forza_a_norm, norm_a_forza
from simulazione import traiettorie_demo_ml, traiettorie_reali_ml, simula_closed_loop
from strade_sintetiche import genera_tracce_sintetiche
from fisica import ricostruisci_strada, rms_accel_passiva
from grafica import salva_figura, salva_figura_efficienza, anima_confronto


def main():
    t0 = time.time()
    auto, cfg = Auto(), Config()
    base = os.path.dirname(os.path.abspath(__file__))
    path_xi = os.path.join(base, "rete_xi.pt")
    path_forza = os.path.join(base, "rete_forza.pt")

    print("=" * 90)
    print(" SOSPENSIONE 2 GDL (PEUGEOT 207) — CONTROLLO APPRESO (ML), REGOLA √2")
    print(f" omega_n = {auto.puls_nat_cassa:.2f} rad/s ({auto.puls_nat_cassa/(2*np.pi):.2f} Hz)"
          f"   |   crossover a r = √2 ≈ {RADQ2:.3f}")
    print("=" * 90)

    device, n_core = rileva_hardware(cfg)

    # 1-2) dati + etichette + finestre
    print("\n[1] Caricamento tracce e generazione etichette (parallela sui core)...")
    tracce_tr, tracce_va = carica_tracce(cfg)
    if cfg.usa_augmentation:                         # aggiunge strade sintetiche varie al TRAINING
        print(f"    Data augmentation: {cfg.n_tracce_sintetiche} strade sintetiche ISO 8608 + buche/salite")
        tracce_tr = tracce_tr + genera_tracce_sintetiche(auto, cfg)
    az_tr, v_tr, y_xi_tr, y_f_tr, r_tr, comfort_tr = costruisci_finestre(tracce_tr, auto, cfg, n_core)
    az_va, v_va, y_xi_va, y_f_va, r_va, comfort_va = costruisci_finestre(tracce_va, auto, cfg, n_core)
    print(f"    Campioni TRAIN : {len(az_tr)}  ({len(tracce_tr)} tracce = reali + sintetiche)")
    print(f"    Campioni VALID : {len(az_va)}  (3a traccia, mai vista)")

    # 3) NORMALIZZAZIONE calcolata SOLO dal training (media/dev-std che emergono dai dati)
    norm = Normalizzatore(az_tr, v_tr)
    print(f"    Normalizzazione (dai dati di training): {norm}")

    def su_device(a):
        return torch.tensor(a, dtype=torch.float32, device=device)

    # ingressi normalizzati + target scalati sui limiti fisici (xi->range damper, forza->F_max)
    AZ_tr = su_device(norm.na(az_tr)).unsqueeze(1); V_tr = su_device(norm.nv(v_tr)).unsqueeze(1)
    XI_tr = su_device(xi_a_norm(y_xi_tr, auto)).unsqueeze(1); F_tr = su_device(forza_a_norm(y_f_tr, cfg)).unsqueeze(1)
    AZ_va = su_device(norm.na(az_va)).unsqueeze(1); V_va = su_device(norm.nv(v_va)).unsqueeze(1)
    XI_va = su_device(xi_a_norm(y_xi_va, auto)).unsqueeze(1); F_va = su_device(forza_a_norm(y_f_va, cfg)).unsqueeze(1)

    # 4) DUE reti separate, loss e optimizer indipendenti
    print("\n[2] Addestramento DUE reti separate (loss e optimizer indipendenti)...")
    rete_xi, rete_forza = crea_reti(cfg)
    rete_xi.to(device); rete_forza.to(device)
    opt_xi = optim.AdamW(rete_xi.parameters(), lr=1e-3, weight_decay=1e-4)
    opt_forza = optim.AdamW(rete_forza.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_xi = nn.HuberLoss(delta=0.25)     # robusta agli outlier per xi
    loss_forza = nn.MSELoss()

    Ntr, Nva, bs = AZ_tr.size(0), AZ_va.size(0), cfg.batch

    def loss_val(rete, target, criterio):
        rete.eval(); s = 0.0
        with torch.no_grad():
            for i in range(0, Nva, bs):
                nb = min(bs, Nva - i)
                s += criterio(rete(AZ_va[i:i+bs], V_va[i:i+bs]), target[i:i+bs]).item() * nb
        return s / Nva

    best_xi = best_f = float("inf")
    for epoca in range(1, cfg.n_epoche + 1):
        rete_xi.train(); rete_forza.train()
        perm = torch.randperm(Ntr, device=device)
        tl_xi = tl_f = 0.0
        for i in range(0, Ntr, bs):
            idx = perm[i:i+bs]; az = AZ_tr[idx]; v = V_tr[idx]
            opt_xi.zero_grad(); lx = loss_xi(rete_xi(az, v), XI_tr[idx]); lx.backward(); opt_xi.step()
            opt_forza.zero_grad(); lf = loss_forza(rete_forza(az, v), F_tr[idx]); lf.backward(); opt_forza.step()
            tl_xi += lx.item() * idx.size(0); tl_f += lf.item() * idx.size(0)
        tl_xi /= Ntr; tl_f /= Ntr
        vl_xi = loss_val(rete_xi, XI_va, loss_xi); vl_f = loss_val(rete_forza, F_va, loss_forza)
        if vl_xi < best_xi:
            best_xi = vl_xi; torch.save(rete_xi.state_dict(), path_xi)
        if vl_f < best_f:
            best_f = vl_f; torch.save(rete_forza.state_dict(), path_forza)
        if epoca % 5 == 0 or epoca == 1:
            print(f"    Epoca {epoca:02d}/{cfg.n_epoche}"
                  f"  |  RETE xi:    Loss(train)={tl_xi:.5f}  Loss(val)={vl_xi:.5f}"
                  f"  |  RETE forza: Loss(train)={tl_f:.5f}  Loss(val)={vl_f:.5f}")

    # 5) valutazione su traccia isolata
    def predici(rete, path):
        rete.load_state_dict(torch.load(path, map_location=device)); rete.eval()
        out = []
        with torch.no_grad():
            for i in range(0, Nva, bs):
                out.append(rete(AZ_va[i:i+bs], V_va[i:i+bs]).cpu())
        return torch.cat(out).numpy().flatten()

    xi_pred = norm_a_xi(predici(rete_xi, path_xi), auto)
    forza_pred = norm_a_forza(predici(rete_forza, path_forza), cfg)

    print("\n" + "=" * 90)
    print(" VALIDAZIONE SU TRACCIA ISOLATA (mai vista)")
    print("=" * 90)
    rmse_xi = float(np.sqrt(np.mean((y_xi_va - xi_pred) ** 2)))
    rmse_f = float(np.sqrt(np.mean((y_f_va - forza_pred) ** 2)))
    print(f" xi(t)    -> R^2: {r2_score(y_xi_va, xi_pred):.4f} | RMSE: {rmse_xi:.4f} | MAE: {mean_absolute_error(y_xi_va, xi_pred):.4f}")
    print(f" forza(t) -> R^2: {r2_score(y_f_va, forza_pred):.4f} | RMSE: {rmse_f:.1f} N | MAE: {mean_absolute_error(y_f_va, forza_pred):.1f} N")
    print(f" |forza| max prevista dal ML: {np.abs(forza_pred).max():.1f} N  (limite {cfg.forza_max:.0f} N)")

    # comfort IDEALE (riferimento fisico) e comfort REALE del ML in CLOSED-LOOP
    rms_pas_id = np.mean([p for p, o in (comfort_tr + comfort_va)])
    rms_ideale = np.mean([o for p, o in (comfort_tr + comfort_va)])
    az_val, v_val = tracce_va[0]
    nseg = min(len(az_val), int(cfg.comfort_secondi * cfg.freq_campion))
    traj_cl = traiettorie_reali_ml(az_val[:nseg], v_val[:nseg], rete_xi, rete_forza, norm, auto, cfg, device)
    rms_pas = float(np.sqrt(np.mean(traj_cl["acc_cassa_pas"] ** 2)))
    rms_ml = float(np.sqrt(np.mean(traj_cl["acc_cassa_ml"] ** 2)))
    print(f"\n Comfort CLOSED-LOOP (valori del ML, {nseg/cfg.freq_campion:.0f} s):")
    print(f"   passiva {rms_pas:.3f} m/s^2  ->  ML {rms_ml:.3f} m/s^2  ({100*(1-rms_ml/rms_pas):.1f}% meglio)")
    print(f"   [riferimento ideale fisico: {100*(1-rms_ideale/rms_pas_id):.1f}%]")

    # tabella regola √2
    print("\n" + "-" * 66)
    print(" REGOLA √2: xi in funzione del rapporto r = omega/omega_n (target validazione)")
    print("-" * 66)
    bins = [0.0, 1.0, RADQ2, 2.0, 10.0]
    etich = ["r<1", "1<r<√2", "√2<r<2", "r>2"]
    print(f"{'Range r':<10} | {'v [km/h]':<9} | {'xi medio':<9} | {'kp [Ns/m]':<10} | {'|forza| [N]'}")
    for i in range(len(bins) - 1):
        m = (r_va >= bins[i]) & (r_va < bins[i + 1])
        if np.any(m):
            print(f"{etich[i]:<10} | {v_va[m].mean()*3.6:<9.1f} | {y_xi_va[m].mean():<9.3f} | "
                  f"{kp_da_r(r_va[m].mean(), cfg):<10.0f} | {np.abs(y_f_va[m]).mean():.0f}")

    # 6) figura + animazione
    # tenuta di strada ed energia (dal closed-loop gia' calcolato)
    rh_pas = float(np.sqrt(np.mean(traj_cl["defl_gomma_pas"] ** 2)) * 1000)
    rh_ml = float(np.sqrt(np.mean(traj_cl["defl_gomma_ml"] ** 2)) * 1000)
    print(f" Tenuta strada (RMS deflessione gomma): passiva {rh_pas:.2f} mm -> ML {rh_ml:.2f} mm")
    print(f" Energia: damper {traj_cl['E_damp']:.1f} J (dissipata, gratis) | "
          f"attuatore iniettata {traj_cl['E_att_inj']:.1f} J | recuperabile {traj_cl['E_att_rec']:.1f} J")
    if cfg.salva_figura:
        salva_figura(y_xi_va, xi_pred, y_f_va, forza_pred, r_va, rms_pas, rms_ml, cfg, auto,
                     os.path.join(base, cfg.figura_file))
        salva_figura_efficienza(traj_cl, cfg, os.path.join(base, "efficienza_sospensione.png"))
    if cfg.mostra_anim or cfg.salva_video:
        print("\n[3] Animazione confronto (controllo dai valori ML)...")
        if cfg.anim_usa_ml:
            traj = traiettorie_demo_ml(rete_xi, rete_forza, norm, auto, cfg, device)
        else:
            traj = traiettorie_demo_ml(rete_xi, rete_forza, norm, auto, cfg, device)
        anima_confronto(traj, cfg, os.path.join(base, cfg.video_file))

    print("=" * 90)
    print(f"Completato in {time.time() - t0:.1f} s")


if __name__ == "__main__":
    main()

---

> **Nota sull'output:** a differenza della sezione precedente, qui è stata riportata solo la parte di codice differenziale rispetto alla rete CNN a due teste (le modifiche architetturali e di pipeline), non l'implementazione completa con le funzioni/librerie custom di supporto: la cella non è quindi eseguibile end-to-end all'interno del notebook. L'output viene comunque riportato per dimostrare la bontà dei risultati ottenuti con questo approccio, prima che venga abbandonato a favore della versione finale descritta nel capitolo 2.

> Output da terminale della simulazione:

<pre style="font-size:11px; line-height:1.2; overflow-x:auto;">
==========================================================================================
 SOSPENSIONE 2 GDL (PEUGEOT 207) — CONTROLLO APPRESO (ML), REGOLA √2
 omega_n = 9.41 rad/s (1.50 Hz)   |   crossover a r = √2 ≈ 1.414
==========================================================================================
    HW: Apple Silicon GPU (MPS)
    Core: 8 totali -> uso 4 performance (salto 4 efficiency)

[1] Caricamento tracce e generazione etichette (parallela sui core)...
  [ok] 3 tracce reali -> 2 train / 1 validazione (thessaloniki_19_11_2023.pkl)
    Data augmentation: 6 strade sintetiche ISO 8608 + buche/salite
      strada sintetica 1: classe B, v0=29 km/h, RMS a_z=0.78 m/s^2
      strada sintetica 2: classe C, v0=50 km/h, RMS a_z=1.46 m/s^2
      strada sintetica 3: classe C, v0=72 km/h, RMS a_z=2.00 m/s^2
      strada sintetica 4: classe D, v0=90 km/h, RMS a_z=3.62 m/s^2
      strada sintetica 5: classe D, v0=58 km/h, RMS a_z=2.89 m/s^2
      strada sintetica 6: classe E, v0=40 km/h, RMS a_z=4.68 m/s^2
    Campioni TRAIN : 395708  (8 tracce = reali + sintetiche)
    Campioni VALID : 157412  (3a traccia, mai vista)
    Normalizzazione (dai dati di training): Normalizzatore(a_z: media=+0.0019 std=0.9203 | v: media=10.625 std=6.500)

[2] Addestramento DUE reti separate (loss e optimizer indipendenti)...
    Epoca 01/12  |  RETE xi:    Loss(train)=0.00354  Loss(val)=0.00029  |  RETE forza: Loss(train)=0.00091  Loss(val)=0.00040
    Epoca 05/12  |  RETE xi:    Loss(train)=0.00021  Loss(val)=0.00023  |  RETE forza: Loss(train)=0.00026  Loss(val)=0.00028
    Epoca 10/12  |  RETE xi:    Loss(train)=0.00001  Loss(val)=0.00025  |  RETE forza: Loss(train)=0.00018  Loss(val)=0.00030

==========================================================================================
 VALIDAZIONE SU TRACCIA ISOLATA (mai vista)
==========================================================================================
 xi(t)    -> R^2: 0.9859 | RMSE: 0.0072 | MAE: 0.0027
 forza(t) -> R^2: 0.9180 | RMSE: 22.1 N | MAE: 13.1 N
 |forza| max prevista dal ML: 834.3 N  (limite 1500 N)

 Comfort CLOSED-LOOP (valori del ML, 60 s):
   passiva 0.163 m/s^2  ->  ML 0.100 m/s^2  (38.7% meglio)
   [riferimento ideale fisico: 52.6%]

------------------------------------------------------------------
 REGOLA √2: xi in funzione del rapporto r = omega/omega_n (target validazione)
------------------------------------------------------------------
Range r    | v [km/h]  | xi medio  | kp [Ns/m]  | |forza| [N]
r<1        | 23.5      | 0.543     | 1620       | 41
1<r<√2     | 38.2      | 0.463     | 1824       | 63
√2<r<2     | 50.3      | 0.359     | 2075       | 58
 Tenuta strada (RMS deflessione gomma): passiva 0.23 mm -> ML 0.16 mm
 Energia: damper 12.9 J (dissipata, gratis) | attuatore iniettata 2.9 J | recuperabile 3.2 J
    [ok] Figura salvata: risultati_sospensione.png
    [ok] Figura efficienza salvata: efficienza_sospensione.png

</pre>

<table>
<tr>
<td style="width:50%; vertical-align:middle;">
    <img src="img_TCN/efficienza_sospensione.png"
         style="height:300px; width:auto;">
</td>
<td style="width:50%; vertical-align:middle;">
    <img src="img_TCN/risultati_sospensione.png"
         style="height:300px; width:auto;">
</td>
</tr>
</table>

---

<div align="center">
    <video width="100%" controls style="max-width: 900px; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
        <source src="img_TCN/validazione_strada.mp4" type="video/mp4">
    </video>
</div>

---


**Analisi Critica dei Risultati: Distinzione tra Limite ML e Limite Fisico-Matematico**

I risultati della validazione su traccia isolata confermano l'efficacia del disaccoppiamento delle due TCN: le metriche di fitting sono eccellenti ($R^2 = 0.9859$ su $\xi(t)$ e $R^2 = 0.9180$ su $F_{\text{att}}(t)$). Ciononostante, il miglioramento del comfort si stabilizza al **38.7%** (RMS $a_z$: da $0.163\text{ m/s}^2$ a $0.100\text{ m/s}^2$), rimanendo al di sotto del valore di riferimento ideale che ora è aumentato a **52.6%** a causa del cambio di struttura neurale e dell'aumento di GDL del sistema.

Questa divergenza non costituisce un difetto di addestramento o di scelta della loss, ma è l'effetto di precisi **vincoli fisico-matematici e strutturali** a cui abbiamo dovuto arrenderci:

* **Fedeltà di Apprendimento vs Limite del Target:** 
  I valori di $R^2$ superiori a $0.91$ dimostrano che la rete neurale ha appreso quasi perfettamente la legge di controllo generata nei dati di addestramento. Di conseguenza, il tetto del 38.7% rappresenta la **massima prestazione teoricamente estraibile dalla regola $\omega/\omega_n \approx \sqrt{2}$** sotto i vincoli impostati, non un'incapacità della rete di generalizzare.

* **Trade-off Intrinseco del Modello a 2 GDL (Comfort vs Tenuta di Strada):**
  Nel modello a un quarto di veicolo (2 GDL), l'abbattimento dell'accelerazione della massa sospesa (comfort) entra in diretta competizione con la deflessione dinamica dello pneumatico (tenuta di strada). Per evitare il distacco della ruota e mantenere la deflessione contenuta ($0.16\text{ mm}$ contro i $0.23\text{ mm}$ del sistema passivo), la forza dell'attuatore viene modulata e limitata (picco massimo $834.3\text{ N}$ su $1500\text{ N}$ disponibili).

* **Vincoli di Attuazione e Saturazione:**
  Il riferimento ideale del $52.6\%$ ipotizza una politica di controllo *Skyhook* ideale priva di saturazioni fisiche, ritardi di attuazione ed energia limitata. L'inserimento dei vincoli di forza reale e della banda passante dell'attuatore riduce matematicamente il margine massimo di abbattimento RMS.

* **Semplificazione Dinamica del Quarter-Car:**
  Anche il modello a 2 GDL qui usato resta un *quarter-car*: isola il moto traslatorio verticale di un solo angolo del veicolo (cassa + ruota), ignorando i moti di beccheggio (*pitch*) e rollio (*roll*) e le interazioni cinematiche con gli altri tre angoli dell'auto. Tale formalizzazione limita lo spazio degli stati osservabili rispetto a un modello *Full-Car* a 7 GDL, vale a dire che se si mettesse in forma di spazio di stato le equazioni dinamiche e si analizzasse la matrice $O$ probabilmente, a causa della povertà di sensoristica utilizzata, lo spazio non fosse completamente osservabile (dovremmo verificare con MatLab ma rimane plausibile allo stato delle cose).

**Conclusione:**
Avendo separato i modelli ed eliminato il conflitto tra i gradienti nella loss, la rete ha saturato il limite prestazionale della legge fisica di riferimento. Per superare la soglia del 40% occorrerà evolvere la strategia di controllo sottostante (passando da regole euristiche deterministiche tipo legge $\sqrt{2}$ a controllori predittivi MPC/LQR o Reinforcement Learning continuo) o sottoporre il sistema a un modello di veicolo completo (non attuabile, sia perché non abbiamo i dati e verrebbe meno l'ipotesi di usare il numero minore possibile di sensori sia perché il modello fisico diventerebbe più difficile da calcolare che non la stima del controllo a partire della ML).

---

### 1.3 Abbandono del modello - Analisi critica nell'overall - Perché non sono stati usati gli strumenti studiati durante il corso AML
Nel corso dello sviluppo, è emersa la necessità di abbandonare il modello matematico e di controllo utilizzato finora. L'analisi del comportamento in addestramento ha evidenziato diverse criticità strutturali che impediscono al sistema di generalizzare e ottimizzare correttamente le dinamiche del veicolo:

* **Rischio di "Rete Oracolo" e Race Condition:** Utilizzare una regola di scheduling basata sulla trasmissibilità costringe la rete neurale a comportarsi come un semplice oracolo (memorizzando una regola pre-impostata) piuttosto che imparare *de facto* una vera policy di ottimizzazione. Inoltre, la struttura matematica che prevede due forze dipendenti dalla velocità agenti sul medesimo elemento (la massa dell'auto) crea un legame troppo rigido. Anche adottando un modello a 2 GDL che accoppia i termini in maniera diversa, si innesca una sorta di *race condition* (in senso lato: non e' concorrenza software, ma un accoppiamento algebrico mal condizionato) tra le forze in gioco che de facto dipendono da una stessa velocità verticale, rendendo instabile la convergenza dell'algoritmo.

* **Limiti del Dataset Spaziale:** Un dataset di profili stradali, per quanto parzialmente aumentato, si è rivelato insufficiente a descrivere tutte le possibili configurazioni che la rete deve affrontare. Limitarsi a scalare la rugosità equivalente del terreno non mappa correttamente lo spazio degli stati. Per una generalizzazione robusta, è indispensabile sottoporre il modello a strade dissestate, aspre, con creste e valli, salite e discese, simulando l'intero spettro di sollecitazioni reali a velocità di traslazione orizzontali varie ed in combinazione diverse.

* **Vincoli Sensoriali e Accoppiamento Fisico:** La mancanza di una sensoristica completa, che rappresenta poi la vera sfida del progetto, ci impedisce di calcolare un parametro $\lambda_c(t)$ variabile nel tempo in modo continuo e affidabile. Questa incertezza rende impossibile stimare in maniera consistente le grandezze ausiliarie necessarie all'accoppiamento fisico tra componente attiva e semi-attiva, impedendo di saturare l'efficienza teorica del sistema.

* **Potenziale Inespresso del Deep Learning:** Rimanere ancorati a euristiche fisiche troppo restrittive inibisce la capacità intrinseca delle reti neurali di scoprire relazioni non lineari complesse. Liberando la rete dai rigidi vincoli analitici pre-calcolati (che tra l'altro appesantiscono il carico computazionale in inferenza), le permettiamo di trovare dinamicamente compromessi ottimali tra *Ride Comfort* e *Road Holding* che approcci classici non riuscirebbero a identificare, mantenendo la latenza sufficientemente bassa per un'applicazione *real-time*.

* **Valutazione fuorviante su profili a singolo tono:** Aver inizialmente analizzato e calibrato il modello assumendo eccitazioni stradali armoniche a singolo tono ci ha portato a conclusioni del tutto fuorvianti sulle reali capacità del sistema. I manti stradali reali non sono sinusoidi perfette, ma possiedono uno spettro in frequenza a banda larga (*broadband*) , più simili al contenuto armonico di polinomi finestrati con una $w_R(t)$ o a segnali non periodici che armonici. Addestrare o validare il modello su segnali troppo semplici ha creato l'illusione di prestazioni ottimali che crollano non appena si introduce il rumore stocastico tipico dei profili ISO 8608.

* **Inefficacia della soglia cinematica in scenari a banda larga:** Proprio a causa della natura *broadband* delle strade, le frequenze di risonanza della massa sospesa (circa 1-2 Hz) e di quella non sospesa (10-15 Hz) vengono eccitate simultaneamente. Di conseguenza, basare lo scheduling su un singolo punto di incrocio della trasmissibilità (come il limite teorico $r = \sqrt{2}$) perde totalmente di coerenza fisica e matematica. La regola funziona solo se esiste una singola frequenza dominante, ma fallisce miseramente quando l'input (e cioè la forzante di spostamento che è letteralmente il profilo stradale) è una sovrapposizione complessa di armoniche (variabili nello spazio e nel tempo lungo il profilo stradale) perché è come se ci fossero due attrattori frequenziali (le due frequenze di risonanza) che si palleggiano in tali punti "l'autorità nel richiedere di minimizzare" il modulo della funzione di trasferimento nello stesso momento.  
Questo è possibile vederlo anche nel comportamento di $\xi(t)$ ad alte velocità di traslazione dove per regola dovrebbe assumere valori molto piccoli e invece la non causalità di cui parlato poc'anzi fa in modo che la rete cerchi di trovare una correlazione di $\xi(t)$ con l'accelerazione verticale piuttosto che con la velocità di traslazione.



#### 1.3.1 **Le motivazioni per cui non sono stati utilizzati XGBoost e algoritmi Tree-Based**

Sebbene **XGBoost** (Chen e Guestrin, 2016; e in generale gli algoritmi basati su *Gradient Boosted Decision Trees*) sia l'apoteosi per dati tabulari statici, presenta limitazioni teoriche e pratiche fondamentali quando applicato al **controllo dinamico in anello chiuso** di una sospensione:



##### a. Discontinuità dell'Output e Jerk Meccanico
* **XGBoost:** Produce predizioni "a gradini" (funzioni costanti a tratti), poiché divide lo spazio degli stati con nodi di decisione booleani.
* **Impatto sulla Sospensione:** Generare comandi di forza $F(t)$ o coefficienti $\xi(t)$ discontinui crea derivate dell'accelerazione elevate (**jerk**). In un'auto reale, questo si traduce in colpi d'ariete idraulici, sollecitazioni impulsive sull'attuatore, rumore nell'abitacolo e rapida usura meccanica.
* **Deep Learning:** Le reti neurali (con funzioni di attivazione continue come SiLU, GELU o Tanh) approssimano superfici di controllo estremamente **lisce e continue**, garantendo transizioni morbide nelle forze erogate.



##### b. DAgger e Warm-Starting (Fine-Tuning Iterativo)
* **XGBoost:** Non supporta un *fine-tuning* incrementale naturale. Ad ogni iterazione di DAgger, per integrare i nuovi dati raccolti in closed-loop, XGBoost dovrebbe ri-creare l'insieme di alberi da zero oppure aggiungere nuovi alberi in coda, facendo lievitare la dimensione del modello e rischiando l'overfitting sui dati più recenti (e come si può notare l'overfitting era già presente di suo anche senza peggiorare le cose).
* **Deep Learning:** Con PyTorch è sufficiente riprendere i pesi della rete al ciclo precedente e fare un **fine-tuning di poche epoche** sui dati aggregati. Questo garantisce stabilità nell'apprendimento (*warm-start*) e convergenza rapida.




##### c. Latenza e Determinismo su Hardware Automotive (ECU)
* **XGBoost:** L'inferenza richiede di attraversare centinaia o migliaia di alberi di decisione. Nei microcontrollori automotive (ECU), questo tipo di codice causa *branch mispredictions* e accessi alla memoria non sequenziali (*cache miss*), rendendo il tempo di calcolo non deterministico.
* **Deep Learning:** Una rete neurale feed-forward si riduce a semplici **moltiplicazioni matrice-vettore**, eseguibili con latenza rigorosamente costante $O(1)$ e parallelizzabili su DSP/NPU. Il modello è facilmente esportabile in **ONNX** (Open Neural Network Exchange, Microsoft/Facebook, 2017) o convertibile in C puro (es. via codice C puro/TinyEngine) per il controllo real-time a 100-1000 Hz.



##### d. Gestione delle Dipendenze Temporali
* **XGBoost:** Considera ogni finestra temporale come un vettore isolato di feature tabulari, senza comprendere la continuità dello stato fisico sottostante.
* **Deep Learning:** Si presta nativamente a strutturare ingressi con finestre di memoria temporale (es. tramite convoluzioni 1D o strati ricorsivi/attentivi), estraendo caratteristiche spettrali dall'accelerazione di cassa $a_z(t)$ in modo molto più efficace.




#### 1.3.2 **Perché non abbiamo utilizzato algoritmi di Clustering (es. K-Means o DBSCAN, con `scikit-learn`)?**

L'utilizzo del clustering (apprendimento non supervisionato) non è adatto alla gestione di un sistema di controllo attivo/semi-attivo per sospensioni per quattro ragioni teoriche e pratiche fondamentali:




##### a. Natura del Problema: Apprendimento di una Policy vs Partizionamento
* **Clustering (`scikit-learn`):** È una tecnica **non supervisionata**. Il suo scopo è trovare gruppi naturali (*cluster*) nei dati in base alla distanza geometrica nello spazio delle feature (es. prossimità tra $v$ e $a_z$). Ignora completamente qualsiasi obiettivo di prestazione.
* **Il nostro problema:** È un problema di **regressione supervisionata / Imitation Learning**. Avevamo bisogno di una funzione $f(a_z, v) \to \xi$ che mappa la cinematica al valore di smorzamento ottimo $\xi^*$, calcolato dall'oracolo per minimizzare l'accelerazione di cassa e garantire la tenuta di strada. Un algoritmo di clustering non può usare l'etichetta $\xi^*$ per ottimizzare la risposta dinamica.




##### b. Controllo Discreto vs Continuo (*Chatting* e Jerk Meccanico ANCORA)
* **Clustering:** Assegna ciascun punto di lavoro a un centroide discreto $k \in \{1, 2, \dots, K\}$. Usarlo nel controllo significherebbe trasformare la sospensione in un sistema che commuta bruscamente tra $K$ livelli rigidi di smorzamento (es. "Morbido", "Medio", "Rigido").
* **Impatto Meccanico:** La transizione a gradino tra un cluster e l'altro genererebbe forti colpi di commutazione (*chattering*), picchi elevati di **jerk** (derivata dell'accelerazione) e instabilità dinamica durante la guida su asperità stradali.
* **Deep Learning:** Approssima una superficie di controllo **continua e liscia**, permettendo a $\xi(t)$ di variare in modo fluido al variare delle condizioni stradali.




##### c. Incompatibilità con l'Algoritmo DAgger
* **Clustering:** Non ha un concetto di funzione di costo (*loss*) basata sull'errore di predizione dell'azione. Di conseguenza, non può essere integrato nel ciclo iterativo di DAgger per correggere il *covariate shift*.
* **Deep Learning:** Consente di ri-addestrare i pesi tramite retropropagazione dell'errore (SGD/Adam) incorporando i nuovi stati esplorati in anello chiuso dal veicolo.




##### d. Limiti delle Metriche di Distanza per Dati Temporali
* **Clustering con `sklearn`:** Algoritmi come K-Means usano la distanza euclidea standard tra vettori di feature. Trattano le accelerazioni $a_z(t)$ come punti statici nello spazio $N$-dimensionale, perdendo la correlazione temporale e la struttura spettrale delle vibrazioni.
* **Deep Learning:** Le reti neurali (specialmente con finestre temporali o architetture convolutive/ricorsive) estraggono rappresentazioni interne delle frequenze di sollecitazione, riconoscendo se la strada sta eccitando la frequenza di cassa (~1 Hz) o quella delle masse non sospese (~10-12 Hz).


---

## 2 — Il nuovo sistema utilizzato ed il modello ATTUALE

Il caso di studio prende di nuovo come riferimento la cinematica e la dinamica verticale del veicolo **Peugeot 207**, accantonato il progetto iniziale allora, e ne eredita l'**architettura ibrida di controllo**:

1. **Modulo Semi-Attivo:** Regola continuamente il coefficiente di smorzamento $c_{var}(t)$ dello smorzatore per assorbire l'energia cinetica e stabilizzare le oscillazioni dovute al manto stradale non uniforme.
2. **Modulo Attivo:** Genera una forza controllata $F_a(t)$ per compensare attivamente le sollecitazioni provenienti dall'eccitazione di tipo spostamente alla base della ruota e mantenere la risposta dinamica entro la zona di comfort ottimale in tutto il range frequenziale. 

### 2.1 Premesse e Ipotesi di Modellazione

Per la formalizzazione matematica e la successiva simulazione, si adottano le seguenti ipotesi semplificative:

* **Modello Quarter-Car (2 GDL):** Il modello è un **quarter-car**: un quarto di automobile, cioè una ruota e la porzione di cassa che le grava sopra. Due masse impilate, collegate da molla, ammortizzatore variabile e attuatore; il pneumatico è a sua volta una molla verso il suolo e dunque il veicolo viene modellato attraverso il classico sistema a 2 gradi di libertà intero (massa sospesa $m_s$ e massa non sospesa $m_u$). Si trascurano momentaneamente i moti di rollio e beccheggio.  
* **Elasticità Lineare dello Pneumatico:** Lo pneumatico è schematizzato come una molla lineare di rigidezza $k_t$, privo di smorzamento interno.
* **Saturazione dell'Attuatore:** L'attuatore attivo presenta un limite fisico di forza erogabile $F_a \in [-F_{max}, F_{max}]$ e una banda passante limitata.
* **Profilo Stradale:** Gli ingressi delle forzanti di tipo spostamento stradali $z_r(t)$ considerati includono:
  1. *Profilo deterministico:* Gradino/Bump sinusoidale per la risposta ai transitori.
  2. *Profilo stocastico:* Discontinuità modellate secondo la normativa **ISO 8608:2016** *"Mechanical vibration — Road surface profiles"* (classi di strada A–E, coerentemente con la griglia usata poi in §2.5) descritte esaustivamente da codici internazionali ricavabile facilmente online (verrano discusse dopo le dinamiche).  

---
 



### 2.2 Schema del sistema (Quarter-Car a 2 GDL)
Un angolo di Peugeot 207 anteriore stilizzato può essere visto come segue (non bisogna commettere l'errore di pensare che molla,smorzatore e attuatore siano "in serie" poiché sono tutte 3 evidentemente in parallelo ed in serie alla massa del cielo):

$$
\begin{array}{lll}
\text{cassa} \quad m_s = 260\text{ kg} & z_s(t) & \leftarrow \text{accelerometro appoggiato al cielo (cassa): misura } a_z(t) = \ddot{z}_s(t) \\
\quad \parallel \quad \text{molla } k_s = 23\text{ kN/m} & & \\
\quad \parallel \quad \text{ammortizzatore variabile } c(t) & & \leftarrow \text{comando 1 da costruire} \\
\quad \parallel \quad \text{attuatore } F_{att} & & \leftarrow \text{comando 2 da costruire} \\
\text{ruota} \quad m_u = 38\text{ kg} & z_u(t) & \\
\quad \parallel \quad \text{pneumatico } k_t = 190\text{ kN/m} & & \\
\text{strada} & z_r(t) & \leftarrow \text{il disturbo, sconosciuto, di tipo spostamento}
\end{array}
$$

---



### 2.3 Tabelle di riferimento per la nomenclature delle variabili e dei parametri del modello (Quarter-Car a 2 GDL)

| Simbolo | Significato | Unità |
|---|---|---|
| $z_r(t)$ | quota della **strada** sotto la ruota — è l'**ingresso/disturbo** del sistema | m |
| $z_u(t)$ | quota della **ruota** (*unsprung*, massa non sospesa) | m |
| $z_s(t)$ | quota della **cassa** (*sprung*, la carrozzeria) | m |
| $\dot z_s(t)$ | velocità (derivata prima): velocità verticale della cassa | m/s |
| $\ddot z_s(t)$ | accelerazione (derivata seconda): $\ddot z_s(t) = a_z(t)$ accelerazione della cassa | m/s² |
| $a_z(t)$ | accelerazione verticale della cassa seconda nomenclatura — **ciò che l'accelerometro misura** | m/s² |
| $v = v(t)$ | velocità di avanzamento del veicolo (GPS) — il secondo ingresso sensore | m/s |
| $v_{rel}(t)$ | $= \dot z_s(t) − \dot z_u(t)$, velocità relativa di sospensione | m/s |
| $m_s$, $m_u$ | massa cassa = 260 kg, massa ruota = 38 kg | kg |
| $k_s$, $k_t$ | rigidezza molla = 23 000, rigidezza pneumatico = 190 000 | N/m |
| $c(t)$ | smorzamento dell'ammortizzatore, **variabile nel tempo**, in [400, 3500] | Ns/m |
| $c_{crit}$ | $= 2·√(k_s·m_s)$ = 4891 Ns/m — smorzamento critico della cassa | Ns/m |
| $ξ(t)$ | $= c / c_crit$, fattore di smorzamento adimensionale — **primo comando** | – |
| $F_{att}(t)$ | forza dell'attuatore, saturata a ±1500 N — **secondo comando** | N |
| $k_p$ | guadagno skyhook: $F = clip(−k_p · \dot z_s(t))$, in [400, 6000] | Ns/m |
| $ω_n$ | $= √(k_s/m_s)$ = 9.405 rad/s = **1.497 Hz** — risonanza della cassa | rad/s |
| $ω_u$ | $= √((k_s+k_t)/m_u)$ = 74.87 rad/s = **11.9 Hz** — *wheel hop*, risonanza della ruota | rad/s |
| $r$ |  $= ω/ω_n$ — dove sta l'eccitazione rispetto alla risonanza (solo regola √2) | – |
| $a_{z,w}$ | accelerazione **pesata Wk** secondo ISO 2631-1 — l'indice di comfort | m/s² |

**Annotazioni:**

**Range effettivo del damper**  
$\xi$ non è libero: dai dati si ricava che $\xi_{min} = c_{min}/c_{crit} = 400/4891 = 0.0818$ e $\xi_{max} = c_{max}/c_{crit} = 3500/4891 = 0.7156$. Il valore passivo nominale, cioè l'ammortizzatore di serie, è $\xi_{nom} = 1500/4891 = 0.307$. Tutte le uscite delle reti sono vincolate a questo intervallo per costruzione (Sigmoid scalata), non per penalizzazione: **il controllore non può chiedere un damper che non esiste** perché violerebbe i limiti costruttivi meccanici del damper reale (e se si chiedesse $\xi<0$ si violerebbe la passività del sistema, dato che un ammortizzatore reale può solo dissipare energia, mai fornirla).  

**Cosa si intende RMS per segnali non sinusoidali puri**    
$RMS(x) = \sqrt{(media(x^2))}$, il valore efficace: l'ampiezza tipica di un segnale che oscilla attorno a zero e non è il picco diviso √2 — quella relazione vale solo per una sinusoide pura: su segnali a banda larga il rapporto picco/RMS si **misura** tramite la formula.

**Disambiguizione sulle accelerazioni in gioco**  
Le tre accelerazioni che compaiono nei grafici e nelle stampe:

* **acc_cassa_pas** — veicolo **passivo**: $c = c_{nom}$ fisso, nessun attuatore. È il riferimento originale da battere tramite il controllore (e l'apprendimento neurale).  
* **acc_cassa_ml** — veicolo **controllato dalle reti**, in closed-loop.
* **acc_ideale** — controllo di riferimento applicato allo **stato vero** (non stimato). È il best case teorico: nessun controllore causale che stima lo stato può fare meglio. Tutte e tre girano nello stesso simulatore in **anello chiuso** (RK4, §2.4): cambia solo chi decide $(\xi, F)$ a ogni passo — si veda §4.1 per la distinzione fra questa modalità (usata per misurare) e l'addestramento supervisionato (che invece è open-loop, su dati precalcolati).


---



### 2.4 Modello Fisico ed Equazioni

Stato integrato $x = [x_1, x_2, x_3, x_4]$, scelto in **coordinate relative** affinché i due
vincoli meccanici siano direttamente due componenti dello stato in maniera comoda per i conti:

$$
\begin{array}{l l l}
x_1 = z_s(t) - z_u(t) & \text{corsa della sospensione} & [\text{vincolo: fine corsa}] \\
x_2 = \dot{z}_s(t) & \text{velocità della cassa} & [\text{obiettivo: comfort}] \\
x_3 = z_u(t) - z_r(t) & \text{deflessione del pneumatico} & [\text{vincolo: distacco ruota}] \\
x_4 = \dot{z}_u(t) & \text{velocità della ruota} & [\text{sensoristica implicita derivante da GPS, auto e ruota traslano a $v_x(t)$ }]
\end{array}
$$



Equazioni del moto derivanti dall'equilibrio dinamico, notando che il sistema è intrinsecamente non lineare ed a variabili accoppiate:

$$
\begin{cases}
m_s \ddot{z}_s = -k_s x_1 - c(t)(x_2 - x_4) + F_{att} \\
m_u \ddot{z}_u = k_s x_1 + c(t)(x_2 - x_4) - F_{att} - k_t x_3
\end{cases}
$$



#### 2.4.1 Metodo di Integrazione Numerica

Per la risoluzione numerica del sistema dinamico è stato adottato l'algoritmo di **Runge-Kutta al 4° ordine (RK4)** (Runge, 1895; Kutta, 1901). A fronte di dati acquisiti a una frequenza di campionamento derivante dai dataset di $f_s = 100\text{ Hz}$ (equivalente a un periodo di campionamento di $T_s = 10\text{ ms}$), è stata implementata una suddivisione interna con un fattore di sotto-passo pari a $n_{sottopassi} = 2$.

Questo configura un passo di integrazione effettivo pari a:
$$h = \frac{T_s}{n_{\text{sottopassi}}} = \frac{10\text{ ms}}{2} = 5\text{ ms}$$



#### 2.4.2 Motivazione della scelta del passo
La dinamica più veloce del sistema è associata al moto della massa non sospesa (rimbalzo della ruota o *wheel hop*), la cui frequenza caratteristica si attesta a circa $f_{\text{ruota}} \approx 11.9\text{ Hz}$ (corrispondente a un periodo proprio $T_{\text{ruota}} \approx 84\text{ ms}$).

* **Criticità del passo a $10\text{ ms}$:** Se si fosse utilizzato direttamente il passo di campionamento di $10\text{ ms}$ senza sotto-passi, si sarebbero ottenuti soltanto $\approx 8.4$ punti per ogni periodo di oscillazione della ruota. Un campionamento così rado risulta insufficiente per un metodo esplicito ad alto ordine come RK4, esponendo la simulazione a significativi errori di fase e a potenziali instabilità numeriche.
* **Vantaggio dell'integrazione a $5\text{ ms}$ ($n_{sottopassi} = 2$):** Raddoppiando la risoluzione interna si ottengono circa $17$ punti per periodo, garantendo la convergenza numerica, la stabilità dell'algoritmo e una rappresentazione fedele dei transitori ad alta frequenza.

---



### 2.5 Il profilo stradale: DataSet reali & Physics-Based Data Augmentation

Un ostacolo strutturale e critico riscontrato nella fase iniziale dello sviluppo è emerso dall'analisi del dataset di profili stradali reperito su Kaggle: il dataset iniziale constava di appena 3 viaggi registrati, di cui due fortemente ridondanti e sovrapponibili per caratteristiche dinamiche. Una base dati così drammaticamente ristretta ha palesato tutti i classici fallimenti tipici del machine learning quando si opera in regime di estrema povertà informativa: 

* **Overfitting estremo e "memoria":** Con un numero così basso di campioni, la rete neurale o il modello di regressione smette di apprendere le leggi fisiche del sistema e si limita a fare un lookup mascherato, memorizzando sequenzialmente i singoli profili come se fossero tabelle rigide.  
* **Andamenti della Loss anomali:** Durante il training, le curve di perdita (loss function) mostrano comportamenti paradossali: la loss di addestramento crolla a zero in poche epoche (dando un'illusione di convergenza perfetta), mentre la loss di validazione oscilla selvaggiamente o diverge non appena si testa il modello su un profilo anche minimamente differente. 
* **Il paradosso del coefficiente $R^2$:** Si ottengono metriche apparentemente eccellenti sulla carta (con un $R^2$ prossimo a 1 o errori quadratici medi irrealisticamente bassi), che però rappresentano un indicatore puramente fittizio: il modello è perfetto solo su quei tre micro-scenari ma perde totalmente validità predittiva nel mondo reale non appena si fa validazione su una strada che non è stata coperta dai casi di training.  
* **Esplosione di Falsi Positivi e Negativi:** La totale assenza di variabilità statistica spinge il sistema a generare un'enorme quantità di falsi positivi (segnalazioni di eventi anomali inesistenti su strade regolari) e falsi negativi (incapacità di riconoscere buche o transitori severi non presenti nei dati di training). 

**La Soluzione: Physics-Based Data Augmentation (ISO 8608)**  
Per uscire da questa situazione di stallo e impedire al modello di crollare in produzione, si è abbandonata la dipendenza dei soli dati empirici scarsi implementando una strategia di potenziamento basato sulla modellazione fisica (physics-based data augmentation) da giustapporre al dataset reale, simulando per varie fasce di velocità i tracciati stradali sintetici.   

Invece di addestrare su registrazioni casuali, sono stati generati profili stradali sintetici rigorosamente modellati secondo lo standard internazionale ISO 8608 (Mechanical vibration — Road surface profiles). Attraverso la generazione di densità spettrali di potenza (PSD) conformi alla normativa, è stato possibile sintetizzare un set massiccio di migliaia di chilometri di strade virtuali che coprono l'intero spettro delle classi di asperità (modulo per python trovato su github):  

* **Classe A e B:** Strade autostradali e urbane scorrevoli in condizioni eccellenti o buone.  
* **Classe C e D:** Strade extraurbane ordinarie con rugosità media e discontinuità frequenti.  
* **Classe E:** Percorsi degradati, sterrati e superfici ad alto contenuto energetico — il limite superiore di rugosità effettivamente generato nella griglia di training (lo standard ISO 8608 definisce classi fino a H, ma qui si è usato A–E, coerentemente con `aug_classi` in `config.py`). 

Questo flusso continuo di dati sintetici statisticamente validati ha costretto il modello a imparare la vera fisica del quarter-car anziché memorizzare tre file CSV, azzerando l'overfitting e garantendo risposte robuste, stabili e prive di falsi allarmi in qualsiasi condizione operativa reale.   

Come accennato le strade sintetiche seguono **ISO 8608**: densità spettrale di potenza $G_d(n) = G_d(n_0) \cdot (n/n_0)^{-2}$, con `n` frequenza spaziale in cicli/m, $n_0 = 0.1$ cicli/m, e $G_d(n_0)$ fissato dalla classe A–E. La pendenza $-2$ è la ragione fisica per cui la rugosità cambia l'**ampiezza** dello spettro ma non la sua **forma**: due strade di classe diversa percorse alla stessa velocità eccitano il veicolo nelle stesse bande, con intensità diverse.

Per le tracce **registrate** proveniente dal data di Kaggle la strada non è nota e viene ricostruita da $a_z(t)$ per doppia integrazione con passa-alto a 0.3 Hz, poi calibrata in ampiezza (`calibra_strada=True`) imponendo che il modello passivo riproduca l'RMS di $a_z(t)$ misurato. Il fattore di scala è stato limitato a $[0.2, 20]$ per sicurezza numerica perché era l'unico modo di ottenere forze in Newton consistenti con il modello fisico realistico invece che in unità arbitrarie.


---



### 2.6 Riflessioni sui dati provenienti dai dataset



#### 2.6.1 **Tracce registrate proveniente da KaggleHub**

Il set di dati empirici preliminari, acquisito tramite kagglehub "nickkotarelas/road-quality-dataset", si compone di 3 tracce stradali registrate via IMU , che sono state suddivise in 2 tracce destinate al training e 1 traccia di validazione mantenuta rigorosamente inedita.  
Il contenuto del dataset è organizzato come segue tramite una funzione inspector:  

```python
{
    'imu': {
        'accel': {
            'x': np.ndarray[float],     # in m/s^2
            'y': np.ndarray[float],     # in m/s^2
            'z': np.ndarray[float]      # in m/s^2
        },
        'gyro': {
            'x': np.ndarray[float],     # in rad/s
            'y': np.ndarray[float],     # in rad/s
            'z': np.ndarray[float]      # in rad/s
        },
        'time': {
            'rel': np.ndarray[float],       # in sec
            'abs': np.ndarray[np.int64]     # in nanosec
        }
    },
    'gps': {
        'lon': np.ndarray[float],       # in degrees
        'lat': np.ndarray[float],       # in degrees
        'speed': np.ndarray[float],     # in m/s
        'track': np.ndarray[float],     # in degrees
        'time': {
            'rel': np.ndarray[float],       # in sec
            'abs': np.ndarray[np.int64]     # in nanosec
        }
    },
    'camera': {
        'data': np.ndarray[bytes],      # JPEG bytes
        'time': {
            'rel': np.ndarray[float],       # in sec
            'abs': np.ndarray[np.int64]     # in nanosec
        }
    },
    'labels': [   # OPTIONAL
        {
            'rel_t_start': float,   # in sec
            'rel_t_end': float,     # in sec
            'anomaly': str,         # 'no_anomaly', 'manhole', 'depression', 'bump', 'crack'
            'transversity': str,    # 'no_transverse', 'transverse'
            'severity': str         # 'small_severity', 'medium_severity', 'high_severity'
        }
    ]
}
```

--- 



L'analisi approfondita di queste registrazioni dopo aver estratto i dati ha evidenziato una criticità fondamentale tipica dell'acquisizione di accelerometri di bordo non troppo performanti, ed è stato indispensabile un rigoroso passaggio di preprocessing:  

* **Verifica della Cadenza e Jitter Temporale:** Sebbene la frequenza di campionamento nominale registri una mediana di $100.1 \text{ Hz}$, il segnale reale soffre di un jitter compreso tra il $13\%$ e il $15\%$. Questa fluttuazione è causata dalla natura asincrona dei sistemi operativi mobili nella gestione dei buffer dei sensori e dello scheduling dei processi.  
* **Ricampionamento su Griglia Uniforme:** Per neutralizzare l'anomalia, la fase di caricamento implementa una verifica puntuale della cadenza seguita da un ricampionamento forzato su una griglia temporale a passo costante.  

**Pericoli dei Failure Silenziosi (Senza Pre-processing)**
L'eventuale omissione del ricampionamento avrebbe introdotto errori subdoli e silenziosi all'interno della pipeline di simulazione, compromettendo l'affidabilità dei risultati senza attivare eccezioni nel codice:

1. **Fallimento dei Filtri di Butterworth:** Un segnale temporale non uniforme fornito in ingresso a filtri digitali IIR (come i filtri Butterworth (Butterworth, 1930) impiegati per l'estrazione delle bande di frequenza stradale) subisce una variazione dinamica della frequenza di taglio effettiva, generando distorsioni di fase e attenuazioni spettrali incontrollate.
2. **Errori di Scala nelle Integrazioni Numeriche:** Algoritmi come il metodo Runge-Kutta che stiamo utilizzando per risolvere il sistema dinamico richiedono un intervallo temporale rigorosamente costante ($dt$). La presenza di un passo variabile non regolarizzato altera i coefficienti di accumulo nei transitori, falsificando le scale di ampiezza e di energia del modello quarter-car in modo del tutto invisibile alle metriche standard di errore: catastrofe nel fornire dati poi alla rete neurale.  

#### 2.6.2 **Strade sintetiche come disegno sperimentale**

Le tracce registrate sono tutte guida urbana a velocità simile su fondo relativamente buono e dunque mancano interamente i casi che decidono fondo sconnesso e alta velocità.  
Per questo motivo sono state costruite 48 strade sintetiche ISO 8608 che riempiono quel vuoto, e sono costruite come **disegno sperimentale**, non come semplice generatore di dati casuale usando funzioni armoniche e serie di fourier.  

Il motivo è che $RMS(a_z(t)) \approx \sqrt{\text{rugosità} \cdot v}$ e dunque se nel dataset rugosità e velocità variano insieme lungo il tragitto dell'auto, sono statisticamente indistinguibili e **nessuna rete può separarle** e tantomeno addestrarsi.  
Da qui tre requisiti per la generazione delle strade per costruire un dataset ausiliario completo:  

1. **Griglia fattoriale** classe × velocità (5 × 5 = 25 celle), corner compresi: classe E a 100 km/h e classe A a 20 km/h devono esistere entrambe. Le celle di validazione sono **disgiunte** da quelle di training, così la validazione misura generalizzazione e non impara a memoria "le regole".
2. **Velocità variabile dentro la traccia.** Metà delle tracce sono rampe, sweep o stop-and-go che percorrono la *stessa* strada a velocità molto diverse. È il dato più informativo del dataset: rugosità costante, firma spettrale costante, controllo ottimo che cambia.
3. **Jitter di rugosità** indipendente dalla classe. Senza, l'RMS di $a_z(t)$ sarebbe una funzione deterministica di (classe, v) e quindi un identificatore della traccia: la rete potrebbe riconoscere *quale* traccia sta guardando invece di stimare la strada e si ritorna al problema del training.

#### 2.6.3 **Vincoli di campionamento**

Tutti i dataset sintetici e reali vengono fatti lavorare a 100 Hz (frequenza di campionamento delle strade reali per consistenza) quindi il contenuto generato deve essere rappresentabile: poiché il problema è generalizzabile come un onda nello spazio e nel tempo, la frequenza spaziale massima è limitata a "$\text{margine} \cdot \frac{f_s}{v_{max}}$" (altrimenti ad alte velocità di traslazione del veicolo il profilo supera la frequenza di Nyquist e l'aliasing riporta energia spuria proprio dove vive il modo della cassa, ovvero si impesta la banda utile di frequenze che non esistono), la larghezza minima di un ostacolo è ~2.5 passi spaziali, e pendenza dei fianchi e ampiezza dell'orografia sono limitate da vincoli fisici sensati (una valle o una cresta non possono essere piu strette in valore assoluto della lunghezza di un'auto media altrimenti ci si blocca nello spazio).
  

---



## 3 — Funzioni Ottime & Controllo



### 3.1 Il problema di riferimento: cosa deve imparare la rete



#### 3.1.1 **Il modulo "Esperto" e la logica di riferimento**

In questo approccio le reti neurali **non imparano direttamente una legge di controllo**. Il loro obiettivo è apprendere un riferimento calcolato da un modulo "esperto", costruito esplicitamente. Questo modulo dispone di informazioni che la rete non possiede, cioè il profilo stradale esatto e la sua evoluzione nella finestra considerata, e per ogni istante risolve il problema di progetto:

$$
(\xi^*,k_p^*)=
\arg\min
\mathrm{RMS}_{W_k}(a_z)
$$

soggetto ai vincoli

$$
\mathrm{RMS}(z_u-z_r)\leq\text{limite}
$$

$$
\mathrm{RMS}(z_s-z_u)\leq\text{limite}
$$

$$
\mathrm{RMS}(F)\leq\text{limite}
$$

dove:

* $\mathrm{RMS}(z_u-z_r)$ limita la deflessione del pneumatico, evitando il distacco della ruota;
* $\mathrm{RMS}(z_s-z_u)$ limita l'escursione della sospensione, evitando il fine corsa;
* $\mathrm{RMS}(F)$ limita la forza dell'attuatore, evitando la saturazione.

L'ottimo viene cercato tramite **enumerazione su una griglia** di coppie $(\xi,k_p)$, simulate in parallelo. Per ogni finestra mobile si individua quindi la coppia costante che avrebbe minimizzato il comfort sul tratto appena percorso. Questa è la formulazione classica del punto di progetto di un sistema **gain scheduling** (si veda la rassegna di Rugh e Shamma, 2000).



#### 3.1.2 **$\xi$ e $k_p$ si ottimizzano insieme**

Il coefficiente di smorzamento $\xi$ e il guadagno dell'attuatore $k_p$ agiscono entrambi sulla stessa massa sospesa.

Ottimizzarli separatamente produce un sistema incoerente: il damper viene scelto assumendo un certo comportamento dell'attuatore, mentre l'attuatore viene scelto assumendo un comportamento del damper che non è quello realmente utilizzato. Si può così ottenere un damper morbido dove l'attuatore è debole, oppure un attuatore molto aggressivo associato a uno smorzamento elevato.

L'ottimo di ciascun parametro dipende inoltre dal valore assunto dall'altro, per cui un'ottimizzazione separata diventa circolare.

Per questo motivo il problema viene risolto cercando direttamente la coppia

$$
(\xi^*,k_p^*)
$$

e non i due parametri in modo indipendente.

---



### 3.2 Perché una somma pesata non basta



#### 3.2.1 **Nuova formula di costo da usare nei layer**

Una formulazione apparentemente naturale consiste nel minimizzare una funzione di costo del tipo

$$
J=
w_1\,\text{comfort}
+
w_2\,\text{gomma}
+
w_3\,\text{corsa.}
$$

Questa scelta però non è adatta per due motivi indipendenti.



##### 1. Le grandezze non sono commensurabili

I termini rappresentano quantità fisiche con unità di misura diverse (m/s², metri e Newton). I pesi $w_i$ devono quindi compensare tali differenze dimensionali e finiscono per diventare parametri arbitrari che determinano direttamente la soluzione.

Ad esempio, raddoppiare $w_2$ modifica il valore di $\xi^*$: in questo modo l'ottimo non viene realmente calcolato, ma scelto implicitamente attraverso i pesi.



##### 2. Il sistema è omogeneo di grado 2

Il modello quarter-car è lineare nello stato. Se l'ampiezza del profilo stradale raddoppia, raddoppiano anche tutte le RMS coinvolte nel costo.

Di conseguenza tutti i termini della somma scalano nello stesso modo e il punto di minimo rimane invariato. L'ottimo risulterebbe quindi indipendente dalla rugosità della strada.

Questo è in contrasto con la fisica del problema: su una strada liscia è possibile privilegiare il comfort, mentre su una strada molto sconnessa occorre aumentare lo smorzamento per evitare il distacco della ruota.

Una semplice somma pesata è quindi **cieca rispetto alla variabile che dovrebbe modificare il controllo**, cioè la rugosità del fondo stradale.



#### 3.2.2 **Il ruolo dei vincoli fisici**

L'omogeneità viene spezzata dall'introduzione di **limiti assoluti**, che non scalano con l'ampiezza della strada.

I principali sono:

* la ruota perde il contatto quando

$$|z_u-z_r|>d_{\max} =\frac{(m_s+m_u)g}{k_t}=15.38\ \text{mm}$$

* la sospensione raggiunge il fine corsa quando (da modello auto)

$$|z_s-z_u| > d^{\text{corsa}}_{\max}=80\ \text{mm}$$

supera il *rattle space* disponibile;

* l'attuatore satura quando

$|F|>F_{\max}=1500\ \text{N}$

Queste soglie sono espresse in metri o Newton e rappresentano limiti fisici assoluti. Su una strada liscia possono non essere mai raggiunte, mentre aumentando la rugosità diventano progressivamente attive.

È proprio l'attivazione di questi vincoli a modificare la soluzione ottima. Di conseguenza

$$
\xi^*=f(\text{velocità},\text{rugosità}),
$$

e la rugosità diventa l'informazione che la rete deve imparare a ricavare dall'accelerazione verticale.




#### 3.2.3 **Picco dei segnali ed RMS**

I limiti fisici del sistema sono definiti sui **valori di picco**, mentre la funzione di costo utilizza le **RMS**. È quindi necessario convertire i limiti di picco in limiti RMS tramite il **fattore di picco**

$$
p=\frac{\max|x|}{\mathrm{RMS}(x)}.
$$

Il fattore di picco **non viene assunto**, ma misurato direttamente sui segnali di ogni traccia.

* $\sqrt{2}\approx1.41$ è il valore valido per una sinusoide, ma la risposta del quarter-car a un profilo ISO non è sinusoidale e usarlo sottostimerebbe il picco di oltre un fattore due.
* $p=3$ è la convenzione dei tre sigma per un processo gaussiano: rappresenta una buona approssimazione, ma non è il valore reale.
* Sulle simulazioni il fattore di picco vale tipicamente **3.3–3.7** per la deflessione del pneumatico e **3.0–3.4** per la corsa della sospensione.

*Nota metodologica:* la norma **ISO 2631-1:1997** prescrive il metodo RMS di base (quello usato qui per $J_{\text{comfort}}$) quando il fattore di picco del segnale $a_z(t)$ resta sotto **9**; oltre quella soglia richiede invece il *Vibration Dose Value* (VDV, quarta potenza) perché la RMS sottostima l'impatto di urti isolati. I fattori di picco misurati qui (3.0–3.7) restano ben al di sotto di 9, quindi il metodo RMS di base è appropriato e non serve VDV.

Poiché il fattore di picco dipende solo debolmente dalla coppia $(\xi,k_p)$, viene utilizzata la **mediana** dei valori misurati. Usare il fattore del candidato in esame renderebbe infatti il criterio circolare, perché il costo finirebbe per definire il proprio metro di misura.

---



### 3.3 La funzione di costo finale



#### 3.3.1 **La funzione di costo globale**

L'obiettivo dell'ottimizzazione è minimizzare il comfort percepito, misurato tramite l'accelerazione verticale pesata secondo lo standard **ISO 2631-1:1997** *"Mechanical vibration and shock — Evaluation of human exposure to whole-body vibration"* (filtro $W_k$). La funzione di costo utilizzata è

$$
J=
\frac{\mathrm{RMS}_{W_k}(a_z)}
{a_{\mathrm{rif}}}
+
\lambda
\sum_i
\max\left(
0,
\frac{\mathrm{RMS}_i}{\mathrm{limite}_i}-1
\right)^2,
\qquad
\lambda=1000.
$$

La funzione è composta da due termini.




#### 3.3.2 **Termine di comfort**

Il primo termine

$$
J_{\text{comfort}} =\frac{\mathrm{RMS}_{W_k}(a_z)}
{a_{\mathrm{rif}}}
$$

misura il comfort secondo la norma ISO 2631-1.

La normalizzazione tramite $a_{\mathrm{rif}}$ rende il costo adimensionale.




#### 3.3.3 **La penalità (penalizzazione) nelle funzioni di costo: Le *hinge***

Il secondo termine introduce i vincoli tramite **penalità a cerniera (*hinge*)** (concetto mutuato dalla hinge loss delle SVM: Cortes e Vapnik, 1995).

Per ciascun vincolo viene definita la penalità

$$
P_i=
\max\left(
0,
\frac{\mathrm{RMS}_i}{\mathrm{limite}_i}-1
\right)^2.
$$

Il comportamento è il seguente:

* se il vincolo è rispettato,

$$
\mathrm{RMS}_i\le\mathrm{limite}_i,
$$

allora

$$
P_i=0;
$$

* se il limite viene superato,

$$
\mathrm{RMS}_i>\mathrm{limite}_i,
$$

la penalità cresce quadraticamente.

Di conseguenza:

* **nella regione ammissibile** tutte le penalità sono esattamente nulle e

$$
J=J_{\text{comfort}};
$$

* **fuori dalla regione ammissibile** anche una piccola violazione produce un aumento molto elevato del costo.

Il coefficiente

$$
\lambda=1000
$$

è scelto sufficientemente grande da rendere dominante qualsiasi violazione dei vincoli. Per questo motivo, pur essendo implementati come penalità, i vincoli si comportano di fatto come **vincoli rigidi**.

L'uso della *hinge loss* ha inoltre un vantaggio numerico importante: mantiene la funzione di costo continua. L'ottimo $(\xi^*,k_p^*)$ varia quindi in modo continuo al variare della strada, invece di cambiare bruscamente quando un vincolo si attiva.



#### 3.3.4 **Perché la *hinge* è fondamentale come penalizzazione nell'equazione di costo**

L'aspetto fondamentale non è soltanto penalizzare le violazioni, ma **rompere l'omogeneità** del problema.

Con una semplice somma pesata tutte le componenti del costo scalano allo stesso modo al crescere dell'ampiezza del profilo stradale, quindi il punto di minimo rimane invariato.

Le penalità *hinge*, invece, dipendono da **soglie assolute**:

* distacco della ruota;
* fine corsa della sospensione;
* saturazione dell'attuatore.

Queste soglie non scalano con la rugosità della strada. Quando una di esse viene raggiunta, la relativa penalità si attiva e modifica la funzione di costo, spostando la soluzione ottima.

È quindi l'attivazione dei vincoli a rendere

$$
\xi^*=f(\text{velocità},\text{rugosità}),
$$

mentre una semplice somma pesata produrrebbe un ottimo sostanzialmente indipendente dalla rugosità.



#### 3.3.5 **Comportamento della funzione costo nella ricerca dell'ottimo**

Il riferimento è una funzione **di due variabili** e si legge come tre regimi:

- **strada liscia, qualunque velocità** — nessun vincolo attivo: ξ* va al valore che minimizza
  il discomfort, cioè verso il basso;
- **strada rotta, velocità crescente** — si attiva il vincolo sulla gomma: ξ* **sale** per
  tenere la ruota a terra, sacrificando comfort;
- **velocità molto alta su strada molto rotta** — si attiva anche la corsa: ξ* sale ancora.

Questa dipendenza dalla rugosità è ciò che era assente nella regola $\sqrt{2}$, e la ragione per cui
la mappa $\xi(v(t), a_z(t))$ della diagnostica è il grafico più importante del progetto.

---



## 4 — Le reti Neurali



### 4.1 **Comportamento delle reti neurali**
* **In addestramento** il modello fisico calcola le etichette e nient'altro. Le reti vedono solo una finestra di $a_z$ più la velocità $v$, entrambi segnali sensore reali. Il modello fisico **non entra** nella rete.
* **In inferenza** il modello fisico **sparisce**. Le reti lo sostituiscono: leggono $a_z$ e $v$, producendo $\xi$ e $F$. Non si inietta nulla.
* Non è un oracolo: la frequenza e la velocità verticale della cassa non vengono passate; le reti le devono dedurre dai campioni grezzi.
* Dentro le reti **non c'è nessun loop nascosto**: sono *feedforward*, un passaggio ingresso-uscita senza stato interno né ricorrenza. Il loop, quando c'è, è **esterno** alla rete: dentro non c'è nulla che ricordi il passo precedente.

**Una precisazione terminologica prima di procedere.** "Closed-loop" qui è usato nel senso della *teoria del controllo* — lo schema controllore/impianto in retroazione — non nel senso di teoria dei segnali, dove basta che un sensore prelevi un'uscita e la riporti nell'ingresso di un blocco a valle. Il ruolo di ciascun blocco è preciso:

| Blocco | Chi è | Cosa fa |
|---|---|---|
| **Impianto** | il quarter-car (RK4, §2.4) | riceve $\xi(t)$ e $F(t)$, restituisce lo stato $\vec x(t)=[z_s-z_u,\dot z_s,z_u-z_r,\dot z_u]$ |
| **Sensore** | accelerometro + GPS | misura $a_z(t)=\ddot z_s(t)$ e $v(t)$ — **non** lo stato $x(t)$, solo una sua funzione parziale e rumorosa |
| **Controllore** | le due reti | da $a_z(t), v(t)$ decide il comando $(\xi(t), F(t))$ |
| **Attuatore** | damper variabile + forza attiva | traduce il comando in $c(t)$ e $F(t)$ reali, chiudendo il cerchio sull'impianto |

La retroazione è quindi il percorso $a_zt(),v(t) \to$ controllore $\to (\xi(t),F_a(t)) \to$ attuatore $\to$ impianto $\to a_z(t{+}1) \to \dots$, che si richiude fisicamente a ogni campione. Non è *tracking* di un riferimento come un servomeccanismo: il "controllore" è più vicino a un **regolatore ottimo** (nello spirito di un LQR, §1) che minimizza un costo — il comfort — invece di inseguire un setpoint.

<div align="center">
    <img src="img/open_closed_loop.png" style="width: 92%; max-width: 950px; height: auto; margin: 14px 0;">
</div>

Un'ultima distinzione, utile perché riprende un filo già aperto in §1: questa è retroazione sull'**uscita**, non sullo **stato**. Le reti fanno *output feedback* — vedono solo $a_z(t), v(t)$ — non *state feedback* come un controllore che leggesse $x(t)$ per intero. È la stessa idea degli osservatori alla Luenberger citati in §1: un controllore state-feedback "vero" stimerebbe esplicitamente $x(t)$ in un blocco dedicato, mentre qui la rete impara direttamente la mappa $(a_z(t),v(t))\to(\xi(t),F_a(t))$ e incorpora quella stima implicitamente nei suoi pesi. I controllori "ottimo" e "√2" usati come riferimento, quando girano con `usa_riferimento=True`, **leggono invece lo stato vero** $x_1=\dot z_s(t)$ per il termine skyhook (`simulazione.py: _closed_loop`): sono controllori state-feedback non realizzabili, un limite superiore teorico (coerente con "acc_ideale", §2.3), non un confronto alla pari con le reti.

**I tre casi concreti**, tutti istanze dello stesso schema — cambia solo se e quando l'anello si chiude davvero:

1. **Addestramento supervisionato (§5) — ciclo APERTO.** Le finestre di $a_z(t), v(t)$ vengono da tracce precalcolate (modello passivo $c=c_{\text{nom}}$, o traccia reale registrata) e restano fisse: l'uscita della rete non torna mai indietro a cambiare l'ingresso. È training/validazione da dataset in senso classico — nessuna retroazione, la rete non pilota nulla.
2. **Simulazione closed-loop (§6) — ciclo CHIUSO, solo per misurare, a rete già addestrata.** L'anello `rete → (ξ,F) → RK4 → nuova a_z → rete` si richiude a ogni campione: la $a_z(t)$ letta al passo dopo è conseguenza fisica della decisione al passo prima. Nessun peso si aggiorna qui — è così che si misurano comfort/tenuta (§6.2) e si generano le animazioni.
3. **Rollout DAgger (§4.5) — ciclo CHIUSO, ma per RACCOGLIERE dati, non per misurare o addestrare.** Stesso anello del punto 2, usato solo per scoprire dove la rete si porta da sola; quegli stati vengono poi rietichettati dall'esperto e il riaddestramento che segue torna a essere un ciclo APERTO sul dataset così aggregato (punto 1).

### 4.2 **Tronco comune**

Entrambe le reti utilizzano un'architettura **TCN** (convoluzioni 1D a dilatazione crescente), che garantisce un ampio campo ricettivo con un numero ridotto di parametri. 

* **Riduzione di risoluzione:** Le riduzioni sono interleavate fra i livelli ($200 \to 100 \to 50 \to 25$ campioni) e le dilatazioni vengono **riscalate dopo ogni riduzione** ($1, 2, \downarrow, 4, \downarrow, 8, \downarrow, 8$), permettendo al campo ricettivo in campioni della finestra originale di continuare a crescere anziché restringersi.
  * *Quantificazione (calcolo analitico dai layer di `reti.py`, formula standard del campo ricettivo per CNN dilatate):* il tronco arriva a un campo ricettivo teorico di **≈350 campioni (3.5 s)**, superiore sia ai 200 campioni (2.0 s) della finestra $\xi$ sia ai 100 campioni (1.0 s) della finestra forza. In pratica l'ultimo strato convoluzionale vede già l'intera finestra disponibile: il collo di bottiglia informativo è la lunghezza della finestra stessa, non la profondità della rete.
  * *Dimensione dei modelli (conteggio analitico dei parametri, stessa fonte):* il tronco TCN condiviso pesa **≈35 mila parametri**; con le teste di fusione e uscita, la rete $\xi$ arriva a **≈166 mila parametri** e la rete forza (due teste) a **≈187 mila**, per un totale di **≈350 mila parametri** su entrambe le reti — un ordine di grandezza compatibile con l'inferenza a centinaia di Hz su hardware embedded citata più sopra.
* **Operatore compatibile (MPS):** Le riduzioni sono implementate tramite `MediaMobile`, una convoluzione *depthwise* con pesi fissi $1/2$ e *stride 2*, numericamente identica a `AvgPool1d` ma espressa con un operatore supportato nativamente dal backend **MPS** di Apple Silicon.
* **Fusione dei rami:** I due ingressi confluiscono in `FusioneDueRami` con capacità bilanciata ($64$ canali dal ramo $a_z + 64$ dal ramo $v$), evitando che la velocità domini la fusione per il solo fatto di essere uno scalare pulito.
* **Normalizzazione:** Gli ingressi sono normalizzati con media e deviazione standard **calcolate sui dati di training** tramite `dati.Normalizzatore`, serializzato in `normalizzatore.json` per garantire coerenza assoluta con la diagnostica.

---



### 4.3 **Rete $\xi$ — Baseline fisico più correzione appresa**

La rete non produce direttamente $\xi$, ma calcola una correzione applicata a un riferimento strutturato:

$$\xi = \xi_{\min} + (\xi_{\max} - \xi_{\min}) \cdot \text{sigmoid}\left( L(v) + \alpha \cdot \tanh(f(a_z(t), v(t))) \right)$$

* **Il baseline $L(v)$:** Rappresenta il valore costante corrispondente al damper più morbido (dove cade l'ottimo di comfort quando i vincoli non mordono), con un margine del $3\%$ (`xi_margine_logit`, poiché il logit diverge esattamente sul bordo). La rete impara così *solo l'irrigidimento richiesto dalla tenuta di strada*, isolando una quantità dotata di preciso significato fisico.
* **Il ruolo di $\alpha$:** **$\alpha$ è un tetto, non una statistica da centrare.** Si tara sul massimo scarto fra etichette e baseline (misurato in logit) più un margine. Tararlo su un percentile taglierebbe via per costruzione le strade peggiori, che sono proprio quelle che decidono la tenuta. 
  * *Sintomo di errata taratura:* Con un tetto troppo basso la rete **satura** e si blocca sullo stesso valore ($\text{sigmoid}(L + \alpha)$) su ogni riga e colonna della mappa diagnostica.
  * *Ramping:* Durante l'addestramento $\alpha$ cresce da un valore iniziale ridotto (`autorita_az_iniziale = 0.3`) fino al valore pieno (`autorita_az = 7.0`), gestito automaticamente da `main.py` tramite `scarto_dal_baseline()` ($max \cdot 1.10$).

Vale a dire che:
* **$L(v)$ (Baseline Fisico):** Smorzamento nominale deterministico in funzione della velocità del veicolo (es. Lookup Table).
* **$f(a_z, v)$ (Rete Neurale):** Stima l'aggiustamento dinamico sulla base delle accelerazioni verticali $a_z(t)$ e della velocità $v(t)$.
* **$\alpha \cdot \tanh(\cdot)$ (Correzione Bounded):** Limita l'intervento dell'IA nell'intervallo $[-\alpha, +\alpha]$.
* **$\text{sigmoid}(\cdot)$ & $[\xi_{\min}, \xi_{\max}]$:** Normalizzazione e riscalamento per rispettare rigorosamente i limiti meccanici dell'ammortizzatore (*VINCOLO*).

---



### 4.4 **Rete forza — Due uscite che ricompongono l'etichetta**

L'etichetta di riferimento per la forza è un prodotto non lineare soggetto a saturazione:

$$F^* = \text{clip}\left( -k_p^*(t) \cdot \dot{z}_s(t) \right)$$

Chiedere a una singola rete di apprendere direttamente questo prodotto significa costringerla a mediare fra regimi dinamici completamente differenti:
1. $\dot{z}_s$ è un problema di **stima di stato** (ottenuto integrando $a_z$: segnale liscio, a media nulla, con $\text{RMS} \approx 0.37 \text{ m/s}$).
2. $k_p^*$ è un problema di **schedulazione** (dipendente dalla rugosità stradale, con code pesanti).

Per evitare performance mediocri su entrambi i fronti, la rete sfrutta **due teste distinte** (`forza_due_uscite = True`):

$$\begin{aligned}
\text{testa\_{zs}} &\to \hat{z}_s' \in [-2, 2] \text{ m/s} \quad (\text{usa come funzione di attivazione Tanh(x)}) \\
\text{testa\_{kp}} &\to \hat{k}_p \in [k_{p,\min}, k_{p,\max}] \quad (\text{usa come funzione di attivazione Sig(x)}) \\
F &= \text{clip}\left( -\hat{k}_p \cdot \hat{z}_s', \pm F_{\max} \right)\\
\end{aligned}$$



#### 4.4.1 **Vantaggi dell'architettura a due teste di uscita:**
* **Distribuite e trattabili:** Ciascuna uscita possiede una distribuzione statisticamente gestibile.
* **Supervisione diretta:** Entrambi i target intermedi sono noti ($k_p^*$ dall'ottimizzatore, $\dot{z}_s$ dal filtro di Kalman — Kálmán, 1960). `controllo.py` li esporta come target ausiliari pesati da `peso_aux_forza = 0.3`. Senza queste loss ausiliarie esisterebbero infinite coppie di valori il cui prodotto è corretto ma i singoli termini errati, facendo perdere il significato fisico alla scomposizione.
* **Ispezionabilità:** Permette di diagnosticare con precisione chirurgica quale dei due canali stia eventualmente introducendo errori.
* **Saturazione strutturale:** La saturazione fisica dell'attuatore $F_{\max}$ è imposta deterministicamente fuori dalla rete, dove compete per natura.



#### 4.4.2 **Configurazione temporale:**
* `seq_len_forza = 100` rispetto a `seq_len = 200` usato per $\xi$: la forza dipende dalla velocità istantanea della cassa, rendendo una finestra più lunga un fattore di mero *overfitting* (verificato sperimentalmente: estendere la finestra della forza a 200 faceva crollare il comfort da $0.041$ a $0.119$).
* `passo_finestre = 1`: nessun sottocampionamento delle finestre di training.  

---



### **4.5 Imitation Learning Avanzato: L'Algoritmo DAgger**

Per risolvere i limiti di esplorazione della rete e il problema della deriva degli stati, abbiamo integrato nello script `aggregazione.py` l'algoritmo **DAgger** (Dataset Aggregation). 

La teoria alla base di DAgger è stata sviluppata nel 2011 da **Stéphane Ross, Geoffrey J. Gordon e J. Andrew Bagnell**. Nasce per risolvere il problema del *covariate shift* nel Behavioral Cloning: se una rete viene addestrata solo sulle traiettorie perfette di un esperto, un minimo errore in esecuzione la porterà in uno stato mai visto durante il training. Non sapendo come gestire la situazione, l'errore si accumula portando il sistema al collasso. DAgger risolve questo problema costringendo la rete a imparare come correggere i propri errori.

Il principio di base è iterativo e si articola in questi passaggi implementati in `aggregazione.py`:

1. Si lascia "guidare" il veicolo alla rete neurale parzialmente addestrata (un ciclo di epoche di training è già avvenuto a monte).
2. Si registrano gli stati (le cinematiche) che la rete visita *realmente* a causa delle proprie imprecisioni.
3. Si interroga l'esperto (il nostro modello matematico ottimo, cioè xi_ottimo) per sapere quale azione avrebbe intrapreso in quegli specifici stati anomali.
4. Si rietichettano queste nuove finestre di dati e le si **aggrega** (senza sostituire) al dataset originale.
5. Si riaddestra la rete sul dataset espanso (di nuovo un passo **open-loop**: la rete non pilota nulla durante il riaddestramento, si limita a fare supervised learning sulle finestre appena raccolte).

#### 4.5.1 Spiegazione Intuitiva: Chi fa cosa nel passo 2–3

**1. Chi crea la situazione (Lo Stato)**
Durante l'addestramento ausiliario del DAgger (l'addestramento principale è già avvenuto prima OFFLINE), a "guidare" la sospensione è l'IA. Le conseguenze fisiche delle sue decisioni (quanto rimbalza l'auto $z_s$, i movimenti della ruota $z_u$, ecc.) non sono inventate, ma vengono calcolate istante per istante dal simulatore matematico (il solutore **RK4**).

**2. Chi dà le risposte corrette (Le Etichette)**
A ogni passo, l'IA propone una sua soluzione. Di fianco a lei c'è un "Maestro" perfetto (lo script `xi_ottimo.py`) che guarda la situazione e fornisce l'etichetta, ovvero dice all'IA: *"Guarda che il valore corretto da usare qui era questo"*. L'IA usa questa risposta per correggere i propri pesi (imparare).

**3. L'Asimmetria**
* **Lo "Stato $\vec{x}(t)$":**
  * L'allievo (la rete neurale) viene messo al volante.
  * Il simulatore fisico ("RK4") calcola come si muove la macchina e dunque lo stato del sistema (composto da $z_s(t), z_u(t), a_z(t)$, cioè i movimenti della cassa dell'auto e delle ruote) mentre la rete neurale prova a controllarla.
  * A volte la rete guida da sola, a volte viene aiutata dal maestro (il parametro $\beta$ esplica quanta percentuale della simulazione è guidata dall'AI e quanto dall'esperto). Ma il punto è che l'auto si muove in base a come decide (o sbaglia) la rete.

* **L' "Etichetta" (La risposta corretta del maestro):**
  Per imparare, la rete ha bisogno che il maestro le dica la correzione. Qui c'è un'asimmetria importante:
  * **Prima cosa da imparare ($\xi^*$):** Dipende *solo* dalle buche della strada ($z_r$). Visto che la strada è la stessa a prescindere da chi guida, il maestro calcola la risposta perfetta una volta sola a inizio giro e basta.
  * **Seconda cosa da imparare ($F^*$ - la forza da applicare):** La formula è $F^* = -k_p^*(t) \cdot \dot{z}_s(t)$. Tradotto: dipende dalla velocità in cui la cassa dell'auto sta andando su e giù ($\dot{z}_s$).


**Morale della favola: La "Rietichettatura fatta dall'esperto"**  
Dato che la rete neurale sta guidando, sicuramente farà degli errori e porterà la macchina a sobbalzare in modi in cui il maestro perfetto non l'avrebbe mai portata.
Quindi, il maestro non può usare la soluzione che aveva pensato all'inizio e deve guardare il casino che ha appena fatto la rete (lo stato *davvero* visitato) e calcolare una nuova correzione su misura per quella specifica situazione.   
La strada è uguale per tutti, quindi una parte del calcolo si fa una volta sola. Ma siccome l'IA guida peggio dell'algoritmo perfetto, l'auto si muoverà in modo diverso dal previsto; quindi, il calcolo della forza da applicare ($F^*$) va rifatto continuamente in base agli errori che l'IA sta commettendo in quel momento.

---

Nel nostro setup specifico, il ciclo esegue **5 iterazioni**, simulando per **20 secondi** e riaddestrando per **2 epoche** ad ogni ciclo (`dagger_epoche`: 4 epoche per giro è stato provato e tarato sulle curve di validazione, risultando peggiore — vedi commento in `config.py`). Viene utilizzato un parametro di mixing $\beta = 0.4$. Al primo giro il controllo è affidato interamente all'esperto ($\beta = 1$) per raccogliere la distribuzione bersaglio iniziale, dopodiché $\beta$ decade per cedere il controllo alla rete e forzarla a esplorare i propri errori.

Solitamente, interrogare l'esperto per rietichettare i dati è il collo di bottiglia di DAgger. Nel nostro caso, tuttavia, l'operazione risulta estremamente efficiente per due motivi:

* **Esperto automatico:** L'oracolo non è un operatore umano o un simulatore lento, ma una regola analitica istantanea.
* **Indipendenza dal controllo della variabile $\xi^*$:** Il rapporto di smorzamento ottimo $\xi^*$ dipende esclusivamente dalle caratteristiche del profilo stradale e non dal controllore. Di conseguenza, non deve essere ricalcolato a ogni iterazione.
* **Rietichettatura On-Policy efficiente:** L'unica grandezza che cambia è l'ingresso di velocità della cassa $\dot{z}_s$ causato dalle azioni della rete. Pertanto, è sufficiente ricalcolare l'etichetta della forza target come $F = -k_p^* \cdot \dot{z}_s$ basandosi sulla cinematica effettiva. Questa è l'esatta definizione di rietichettatura *on-policy* richiesta dall'algoritmo.

---

## 5 — Il codice

**Modularizzazione e Gestione delle Dipendenze**

L'evoluzione progressiva dell'architettura della rete neurale, unita alla definizione inline di numerose funzioni helper, funzioni di perdita personalizzate e classi di supporto, ha portato a un'eccessiva stratificazione del codice all'interno del notebook, compromettendone la leggibilità e rendendo complessa la gestione del progetto.

Per ripristinare la chiarezza strutturale e facilitare la manutenzione, si è reso necessario un total refactoring basato su tre interventi principali:

* **Estrazione in Moduli Python Esterni:** Tutta la logica di supporto — inclusa la prototipazione delle architetture neurali, le routine di preprocessing dei dati e i solutori per il modello dinamico — è stata ingegnerizzata ed estratta in librerie e moduli `.py` dedicati esterni al notebook.
* **Risoluzione Dinamica dei Percorsi per Jupyter:** Per garantire il corretto caricamento di tali componenti, all'interno del notebook è stato integrato un sistema di gestione automatica dei percorsi di ambiente (tramite `sys.path`). All'avvio, il codice individua ed esplora autonomamente le sotto-cartelle di progetto per importare tutte le dipendenze locali requisite, consentendo un'esecuzione fluida e trasparente su qualsiasi ambiente Jupyter senza errori di importazione. 
* In alternativa basta runnare il file `main.py` direttamente dalla cartella dove si trovano questi documenti dentro IDE a scelta (è stato utilizzato VScode nel nostro caso): in entrambi i casi le dipendenze vengono cercate dal codice da solo.


**Tabella riassuntiva dei prototipi e delle librerie costruite**
| File | Contenuto |
|---|---|
| `config.py` | parametri fisici (`Auto`) e di progetto (`Config`), ognuno etichettato per provenienza: [FISICO] [DERIVATO] [DATI] [STANDARD] [MANOPOLA] [NUMERICO] [GRAFICA] |
| `fisica.py` | modello 2 GDL, RK4, ricostruzione strada, stima frequenza, Kalman, Hampel |
| `xi_ottimo.py` | **l'esperto**: problema vincolato, griglia, filtro Wk, riferimenti |
| `controllo.py` | generazione delle etichette; regola √2 come baseline |
| `strade_sintetiche.py` | strade ISO 8608 con disegno fattoriale, buche, dossi, pendenze |
| `dati.py` | caricamento, uniformazione, finestre, `Normalizzatore` |
| `reti.py` | `ReteXiFisica`, `ReteForza`, tronco TCN, `MediaMobile`, `FusioneDueRami` |
| `perdite.py` | δ robusto da MAD, pesi stratificati, `PerditaForza` |
| `simulazione.py` | closed-loop singolo e batchato, strade demo, energie |
| `aggregazione.py` | DAgger: rollout, finestre, β-mixing, validazione closed-loop |
| `diagnostica.py` | mappe ξ(v, a_z), distribuzioni, statistiche, sensibilità, report txt |
| `contesto.py` | etichette di provenienza per stampe, report e figure |
| `grafica.py` | figura risultati, figura efficienza, animazione con selettore |
| `hardware.py` | scelta del device, verifica degli operatori sul backend |
| `portabilita.py` | UTF-8 su stampe e file di testo, suggerimenti dipendenti dal sistema |
| `main.py` | orchestratore della pipeline in 7 passi |

**Architettura.** Gli import a livello di modulo si stratificano in 4 livelli senza cicli:

```
livello 0 =>   config  contesto  fisica  hardware  perdite  portabilita  reti
livello 1 =>   controllo  dati  simulazione  strade_sintetiche
livello 2 =>   diagnostica  grafica  xi_ottimo
livello 3 =>   aggregazione  main
```

---

In [ ]:
# -*- coding: utf-8 -*-
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import r2_score, mean_absolute_error

from config import Auto, Config
from portabilita import descrivi_sistema, forza_utf8
from hardware import rileva_hardware
from dati import carica_tracce, costruisci_finestre, Normalizzatore, nome_traccia
from controllo import GeneratoreEtichette, kp_da_r, RADQ2
from reti import crea_reti, xi_a_norm, norm_a_xi, forza_a_norm, norm_a_forza
from simulazione import (traiettorie_demo_ml, traiettorie_demo_confronto,
                         traiettorie_reali_ml, simula_closed_loop, bilancio_energetico)
from strade_sintetiche import (genera_tracce_sintetiche, genera_tracce_validazione,
                              genera_tracce_aggregazione,
                               riepilogo_copertura)
from fisica import ricostruisci_strada, rms_accel_passiva
from grafica import salva_figura, salva_figura_efficienza, anima_confronto
from diagnostica import esegui_diagnostica


def main():
    t0 = time.time()
    auto, cfg = Auto(), Config()
    # Cerca __file__ per jupyter altrimenti sclera
    base = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()

    path_xi = os.path.join(base, "rete_xi.pt")
    path_forza = os.path.join(base, "rete_forza.pt")
    path_norm = os.path.join(base, "normalizzatore.json")

    print("=" * 90)
    legge = ("OTTIMO VINCOLATO (comfort ISO 2631 sotto i limiti hardware)"
             if cfg.metodo_xi == "ottimo" else "REGOLA DELLA TRASMISSIBILITA' √2")
    print(" SOSPENSIONE 2 GDL (PEUGEOT 207) — CONTROLLO APPRESO (ML)")
    print(f" Legge di riferimento (etichette): {legge}")
    print(f" omega_n = {auto.puls_nat_cassa:.2f} rad/s ({auto.puls_nat_cassa/(2*np.pi):.2f} Hz)"
          f"   |   crossover a r = √2 ≈ {RADQ2:.3f}")
    print(f" Sistema: {descrivi_sistema()}")   # nei log condivisi serve sapere su che macchina
    print("=" * 90)

    device, n_core = rileva_hardware(cfg)

    # 1-2) dati + etichette + finestre
    print("\n[1] Caricamento tracce e generazione etichette (parallela sui core)...")
    tracce_tr, tracce_va = carica_tracce(cfg)
    if cfg.metodo_xi == "ottimo":
        from xi_ottimo import riferimenti, verifica_wk, descrivi_riferimenti
        ar, dr, sr, fr = riferimenti(auto, cfg)
        # AUTOTEST del filtro ISO 2631 Wk contro la Tabella 3 della norma. Era una funzione
        # scritta e mai chiamata: un filtro di ponderazione sbagliato non da' errori, rende
        # solo ottimo il problema sbagliato — esattamente il genere di guasto silenzioso che
        # conviene far urlare all'avvio.
        _f, _g, _t = verifica_wk()
        _err = float(np.max(np.abs(_g - _t) / _t))
        print(f"    Filtro ISO 2631-1 Wk verificato sulla Tabella 3 della norma: "
              f"errore massimo {_err:.2%}" + ("" if _err < 0.02 else "   [!] CONTROLLARE"))
        print("    Etichette da OTTIMIZZAZIONE VINCOLATA: si cerca la coppia (xi, kp) che")
        print("    minimizza il comfort ISO 2631 Wk restando entro i limiti hardware.")
        # ogni limite stampato con la sua FORMULA e la sua provenienza: i valori qui sotto
        # usano il fattore di picco di ripiego, quello vero viene misurato per traccia
        for _r in descrivi_riferimenti(auto, cfg):
            print(_r)
        print(f"    Griglia {cfg.xi_ott_n_candidati}x{cfg.kp_ott_n_candidati}"
              f" = {cfg.xi_ott_n_candidati*cfg.kp_ott_n_candidati} candidati,"
              f" finestra {cfg.xi_ott_finestra_s:.0f} s, penalita' {cfg.xi_ott_penalita:.0f}")
    else:
        v_ind = np.sqrt(2.0) * auto.puls_nat_cassa * cfg.lambda_c_design / (2 * np.pi)
        print(f"    Regola √2: r = omega/omega_n (crossover FISSO a √2); omega = 2π·v/lambda_c, "
              f"lambda_c = {cfg.lambda_c_design:.0f} m -> r=√2 a ~{v_ind * 3.6:.0f} km/h")
    tracce_va_reali = tracce_va
    nome_val = nome_traccia(tracce_va_reali[0], "traccia reale di validazione")
    if cfg.usa_augmentation:                         # aggiunge strade sintetiche varie al TRAINING
        print(f"    Data augmentation: {cfg.n_tracce_sintetiche} strade sintetiche ISO 8608"
              f" (buche/dossi bruschi, gradini, salite/discese)")
        riepilogo_copertura(auto, cfg)
        tracce_tr = tracce_tr + genera_tracce_sintetiche(auto, cfg)
        # VALIDAZIONE anche su strade sintetiche, su celle (classe x velocita') MAI viste:
        # e' il test che smaschera la scorciatoia "a_z -> xi", perche' presenta combinazioni
        # rugosita' x velocita' assenti dal training (es. strada sconnessa a 90 km/h).
        tracce_va_sint = genera_tracce_validazione(auto, cfg)
    else:
        tracce_va_sint = []

    az_tr, v_tr, y_xi_tr, y_f_tr, r_tr, comfort_tr, kp_tr, zs_tr = costruisci_finestre(
        tracce_tr, auto, cfg, n_core)
    az_va, v_va, y_xi_va, y_f_va, r_va, comfort_va, kp_va, zs_va = costruisci_finestre(
        tracce_va_reali, auto, cfg, n_core)
    if tracce_va_sint:
        (az_vs, v_vs, y_xi_vs, y_f_vs, r_vs, comfort_vs,
         kp_vs, zs_vs) = costruisci_finestre(tracce_va_sint, auto, cfg, n_core)
    else:
        az_vs = v_vs = y_xi_vs = y_f_vs = r_vs = kp_vs = zs_vs = None; comfort_vs = []
    print(f"    Campioni TRAIN      : {len(az_tr)}  ({len(tracce_tr)} tracce = reali + sintetiche)")
    # SATURAZIONE DELL'ETICHETTA DI FORZA. F* = clip(-kp*·z_s', ±F_max): quando l'ottimo
    # chiede piu' di quanto l'attuatore possa dare, l'etichetta viene TAGLIATA e insegna alla
    # rete a stare incollata al limite. Senza questo numero, un "|F| max previsto = 1214 N su
    # 1500" sembra la rete che esagera, mentre puo' essere il riferimento che chiede troppo.
    # Misurato sulle strade sintetiche: |F*| max tocca esattamente 1500 N gia' dalla classe B
    # a 76 km/h, e su classe E a 97 km/h l'11% dei campioni e' saturo con RMS 657 N contro un
    # limite ammesso di 455 N — li' il problema di progetto e' INFATTIBILE con questo
    # attuatore, e l'etichetta e' il male minore, non l'ottimo.
    _sat = float(np.mean(np.abs(y_f_tr) >= 0.999 * cfg.forza_max))
    print(f"    Etichetta FORZA     : |F*| max {np.abs(y_f_tr).max():.0f} N su un limite di"
          f" {cfg.forza_max:.0f} N  |  satura nel {_sat:.2%} dei campioni")
    if _sat > 0.01:
        print(f"                          [!] oltre l'1%: su quei campioni l'ottimo vorrebbe piu'"
              f" forza di quanta ce n'e'.")
        print(f"                          L'etichetta li' e' un compromesso, e il picco predetto"
              f" dal ML va letto contro quello del target, non contro {cfg.forza_max:.0f} N.")
    print(f"    Campioni VALID reale: {len(az_va)}  (3a traccia registrata, mai vista)")
    if az_vs is not None:
        print(f"    Campioni VALID sint.: {len(az_vs)}  ({len(tracce_va_sint)} strade su celle "
              f"classe x velocita' DISGIUNTE dal training)")

    # 3) NORMALIZZAZIONE calcolata SOLO dal training (media/dev-std che emergono dai dati)
    norm = Normalizzatore(az_tr, v_tr)
    norm.salva(path_norm)        # viaggia con i .pt: la diagnostica non deve piu' inventarlo
    print(f"    Normalizzazione (dai dati di training): {norm}")
    print(f"    Salvata in {os.path.basename(path_norm)}")

    def su_device(a):
        return torch.tensor(a, dtype=torch.float32, device=device)

    # ingressi normalizzati + target scalati sui limiti fisici (xi->range damper, forza->F_max)
    AZ_tr = su_device(norm.na(az_tr)).unsqueeze(1); V_tr = su_device(norm.nv(v_tr)).unsqueeze(1)
    XI_tr = su_device(xi_a_norm(y_xi_tr, auto)).unsqueeze(1); F_tr = su_device(forza_a_norm(y_f_tr, cfg)).unsqueeze(1)
    AZ_va = su_device(norm.na(az_va)).unsqueeze(1); V_va = su_device(norm.nv(v_va)).unsqueeze(1)
    XI_va = su_device(xi_a_norm(y_xi_va, auto)).unsqueeze(1); F_va = su_device(forza_a_norm(y_f_va, cfg)).unsqueeze(1)
    if az_vs is not None:
        AZ_vs = su_device(norm.na(az_vs)).unsqueeze(1); V_vs = su_device(norm.nv(v_vs)).unsqueeze(1)

    # 4) DUE reti separate, loss e optimizer indipendenti
    print("\n[2] Addestramento di DUE reti separate (Loss() e Optimizer indipendenti)...")
    rete_xi, rete_forza = crea_reti(cfg, auto)
    # la rete xi vincolata deve conoscere le statistiche di v per ricostruire la
    # schedulazione fisica L(v) dal proprio ingresso normalizzato
    rete_xi.imposta_normalizzazione(norm)

    # alpha SUGGERITO DAI DATI: alpha e' un TETTO, non una statistica da centrare. Si tara
    # sul MASSIMO scarto etichetta-baseline (in logit, la scala su cui agisce la correzione)
    # con un margine, perche' la sigmoide raggiunge il bordo solo asintoticamente. Tararlo
    # su un percentile lascia fuori portata la coda — e la coda sono le strade sconnesse,
    # cioe' i casi che decidono la tenuta. Con metodo_xi="sqrt2" lo scarto e' zero per
    # costruzione e alpha non serve.
    alpha_fin = float(cfg.autorita_az)
    if cfg.xi_vincolo_fisico:
        from xi_ottimo import scarto_dalla_regola
        sc = scarto_dalla_regola(y_xi_tr, v_tr, auto, cfg)
        print(f"    Scarto etichette dal baseline '{sc['baseline']}' [logit]: mediana {sc['mediana']:.2f}"
              f" | p90 {sc['p90']:.2f} | p99 {sc['p99']:.2f} | max {sc['massimo']:.2f}")
        print(f"    -> alpha suggerito (max scarto +10%) = {sc['alpha_suggerito']:.2f}"
              f"   |   alpha finale in config = {alpha_fin:.2f}")
        if alpha_fin < sc["alpha_suggerito"]:
            print(f"    [!] alpha finale TROPPO PICCOLO: la correzione satura e la rete non"
                  f" puo' raggiungere xi*. Alzo a {sc['alpha_suggerito']:.2f}.")
            alpha_fin = sc["alpha_suggerito"]
        print(f"    Rete xi = sigmoid(L(v) + alpha*Delta(a_z,v)) con alpha da"
              f" {cfg.autorita_az_iniziale:.2f} a {alpha_fin:.2f} (curriculum)")
    else:
        print("    Rete xi LIBERA (ablation): nessun baseline fisico, nessun limite alla"
              " correzione")
    rete_xi.to(device); rete_forza.to(device)
    # lr e weight decay vengono da config, non scritti qui: sono manopole di taratura e
    # devono stare dove si guardano, insieme al motivo per cui valgono quel che valgono
    opt_xi = optim.AdamW(rete_xi.parameters(), lr=cfg.lr_xi, weight_decay=cfg.wd_xi)
    opt_forza = optim.AdamW(rete_forza.parameters(), lr=cfg.lr_forza, weight_decay=cfg.wd_forza)
    print(f"    Regolarizzazione: rete xi lr={cfg.lr_xi:.0e} wd={cfg.wd_xi:.0e}"
          f"  |  rete forza lr={cfg.lr_forza:.0e} wd={cfg.wd_forza:.0e}")

    # LOSS CON PARAMETRI RICAVATI DAI DATI (vedi perdite.py per il perche' e per la formula).
    # xi: MSE PESATA per strato. Il target non ha code — ha una massa concentrata al minimo
    #     del damper, cioe' un problema di sbilanciamento, che si aggredisce coi pesi e non
    #     con la robustezza (un Huber con delta ampio resterebbe sempre nel ramo quadratico,
    #     cioe' sarebbe MSE con un altro nome).
    # forza: Huber con delta ricavato dalla MAD del target. Qui le code sono vere e la
    #     saturazione hardware produce valori estremi legittimi che non devono dominare.
    from perdite import (delta_robusto, descrivi_scelta, pesi_per_strato,
                         MSEPesata, PerditaForza, HuberPesata)
    _f_n = forza_a_norm(y_f_tr, cfg)
    _kp_n = kp_tr / max(cfg.kp_max_ott, 1e-9)
    _zs_n = zs_tr / max(cfg.zs_max, 1e-9)
    d_f, i_f = delta_robusto(_f_n, k=2.0)
    d_zs, i_zs = delta_robusto(_zs_n, k=2.0)
    d_kp, i_kp = delta_robusto(_kp_n, k=2.0)
    print(descrivi_scelta("forza", d_f, i_f))
    print("    loss xi: MSE pesata per strato (target senza code, ma con una massa al "
          f"minimo del damper: serve il peso, non la robustezza)")
    loss_xi = MSEPesata()
    loss_forza = PerditaForza(d_f, d_zs, d_kp, peso_aux=cfg.peso_aux_forza)
    # per la VALIDAZIONE si usa solo il termine su F: le ausiliarie sono un vincolo di
    # coerenza interna, includerle renderebbe i valori non confrontabili fra configurazioni
    loss_forza_val = HuberPesata(d_f)
    due_uscite = bool(getattr(cfg, "forza_due_uscite", True))

    Ntr, Nva, bs = AZ_tr.size(0), AZ_va.size(0), cfg.batch

    # SELEZIONE DEL MODELLO: la validazione usata per il best-checkpoint unisce la traccia
    # reale mai vista E le strade sintetiche su celle (classe x velocita') disgiunte. Cosi'
    # il criterio di stop premia chi generalizza a combinazioni rugosita' x velocita' nuove,
    # non chi memorizza la firma spettrale delle tracce viste.
    if az_vs is not None:
        AZ_sel = torch.cat([AZ_va, AZ_vs]); V_sel = torch.cat([V_va, V_vs])
        XI_sel = torch.cat([XI_va, su_device(xi_a_norm(y_xi_vs, auto)).unsqueeze(1)])
        F_sel = torch.cat([F_va, su_device(forza_a_norm(y_f_vs, cfg)).unsqueeze(1)])
    else:
        AZ_sel, V_sel, XI_sel, F_sel = AZ_va, V_va, XI_va, F_va
    Nsel = AZ_sel.size(0)

    def loss_val(rete, target, criterio, AZ=None, V=None):
        """Validazione: solo la loss PRINCIPALE, senza pesi e senza ausiliarie.
        Deve misurare la qualita' della grandezza che conta, non il termine di coerenza
        interna — altrimenti i valori non sono confrontabili fra configurazioni."""
        AZ = AZ_sel if AZ is None else AZ; V = V_sel if V is None else V
        n = AZ.size(0)
        rete.eval(); s = 0.0
        with torch.no_grad():
            for i in range(0, n, bs):
                nb = min(bs, n - i)
                out = rete(AZ[i:i+bs], V[i:i+bs])
                s += criterio(out, target[i:i+bs]).item() * nb
        return s / n

    stato = {"best_xi": float("inf"), "best_f": float("inf"), "alpha": cfg.autorita_az_iniziale}

    def addestra(az_np, v_np, xi_np, f_np, epoche, etichetta="", alpha_finale=None,
                 kp_np=None, zs_np=None):
        """Un ciclo di addestramento su un dataset arbitrario. Estratto in funzione perche'
        DAgger lo richiama a ogni giro sul dataset aggregato.

        kp_np / zs_np sono i target AUSILIARI della rete forza (i due fattori di cui la
        forza e' il prodotto). Se mancano, la loss usa solo il termine principale su F."""
        AZ = su_device(norm.na(az_np)).unsqueeze(1); V = su_device(norm.nv(v_np)).unsqueeze(1)
        XI = su_device(xi_a_norm(xi_np, auto)).unsqueeze(1)
        FZ = su_device(forza_a_norm(f_np, cfg)).unsqueeze(1)
        # PESI per compensare lo sbilanciamento del target xi (47% al minimo del damper)
        W = su_device(pesi_per_strato(xi_a_norm(xi_np, auto))).unsqueeze(1)
        KP = ZS = None
        if due_uscite and kp_np is not None and zs_np is not None:
            KP = su_device(kp_np / max(cfg.kp_max_ott, 1e-9)).unsqueeze(1)
            ZS = su_device(zs_np / max(cfg.zs_max, 1e-9)).unsqueeze(1)
        n = AZ.size(0)
        af = alpha_fin if alpha_finale is None else alpha_finale
        for e in range(1, epoche + 1):
            # rampa di alpha solo nel ciclo iniziale; nei giri DAgger resta al valore pieno
            if alpha_finale is None:
                n_rampa = max(1, int(cfg.autorita_az_rampa * epoche))
                t = min(1.0, (e - 1) / n_rampa)
                stato["alpha"] = cfg.autorita_az_iniziale + t * (af - cfg.autorita_az_iniziale)
            else:
                stato["alpha"] = af
            rete_xi.imposta_autorita(stato["alpha"])
            rete_xi.train(); rete_forza.train()
            perm = torch.randperm(n, device=device)
            tl_xi = tl_f = 0.0
            for i in range(0, n, bs):
                idx = perm[i:i+bs]; az_b = AZ[idx]; v_b = V[idx]

                opt_xi.zero_grad()
                lx = loss_xi(rete_xi(az_b, v_b), XI[idx], W[idx])
                lx.backward(); opt_xi.step()

                opt_forza.zero_grad()
                if KP is not None:
                    # la rete restituisce i due fattori in unita' fisiche: si normalizzano
                    # con le STESSE scale usate per i target, altrimenti i termini della
                    # loss avrebbero pesi impliciti diversi
                    zs_p, kp_p = rete_forza.fattori(az_b, v_b)
                    f_p = torch.clamp(-(kp_p * zs_p) / cfg.forza_max, -1.0, 1.0)
                    lf = loss_forza(f_p, FZ[idx],
                                    zs_p / cfg.zs_max, ZS[idx],
                                    kp_p / cfg.kp_max_ott, KP[idx])
                else:
                    lf = loss_forza(rete_forza(az_b, v_b), FZ[idx])
                lf.backward(); opt_forza.step()

                tl_xi += lx.item() * idx.size(0); tl_f += lf.item() * idx.size(0)
            tl_xi /= n; tl_f /= n
            # ===== DEBUG UNA VOLTA PER EPOCA =====
            with torch.no_grad():
                L = rete_xi.logit_schedulazione(V[:bs])
                delta_raw = rete_xi.modulazione(AZ[:bs], V[:bs])
                delta_r = rete_xi.beta * delta_raw
                delta_clamp = torch.clamp(delta_raw, -1.0, 1.0)
                delta_c = rete_xi.beta * delta_clamp
                z_1 = L + delta_r
                z_2 = L + delta_c
                print(f"\nDEBUG EPOCA {e}")
                print(f"{'Coefficiente Alpha (beta buffer)':25s} = {rete_xi.beta.item():.4f}")
                print(f"{'L(v)':25s} min={L.min().item():8.6f} max={L.max().item():8.6f} mean={L.mean().item():8.6f}")
                print(f"{'Delta Intoccato':25s} min={delta_raw.min().item():8.6f} max={delta_raw.max().item():8.6f} mean={delta_raw.mean().item():8.6f}")
                print(f"{'beta*Delta Intoccato':25s} min={delta_r.min().item():8.6f} max={delta_r.max().item():8.6f} mean={delta_r.mean().item():8.6f}")
                print(f"{'Delta Clampato [-1,1]':25s} min={delta_clamp.min().item():8.6f} max={delta_clamp.max().item():8.6f} mean={delta_clamp.mean().item():8.6f}")
                print(f"{'beta*Delta Clampato':25s} min={delta_c.min().item():8.6f} max={delta_c.max().item():8.6f} mean={delta_c.mean().item():8.6f}")
                print(f"{'L(v) + beta*Delta':25s} min={z_1.min().item():8.6f} max={z_1.max().item():8.6f} mean={z_1.mean().item():8.6f}")
                print(f"{'L(v) + beta*Delta Clampato':25s} min={z_2.min().item():8.6f} max={z_2.max().item():8.6f} mean={z_2.mean().item():8.6f}")
            # ===== FINE DEBUG =====
            vl_xi = loss_val(rete_xi, XI_sel, loss_xi); vl_f = loss_val(rete_forza, F_sel, loss_forza_val)
            if vl_xi < stato["best_xi"]:
                stato["best_xi"] = vl_xi; torch.save(rete_xi.state_dict(), path_xi)
            if vl_f < stato["best_f"]:
                stato["best_f"] = vl_f; torch.save(rete_forza.state_dict(), path_forza)
            if e % 3 == 0 or e == 1 or epoche <= 4:
                print(f"    {etichetta}Epoca {e:02d}/{epoche}  (alpha={stato['alpha']:.2f})"
                      f"  |  RETE xi: Loss(Train)={tl_xi:.5f}  Loss(Valid)={vl_xi:.5f}"
                      f"  |  RETE F: Loss(Train)={tl_f:.5f}  Loss(Valid)={vl_f:.5f}")

    addestra(az_tr, v_tr, y_xi_tr, y_f_tr, cfg.n_epoche, kp_np=kp_tr, zs_np=zs_tr)

    # 4b) DAgger: la rete ha imparato su a_z del modello PASSIVO ma in closed-loop legge
    # l'accelerazione gia' controllata (~47% di quella). Qui si raccolgono gli stati che
    # visita davvero, si rietichettano con l'esperto e si aggrega. Vedi aggregazione.py.
    if cfg.usa_dagger and cfg.metodo_xi == "ottimo":
        from aggregazione import esegui_dagger
        rete_xi.load_state_dict(torch.load(path_xi, map_location=device))
        rete_forza.load_state_dict(torch.load(path_forza, map_location=device))

        def imposta_validazione(az_v, v_v, xi_v, f_v):
            """Sostituisce il set di selezione con stati CLOSED-LOOP e azzera i minimi.

            Senza questo, DAgger non salvava nulla: la validazione era sulla distribuzione
            passiva, che l'aggregazione peggiora per costruzione, quindi la condizione
            'vl < best' non si verificava mai e il modello finale restava quello pre-DAgger.
            Azzerare i best e' necessario anche cambiando set: i valori vecchi sono
            incommensurabili con i nuovi."""
            nonlocal AZ_sel, V_sel, XI_sel, F_sel, Nsel
            AZ_sel = su_device(norm.na(az_v)).unsqueeze(1)
            V_sel = su_device(norm.nv(v_v)).unsqueeze(1)
            XI_sel = su_device(xi_a_norm(xi_v, auto)).unsqueeze(1)
            F_sel = su_device(forza_a_norm(f_v, cfg)).unsqueeze(1)
            Nsel = AZ_sel.size(0)
            stato["best_xi"] = stato["best_f"] = float("inf")

        # ROLLOUT SU STRADE NUOVE: stesse celle del training, realizzazioni diverse (seme
        # diverso). Vedi strade_sintetiche.genera_tracce_aggregazione per il motivo.
        tracce_agg = list(tracce_tr)
        if cfg.usa_augmentation:
            nuove = genera_tracce_aggregazione(auto, cfg)
            # si tengono le 2 tracce REGISTRATE (non replicabili) e si sostituiscono le
            # sintetiche con realizzazioni fresche
            tracce_agg = [t for t in tracce_tr if len(t) < 3 or t[2] is None] + nuove
            print(f"    Rollout su {len(nuove)} strade sintetiche NUOVE (seme "
                  f"{cfg.aug_seed}+{cfg.dagger_seme_strade}, celle identiche al training) "
                  f"+ {len(tracce_agg)-len(nuove)} registrate")
        rete_xi, rete_forza, _agg = esegui_dagger(
            rete_xi, rete_forza, norm, auto, cfg, device, tracce_agg,
            (az_tr, v_tr, y_xi_tr, y_f_tr, kp_tr, zs_tr), tracce_va_sint,
            lambda a, v, x, f, ep, kp_np=None, zs_np=None: addestra(
                a, v, x, f, ep, etichetta="  ", alpha_finale=alpha_fin,
                kp_np=kp_np, zs_np=zs_np),
            imposta_validazione=imposta_validazione)

    # 5) valutazione: ricarica il best-checkpoint e valuta sui DUE set di validazione
    rete_xi.load_state_dict(torch.load(path_xi, map_location=device)); rete_xi.eval()
    rete_forza.load_state_dict(torch.load(path_forza, map_location=device)); rete_forza.eval()

    def predici(rete, AZ, V):
        out = []
        with torch.no_grad():
            for i in range(0, AZ.size(0), bs):
                out.append(rete(AZ[i:i+bs], V[i:i+bs]).cpu())
        return torch.cat(out).numpy().flatten()

    def riporta(titolo, y_xi, y_f, AZ, V):
        p_xi = norm_a_xi(predici(rete_xi, AZ, V), auto)
        p_f = norm_a_forza(predici(rete_forza, AZ, V), cfg)
        print(f"\n {titolo}")
        # AVVERTENZA su R^2: se il target e' quasi costante (su strada liscia xi* resta
        # incollato a xi_min) la varianza al denominatore va a zero e R^2 esplode a valori
        # assurdamente negativi pur con RMSE minuscolo. Li' R^2 non misura niente: si legge
        # l'RMSE quindi può intuire quando qualcosa non quadra perché R^2<<<<0.
        sd = float(np.std(y_xi))
        nota = "  [R^2 non informativo: target quasi costante]" if sd < 0.02 else ""
        print(f"   xi(t)    -> R^2: {r2_score(y_xi, p_xi):+.4f} | "
              f"RMSE: {np.sqrt(np.mean((y_xi - p_xi) ** 2)):.4f} | "
              f"MAE: {mean_absolute_error(y_xi, p_xi):.4f} | dev.std target: {sd:.4f}{nota}")
        print(f"   forza(t) -> R^2: {r2_score(y_f, p_f):+.4f} | "
              f"RMSE: {np.sqrt(np.mean((y_f - p_f) ** 2)):.1f} N | "
              f"MAE: {mean_absolute_error(y_f, p_f):.1f} N")
        # PICCHI DELLA FORZA. puo' essere la rete che esagera, oppure il riferimento che chiede
        # davvero tanto. Servono i due picchi affiancati, e la quota di SATURAZIONE
        # dell'etichetta: F* = clip(-kp*·z_s', ±F_max), quindi quando l'ottimo chiede piu' di
        # F_max l'etichetta viene tagliata e la rete impara a stare incollata al limite (di base satura).
        # Se la quota di saturazione e' alta, il picco che cresce non e' un difetto della rete:
        # e' l'attuatore troppo piccolo per quello che l'ottimizzatore vorrebbe (non si riesce a
        # generare più forza di quella disponibile dall'attuatore da progetto)
        pk_t, pk_p = float(np.abs(y_f).max()), float(np.abs(p_f).max())
        sat_t = float(np.mean(np.abs(y_f) >= 0.999 * cfg.forza_max))
        sat_p = float(np.mean(np.abs(p_f) >= 0.999 * cfg.forza_max))
        alti = np.abs(y_f) > 0.5 * pk_t                    # solo la coda alta del target
        bias_pk = float(np.mean(np.abs(p_f[alti]) - np.abs(y_f[alti]))) if alti.any() else 0.0
        print(f"   picchi   -> |F| max: target {pk_t:.0f} N, ML {pk_p:.0f} N "
              f"({pk_p/max(pk_t,1e-9)-1:+.0%}) | margine ML sul limite "
              f"{100*(1-pk_p/cfg.forza_max):.0f}%")
        print(f"                saturazione a ±{cfg.forza_max:.0f} N: target {sat_t:.2%}, "
              f"ML {sat_p:.2%} | sui picchi (|F*|>50% del max) il ML sta {bias_pk:+.0f} N")
        return p_xi, p_f

    from contesto import descrivi_traccia, classe_iso_equivalente, riga_origine
    print("\n" + "=" * 90)
    print(" VALIDAZIONE — confronto fra valori APPRESI e valori CALCOLATI, su dati mai visti")
    print("=" * 90)
    print(f" xi predetto / F predetta : {riga_origine('ML')}")
    print(f" xi* / F* di riferimento  : {riga_origine('ottimo')}")
    _azv, _vv = tracce_va_reali[0][0], tracce_va_reali[0][1]
    _cl = classe_iso_equivalente(float(np.sqrt(np.mean(np.asarray(_azv) ** 2))),
                                 float(np.mean(_vv)))
    xi_pred, forza_pred = riporta(
        f"[A] REGISTRATA: {descrivi_traccia(_azv, _vv, cfg, nome_val)}"
        f"  |  rugosita' equivalente classe ISO {_cl}",
        y_xi_va, y_f_va, AZ_va, V_va)
    if az_vs is not None:
        # test decisivo: rugosita' x velocita' mai viste insieme (es. classe sconnessa a 90 km/h)
        _el = "; ".join(nome_traccia(t, "?") for t in tracce_va_sint)
        riporta(f"[B] SINTETICHE, celle classe x velocita' DISGIUNTE dal training\n"
                f"     {_el}",
                y_xi_vs, y_f_vs, AZ_vs, V_vs)
        print("     ^ [B] e' il test severo: rugosita' x velocita' mai viste insieme.")
    _pk = float(np.abs(forza_pred).max())
    print(f"\n |forza| max prevista dal ML: {_pk:.1f} N  (limite {cfg.forza_max:.0f} N,"
          f" margine {100*(1-_pk/cfg.forza_max):.0f}%)")
    if _pk > 0.9 * cfg.forza_max:
        print("    [!] margine sotto il 10%: guardare la riga 'picchi' qui sopra per capire se"
              " e' la rete che esagera o il riferimento che chiede troppo")

    # COMFORT in closed-loop. Il riferimento ottimo va calcolato SULLO STESSO SEGMENTO su
    # cui si misura il ML, altrimenti non e' un confronto: mediarlo su tutte le tracce
    # (comprese le classe E sintetiche) e accostarlo al ML misurato su un tratto urbano da'
    # numeri corretti in un accostamento privo di senso.
    # la traccia e' (a_z, v, z_r'|None, nome): si prendono i primi due elementi, non si
    # spacchetta tutto — aggiungere un campo alla tupla non deve rompere i punti d'uso
    az_val, v_val = tracce_va_reali[0][0], tracce_va_reali[0][1]
    nseg = min(len(az_val), int(cfg.comfort_secondi * cfg.freq_campion))
    traj_cl = traiettorie_reali_ml(az_val[:nseg], v_val[:nseg], rete_xi, rete_forza, norm, auto, cfg, device)
    rms_pas = float(np.sqrt(np.mean(traj_cl["acc_cassa_pas"] ** 2)))
    rms_ml = float(np.sqrt(np.mean(traj_cl["acc_cassa_ml"] ** 2)))

    from xi_ottimo import ottimo as _ott, simula_con_xi as _sim, pondera_wk as _wk
    zr_s, zrd_s = ricostruisci_strada(az_val[:nseg], cfg)
    if cfg.calibra_strada:
        _f = np.clip(np.sqrt(np.mean(az_val[:nseg] ** 2))
                     / (rms_accel_passiva(zrd_s, auto, cfg) + 1e-9), 0.2, 20.0)
        zrd_s = zrd_s * _f
    xi_id, kp_id = _ott(az_val[:nseg], v_val[:nseg], zrd_s, auto, cfg)
    az_id, _g, _c = _sim(zrd_s, v_val[:nseg], xi_id, auto, cfg, kp_serie=kp_id)
    rms_id = float(np.sqrt(np.mean(az_id ** 2)))
    print("\n COMFORT in CLOSED-LOOP — RMS accelerazione cassa")
    print(f"   dati: {nome_val}, primi {nseg/cfg.freq_campion:.0f} s"
          f"  (v {v_val[:nseg].min()*3.6:.0f}-{v_val[:nseg].max()*3.6:.0f} km/h,"
          f" rugosita' equivalente classe ISO {_cl})")
    print(f"   passiva  {rms_pas:.3f} m/s^2")
    print(f"   ML       {rms_ml:.3f} m/s^2   ({100*(1-rms_ml/rms_pas):+.1f}% vs passiva)")
    print(f"   ottimo   {rms_id:.3f} m/s^2   ({100*(1-rms_id/rms_pas):+.1f}% vs passiva)"
          f"  <- il tetto raggiungibile con le etichette")
    if rms_ml < rms_id * 0.98:
        print("   [!] il ML batte l'ottimo sull'accelerazione GREZZA: non e' un paradosso,")
        print("       l'ottimo minimizza l'accelerazione PONDERATA Wk sotto vincoli, quindi")
        print("       puo' accettare piu' RMS grezzo per tenere la ruota a terra.")
        print(f"       Comfort ponderato Wk: ML {np.sqrt(np.mean(_wk(traj_cl['acc_cassa_ml'], cfg.freq_campion)**2)):.3f}"
              f"  contro ottimo {np.sqrt(np.mean(_wk(az_id, cfg.freq_campion)**2)):.3f} m/s^2")

    # Tabella riassuntiva delle etichette di validazione.
    # NB: con metodo_xi="ottimo" NON si bin-a per r (che non guida piu' nulla) ma per
    # RUGOSITA', perche' e' da quella che dipende l'ottimo; e non si stampa kp_da_r, che
    # non e' il guadagno usato. Stampare la vecchia tabella qui sarebbe stato un errore
    # silenzioso: numeri corretti sotto un'intestazione che descrive un'altra legge.
    print("\n" + "-" * 74)
    if cfg.metodo_xi == "ottimo":
        # si usa il set SINTETICO se c'e': sulla traccia reale urbana xi* e' costante a
        # xi_min (strada liscia), quindi la tabella mostrerebbe cinque righe identiche
        _az_t, _v_t, _xi_t, _f_t = ((az_vs, v_vs, y_xi_vs, y_f_vs) if az_vs is not None
                                    else (az_va, v_va, y_xi_va, y_f_va))
        rms_fin = np.sqrt(np.mean(_az_t.astype(np.float64) ** 2, axis=1))
        print(" ETICHETTE OTTIME: xi* in funzione della RUGOSITA' (RMS a_z) e della velocita'"
              + ("   [strade sintetiche]" if az_vs is not None else "   [traccia reale]"))
        print("-" * 74)
        bordi = np.quantile(rms_fin, [0, .25, .5, .75, 1.0])
        print(f"{'RMS a_z':<14} | {'v [km/h]':<9} | {'xi* medio':<10} | {'|forza| [N]'}")
        for i in range(4):
            m = (rms_fin >= bordi[i]) & (rms_fin <= bordi[i + 1])
            if np.any(m):
                print(f"{bordi[i]:.2f}-{bordi[i+1]:<9.2f} | {_v_t[m].mean()*3.6:<9.1f} | "
                      f"{_xi_t[m].mean():<10.3f} | {np.abs(_f_t[m]).mean():.0f}")
    else:
        print(" REGOLA √2: xi in funzione del rapporto r = omega/omega_n (target validazione)")
        print("-" * 74)
        bins = [0.0, 1.0, RADQ2, 2.0, 10.0]
        etich = ["r<1", "1<r<√2", "√2<r<2", "r>2"]
        print(f"{'Range r':<10} | {'v [km/h]':<9} | {'xi medio':<9} | {'kp [Ns/m]':<10} | {'|forza| [N]'}")
        for i in range(len(bins) - 1):
            m = (r_va >= bins[i]) & (r_va < bins[i + 1])
            if np.any(m):
                print(f"{etich[i]:<10} | {v_va[m].mean()*3.6:<9.1f} | {y_xi_va[m].mean():<9.3f} | "
                      f"{kp_da_r(r_va[m].mean(), cfg):<10.0f} | {np.abs(y_f_va[m]).mean():.0f}")

    # 6) figura + animazione
    # tenuta di strada ed energia (dal closed-loop gia' calcolato)
    rh_pas = float(np.sqrt(np.mean(traj_cl["defl_gomma_pas"] ** 2)) * 1000)
    rh_ml = float(np.sqrt(np.mean(traj_cl["defl_gomma_ml"] ** 2)) * 1000)
    from xi_ottimo import riferimenti as _rif
    _a, _d, _s, _f = _rif(auto, cfg)
    print(f" TENUTA DI STRADA su {nome_val} ({nseg/cfg.freq_campion:.0f} s) — RMS deflessione"
          f" pneumatico: passiva {rh_pas:.2f} mm -> ML {rh_ml:.2f} mm"
          f"   (limite ammesso {_d*1000:.2f} mm)")
    en = bilancio_energetico(traj_cl, cfg)
    print(f"\n BILANCIO ENERGETICO — integrali su {nseg/cfg.freq_campion:.0f} s di {nome_val}")
    print(f"   IPOTESI  damper    : {en['ipotesi']}")
    print(f"   IPOTESI  attuatore : {en['ipotesi_attuatore']}")
    print("   CONVENZIONE: le voci misurate sono MODULI (>=0), il verso e' nel nome. Il"
          f" bilancio ha segno")
    print("   dal punto di vista della BATTERIA: <0 esce, >0 rientra. I joule scalano con la"
          f" durata: non sono una potenza.")
    print(f"   MECCANICA  damper dissipato in calore nell'olio"
          f"{' (contato nel recupero)' if en['damper_recuperabile'] else ' — PERSO, fuori bilancio'}"
          f" : {en['E_damp']:7.1f} J")
    print(f"   MECCANICA  attuatore, lavoro fatto sul veicolo (spinge)                :"
          f" {en['E_inj']:7.1f} J")
    print(f"   MECCANICA  attuatore, lavoro ricevuto dal veicolo (frena)              :"
          f" {en['E_abs']:7.1f} J")
    print(f"   ELETTRICO  ADDEBITO  = lavoro in spinta / eta_attuatore                : "
          f"{-en['addebito']:+7.1f} J   (esce dalla batteria)")
    print(f"   ELETTRICO  ACCREDITO = {en['formula']:<47}: {en['accredito']:+7.1f} J"
          f"   (rientra in batteria)")
    print(f"   ELETTRICO  NETTO     = accredito - addebito                            :"
          f" {en['netto']:+7.1f} J  -> {en['verdetto']}")
    print(f"   [sensibilita'] con le perdite di trazione al {en['eta']:.0%} come il generatore,"
          f" il netto sarebbe {en['netto_con_perdite_attuatore']:+.1f} J")
    if not en['damper_recuperabile']:
        # il confronto esplicito serve a mostrare quanto pesa l'ipotesi sull'hardware
        _alt = cfg.eta_generatore * (en['E_damp'] + en['E_abs']) - en['addebito']
        print(f"   [sensibilita'] con un damper ELETTROMAGNETICO al posto dell'idraulico il netto"
              f" sarebbe {_alt:+.1f} J")
    # DIAGNOSTICA FISICA: R^2 e comfort non dicono se xi segue la VELOCITA' o l'accelerazione.
    # Qui si misura direttamente xi(v, a_z), in open-loop e in closed-loop.
    esegui_diagnostica(rete_xi, rete_forza, norm, auto, cfg, device,
                       dati=(az_tr, v_tr, y_xi_tr), figura=cfg.salva_figura,
                       percorso=os.path.join(base, "diagnostica_xi.png"),
                       provenienza=(f"{len(tracce_tr)} tracce di TRAINING "
                                    f"({len(tracce_tr)-cfg.n_tracce_sintetiche} reali registrate"
                                    f" + {cfg.n_tracce_sintetiche} sintetiche ISO 8608), "
                                    f"finestre da {cfg.seq_len/cfg.freq_campion:.0f} s"))

    if cfg.salva_figura:
        salva_figura(y_xi_va, xi_pred, y_f_va, forza_pred, r_va, rms_pas, rms_ml, cfg, auto,
                     os.path.join(base, cfg.figura_file),
                     nome_traccia=nome_val, v_val=v_va,
                     secondi_comfort=nseg / cfg.freq_campion)
        salva_figura_efficienza(traj_cl, cfg, os.path.join(base, "efficienza_sospensione.png"),
                                nome_traccia=nome_val, auto=auto)
    if cfg.mostra_anim or cfg.salva_video:
        # Il vecchio if/else su cfg.anim_usa_ml era morto (i due rami identici). Ora si
        # simulano TUTTI i controllori sulla stessa strada e si commutano dal vivo con i
        # pulsanti nell'animazione: ML, ottimo (xi*, kp*) e regola √2.
        print("\n[3] Animazione: tutti i controllori sulla stessa strada demo...")
        print("    (la strada demo e' severa — equivale a una classe E ISO 8608 — quindi"
              " xi DEVE muoversi parecchio)")
        if cfg.anim_confronto:
            traj = traiettorie_demo_confronto(rete_xi, rete_forza, norm, auto, cfg, device)
            scarto_anim = float(np.mean(np.abs(traj["ML"]["xi"] - traj["ML"]["xi_rif"])))
            print(f"    scarto medio |xi_ML - xi*| sulla demo: {scarto_anim:.3f}")
        else:
            traj = traiettorie_demo_ml(rete_xi, rete_forza, norm, auto, cfg, device,
                                       usa_esperto=not cfg.anim_usa_ml)
        anima_confronto(traj, cfg, os.path.join(base, cfg.video_file), auto=auto)

    print("=" * 90)
    print(f"Completato in {time.time() - t0:.1f} s")


if __name__ == "__main__":
    # PRIMA di qualunque stampa: su Windows la codifica di default non contiene i
    # simboli matematici del log che si sta usando, perchè abbiamo usato MacOS.
    # Vedere portabilita.py, in windows il run può fallire a causa del simboli.
    forza_utf8()
    main()


**Nota sull'output:** anche questa cella è eseguibile end-to-end così com'è, ma la pipeline completa (data augmentation su 48 strade sintetiche, addestramento DAgger su 5 iterazioni, validazione closed-loop) è computazionalmente pesante e può richiedere ore di calcolo anche su hardware dedicato. Per questo motivo l'output viene riportato già calcolato qui sotto, invece di richiedere una rieseguzione della cella ad ogni consultazione del notebook.

In output dal sistema otteniamo:

<pre style="font-size:11px; line-height:1.2; overflow-x:auto;">
==========================================================================================
 SOSPENSIONE 2 GDL (PEUGEOT 207) — CONTROLLO APPRESO (ML)
 Legge di riferimento (etichette): OTTIMO VINCOLATO (comfort ISO 2631 sotto i limiti hardware)
 omega_n = 9.41 rad/s (1.50 Hz)   |   crossover a r = √2 ≈ 1.414
 Sistema: Darwin 24.6.0 / arm64 / Python 3.14.3 / codifica stampe utf-8
==========================================================================================
    HW: Apple Silicon GPU (MPS)
    Core: 8 totali -> uso 4 performance (salto 4 efficiency)
    Operatori verificati su mps (conv dilatate, batchnorm, riduzione depthwise, forward+backward)

[1] Caricamento tracce e generazione etichette (parallela sui core)...
  [ok] 3 tracce reali -> 2 train / 1 validazione (thessaloniki_19_11_2023.pkl)
    [!] larisa_07_10_2023.pkl: cadenza reale 100.1 Hz (jitter 15.3%) contro cfg.freq_campion = 100 Hz
        ricampionata su griglia uniforme: 271177 -> 271314 campioni. Senza questo, filtri e integrazioni lavorerebbero alla frequenza sbagliata.
    [!] thessaloniki_01_10_2023.pkl: cadenza reale 100.1 Hz (jitter 13.5%) contro cfg.freq_campion = 100 Hz
        ricampionata su griglia uniforme: 229148 -> 229200 campioni. Senza questo, filtri e integrazioni lavorerebbero alla frequenza sbagliata.
    [!] thessaloniki_19_11_2023.pkl: cadenza reale 100.1 Hz (jitter 14.7%) contro cfg.freq_campion = 100 Hz
        ricampionata su griglia uniforme: 252392 -> 252468 campioni. Senza questo, filtri e integrazioni lavorerebbero alla frequenza sbagliata.
    Filtro ISO 2631-1 Wk verificato sulla Tabella 3 della norma: errore massimo 0.44%
    Etichette da OTTIMIZZAZIONE VINCOLATA: si cerca la coppia (xi, kp) che
    minimizza il comfort ISO 2631 Wk restando entro i limiti hardware.
    Limiti dei vincoli — ognuno = limite di PICCO / fattore di picco:
      GOMMA  picco  15.38 mm = (m_cassa+m_ruota)*g/k_gomma = (260+38)*9.807/190000   [deflessione statica: oltre, la ruota stacca]
             fattore 3.30 (ripiego cfg)  ->  RMS ammesso 4.66 mm
      CORSA  picco  80 mm = cfg.corsa_disponibile   [rattle space — VALORE NON VERIFICATO sulla 207]
             fattore 3.30 (ripiego cfg)  ->  RMS ammesso 24.24 mm
      FORZA  picco   1500 N  = cfg.forza_max   [saturazione attuatore, spec hardware]
             fattore 3.30 (ripiego cfg)  ->  RMS ammesso 455 N
      COMFORT scala 0.315 m/s^2 = cfg.xi_ott_comfort_rif   [soglia ISO 2631-1, non un limite: e' la scala dell'obiettivo]
    Griglia 21x15 = 315 candidati, finestra 2 s, penalita' 1000
    Data augmentation: 48 strade sintetiche ISO 8608 (buche/dossi bruschi, gradini, salite/discese)
    Griglia fattoriale: 5 classi x 5 velocita' = 25 celle | training 21 celle | validazione 4 celle (DISGIUNTE)
    Crossover r=√2 a v ≈ 53 km/h: le tracce a velocita' variabile lo attraversano a rugosita' costante
      train  1: classe E (x1.09)  v   36-  42 km/h [costante]  RMS a_z= 5.13 m/s^2  ostacoli= 5  n_max=2.83 c/m
      train  2: classe B (x1.10)  v   20-  97 km/h [rampa   ]  RMS a_z= 1.17 m/s^2  ostacoli=15  n_max=1.48 c/m
      train  3: classe C (x0.77)  v   42- 100 km/h [urbano  ]  RMS a_z= 1.99 m/s^2  ostacoli= 7  n_max=1.43 c/m
      train  4: classe C (x1.71)  v   19-  24 km/h [costante]  RMS a_z= 1.59 m/s^2  ostacoli= 9  n_max=2.83 c/m
      train  5: classe E (x1.86)  v   73-  79 km/h [costante]  RMS a_z= 9.20 m/s^2  ostacoli=21  n_max=1.83 c/m
      train  6: classe B (x1.29)  v   19-  24 km/h [costante]  RMS a_z= 0.71 m/s^2  ostacoli= 3  n_max=2.83 c/m
      train  7: classe A (x1.65)  v   19-  78 km/h [rampa   ]  RMS a_z= 0.82 m/s^2  ostacoli= 5  n_max=1.86 c/m
      train  8: classe E (x1.08)  v   95- 100 km/h [costante]  RMS a_z= 8.23 m/s^2  ostacoli=20  n_max=1.44 c/m
      train  9: classe B (x0.78)  v   42-  99 km/h [urbano  ]  RMS a_z= 1.56 m/s^2  ostacoli=15  n_max=1.45 c/m
      train 10: classe A (x1.17)  v   30-  75 km/h [urbano  ]  RMS a_z= 0.72 m/s^2  ostacoli=10  n_max=1.91 c/m
      train 11: classe D (x1.27)  v   24-  58 km/h [rampa   ]  RMS a_z= 3.53 m/s^2  ostacoli=14  n_max=2.47 c/m
      train 12: classe A (x1.71)  v   11-  78 km/h [sweep   ]  RMS a_z= 0.54 m/s^2  ostacoli= 2  n_max=1.85 c/m
      train 13: classe D (x1.35)  v   37-  89 km/h [rampa   ]  RMS a_z= 4.13 m/s^2  ostacoli=16  n_max=1.63 c/m
      train 14: classe C (x0.99)  v   11-  87 km/h [sweep   ]  RMS a_z= 2.09 m/s^2  ostacoli=12  n_max=1.66 c/m
      train 15: classe D (x1.77)  v   11-  75 km/h [sweep   ]  RMS a_z= 4.38 m/s^2  ostacoli=30  n_max=1.91 c/m
      train 16: classe C (x1.91)  v   54-  60 km/h [costante]  RMS a_z= 2.60 m/s^2  ostacoli= 7  n_max=2.40 c/m
      train 17: classe E (x0.68)  v   32-  75 km/h [urbano  ]  RMS a_z= 4.91 m/s^2  ostacoli= 9  n_max=1.91 c/m
      train 18: classe B (x1.88)  v   30-  75 km/h [urbano  ]  RMS a_z= 1.65 m/s^2  ostacoli=12  n_max=1.92 c/m
      train 19: classe B (x1.69)  v   21-  52 km/h [urbano  ]  RMS a_z= 1.09 m/s^2  ostacoli= 4  n_max=2.78 c/m
      train 20: classe A (x0.97)  v   46- 108 km/h [sweep   ]  RMS a_z= 0.61 m/s^2  ostacoli= 5  n_max=1.34 c/m
      train 21: classe D (x1.85)  v   73-  79 km/h [costante]  RMS a_z= 5.49 m/s^2  ostacoli=28  n_max=1.82 c/m
      train 22: classe E (x1.25)  v   36-  42 km/h [costante]  RMS a_z= 5.49 m/s^2  ostacoli=10  n_max=2.83 c/m
      train 23: classe B (x1.63)  v   95- 101 km/h [costante]  RMS a_z= 2.27 m/s^2  ostacoli=22  n_max=1.43 c/m
      train 24: classe C (x1.32)  v   73-  79 km/h [costante]  RMS a_z= 2.57 m/s^2  ostacoli= 7  n_max=1.82 c/m
      train 25: classe C (x1.33)  v   21-  93 km/h [rampa   ]  RMS a_z= 1.93 m/s^2  ostacoli=10  n_max=1.55 c/m
      train 26: classe E (x1.01)  v   41-  92 km/h [sweep   ]  RMS a_z= 6.42 m/s^2  ostacoli=13  n_max=1.57 c/m
      train 27: classe B (x1.78)  v   20-  25 km/h [costante]  RMS a_z= 1.22 m/s^2  ostacoli= 8  n_max=2.83 c/m
      train 28: classe A (x1.37)  v   73-  79 km/h [costante]  RMS a_z= 1.24 m/s^2  ostacoli=11  n_max=1.82 c/m
      train 29: classe E (x1.88)  v   94-  99 km/h [costante]  RMS a_z=10.82 m/s^2  ostacoli=30  n_max=1.45 c/m
      train 30: classe B (x1.49)  v   30-  77 km/h [rampa   ]  RMS a_z= 1.69 m/s^2  ostacoli=14  n_max=1.88 c/m
      train 31: classe A (x1.81)  v   55-  61 km/h [costante]  RMS a_z= 1.15 m/s^2  ostacoli=14  n_max=2.36 c/m
      train 32: classe D (x1.41)  v   32-  74 km/h [urbano  ]  RMS a_z= 3.94 m/s^2  ostacoli=15  n_max=1.95 c/m
      train 33: classe A (x0.87)  v   18-  25 km/h [costante]  RMS a_z= 0.33 m/s^2  ostacoli= 1  n_max=2.83 c/m
      train 34: classe D (x0.98)  v   37-  42 km/h [costante]  RMS a_z= 2.71 m/s^2  ostacoli= 6  n_max=2.83 c/m
      train 35: classe C (x1.67)  v   22-  53 km/h [urbano  ]  RMS a_z= 1.81 m/s^2  ostacoli= 3  n_max=2.70 c/m
      train 36: classe D (x1.03)  v   19-  25 km/h [costante]  RMS a_z= 2.14 m/s^2  ostacoli= 4  n_max=2.83 c/m
      train 37: classe C (x1.07)  v   55-  61 km/h [costante]  RMS a_z= 2.03 m/s^2  ostacoli=10  n_max=2.37 c/m
      train 38: classe E (x1.49)  v   27-  84 km/h [sweep   ]  RMS a_z= 7.21 m/s^2  ostacoli=19  n_max=1.71 c/m
      train 39: classe B (x0.81)  v   21-  91 km/h [sweep   ]  RMS a_z= 1.52 m/s^2  ostacoli=16  n_max=1.58 c/m
      train 40: classe B (x0.63)  v   38-  85 km/h [rampa   ]  RMS a_z= 1.26 m/s^2  ostacoli=12  n_max=1.69 c/m
      train 41: classe A (x1.73)  v   19-  99 km/h [rampa   ]  RMS a_z= 0.72 m/s^2  ostacoli=10  n_max=1.46 c/m
      train 42: classe D (x0.63)  v   73-  78 km/h [costante]  RMS a_z= 3.15 m/s^2  ostacoli=10  n_max=1.84 c/m
      train 43: classe E (x1.04)  v   11-  81 km/h [sweep   ]  RMS a_z= 5.65 m/s^2  ostacoli=16  n_max=1.77 c/m
      train 44: classe B (x1.59)  v   95- 101 km/h [costante]  RMS a_z= 1.77 m/s^2  ostacoli=10  n_max=1.43 c/m
      train 45: classe C (x1.08)  v   73-  79 km/h [costante]  RMS a_z= 2.88 m/s^2  ostacoli=22  n_max=1.82 c/m
      train 46: classe C (x1.05)  v   18-  25 km/h [costante]  RMS a_z= 0.87 m/s^2  ostacoli= 0  n_max=2.83 c/m
      train 47: classe E (x0.72)  v   73-  78 km/h [costante]  RMS a_z= 6.13 m/s^2  ostacoli=14  n_max=1.84 c/m
      train 48: classe B (x1.34)  v   19-  24 km/h [costante]  RMS a_z= 1.07 m/s^2  ostacoli= 9  n_max=2.83 c/m
      valid  1: classe C (x0.62)  v   20-  98 km/h [rampa   ]  RMS a_z= 1.81 m/s^2  ostacoli=18  n_max=1.48 c/m
      valid  2: classe E (x0.62)  v   19-  24 km/h [costante]  RMS a_z= 2.92 m/s^2  ostacoli= 3  n_max=2.83 c/m
      valid  3: classe A (x1.44)  v   11-  86 km/h [sweep   ]  RMS a_z= 0.46 m/s^2  ostacoli= 3  n_max=1.68 c/m
      valid  4: classe D (x1.08)  v   53- 119 km/h [urbano  ]  RMS a_z= 4.59 m/s^2  ostacoli=21  n_max=1.21 c/m
    Campioni TRAIN      : 566864  (50 tracce = reali + sintetiche)
    Etichetta FORZA     : |F*| max 1500 N su un limite di 1500 N  |  satura nel 0.16% dei campioni
    Campioni VALID reale: 157458  (3a traccia registrata, mai vista)
    Campioni VALID sint.: 17200  (4 strade su celle classe x velocita' DISGIUNTE dal training)
    Normalizzazione (dai dati di training): Normalizzatore(a_z: media=+0.0009 std=2.1060 | v: media=12.037 std=6.986)
    Salvata in normalizzatore.json

[2] Addestramento di DUE reti separate (Loss() e Optimizer indipendenti)...
    Scarto etichette dal baseline 'comfort' [logit]: mediana 1.82 | p90 3.31 | p99 4.03 | max 8.77
    -> alpha suggerito (max scarto +10%) = 9.65   |   alpha finale in config = 7.00
    [!] alpha finale TROPPO PICCOLO: la correzione satura e la rete non puo' raggiungere xi*. Alzo a 9.65.
    Rete xi = sigmoid(L(v) + alpha*Delta(a_z,v)) con alpha da 0.30 a 9.65 (curriculum)
    Regolarizzazione: rete xi lr=1e-04 wd=3e-04  |  rete forza lr=1e-05 wd=1e-05
    loss forza: Huber con delta = 0.1779 (= 2 x 1.4826 x MAD = 2 x 0.0890)  |  curtosi in eccesso del target 8.1  |  14.5% dei campioni nel ramo lineare
    loss xi: MSE pesata per strato (target senza code, ma con una massa al minimo del damper: serve il peso, non la robustezza)

DEBUG EPOCA 1
Coefficiente Alpha (beta buffer) = 0.3000
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.422606 max=1.000000 mean=0.979384
beta*Delta Intoccato      min=-0.126782 max=0.300000 mean=0.293815
Delta Clampato [-1,1]     min=-0.422606 max=1.000000 mean=0.979384
beta*Delta Clampato       min=-0.126782 max=0.300000 mean=0.293815
L(v) + beta*Delta         min=-3.602880 max=-3.176099 mean=-3.182283
L(v) + beta*Delta Clampato min=-3.602880 max=-3.176099 mean=-3.182283
    Epoca 01/12  (alpha=0.30)  |  RETE xi: Loss(Train)=0.10505  Loss(Valid)=0.02835  |  RETE F: Loss(Train)=0.01082  Loss(Valid)=0.00241

DEBUG EPOCA 2
Coefficiente Alpha (beta buffer) = 1.6352
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.951502 max=1.000000 mean=0.427364
beta*Delta Intoccato      min=-1.555889 max=1.635191 mean=0.698822
Delta Clampato [-1,1]     min=-0.951502 max=1.000000 mean=0.427364
beta*Delta Clampato       min=-1.555889 max=1.635191 mean=0.698822
L(v) + beta*Delta         min=-5.031987 max=-1.840908 mean=-2.777277
L(v) + beta*Delta Clampato min=-5.031987 max=-1.840908 mean=-2.777277

DEBUG EPOCA 3
Coefficiente Alpha (beta buffer) = 2.9704
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.817071 max=0.826606 mean=0.296728
beta*Delta Intoccato      min=-2.427015 max=2.455338 mean=0.881396
Delta Clampato [-1,1]     min=-0.817071 max=0.826606 mean=0.296728
beta*Delta Clampato       min=-2.427015 max=2.455338 mean=0.881396
L(v) + beta*Delta         min=-5.903113 max=-1.020761 mean=-2.594703
L(v) + beta*Delta Clampato min=-5.903113 max=-1.020761 mean=-2.594703
    Epoca 03/12  (alpha=2.97)  |  RETE xi: Loss(Train)=0.01946  Loss(Valid)=0.01404  |  RETE F: Loss(Train)=0.00245  Loss(Valid)=0.00187

DEBUG EPOCA 4
Coefficiente Alpha (beta buffer) = 4.3056
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.562010 max=0.599580 mean=0.338367
beta*Delta Intoccato      min=-2.419777 max=2.581537 mean=1.456867
Delta Clampato [-1,1]     min=-0.562010 max=0.599580 mean=0.338367
beta*Delta Clampato       min=-2.419777 max=2.581537 mean=1.456867
L(v) + beta*Delta         min=-5.895876 max=-0.894562 mean=-2.019232
L(v) + beta*Delta Clampato min=-5.895876 max=-0.894562 mean=-2.019232

DEBUG EPOCA 5
Coefficiente Alpha (beta buffer) = 5.6408
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.363848 max=0.507912 mean=0.301181
beta*Delta Intoccato      min=-2.052384 max=2.865016 mean=1.698895
Delta Clampato [-1,1]     min=-0.363848 max=0.507912 mean=0.301181
beta*Delta Clampato       min=-2.052384 max=2.865016 mean=1.698895
L(v) + beta*Delta         min=-5.528483 max=-0.611083 mean=-1.777204
L(v) + beta*Delta Clampato min=-5.528483 max=-0.611083 mean=-1.777204

DEBUG EPOCA 6
Coefficiente Alpha (beta buffer) = 6.9760
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.631920 max=0.417333 mean=0.192387
beta*Delta Intoccato      min=-4.408252 max=2.911301 mean=1.342081
Delta Clampato [-1,1]     min=-0.631920 max=0.417333 mean=0.192387
beta*Delta Clampato       min=-4.408252 max=2.911301 mean=1.342081
L(v) + beta*Delta         min=-7.884351 max=-0.564798 mean=-2.134018
L(v) + beta*Delta Clampato min=-7.884351 max=-0.564798 mean=-2.134018
    Epoca 06/12  (alpha=6.98)  |  RETE xi: Loss(Train)=0.00279  Loss(Valid)=0.01584  |  RETE F: Loss(Train)=0.00195  Loss(Valid)=0.00185

DEBUG EPOCA 7
Coefficiente Alpha (beta buffer) = 8.3112
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.759355 max=0.342944 mean=0.151226
beta*Delta Intoccato      min=-6.311113 max=2.850259 mean=1.256863
Delta Clampato [-1,1]     min=-0.759355 max=0.342944 mean=0.151226
beta*Delta Clampato       min=-6.311113 max=2.850259 mean=1.256863
L(v) + beta*Delta         min=-9.787212 max=-0.625840 mean=-2.219235
L(v) + beta*Delta Clampato min=-9.787212 max=-0.625840 mean=-2.219235

DEBUG EPOCA 8
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.631655 max=0.295389 mean=0.082978
beta*Delta Intoccato      min=-6.093166 max=2.849425 mean=0.800432
Delta Clampato [-1,1]     min=-0.631655 max=0.295389 mean=0.082978
beta*Delta Clampato       min=-6.093166 max=2.849425 mean=0.800432
L(v) + beta*Delta         min=-9.569264 max=-0.626674 mean=-2.675667
L(v) + beta*Delta Clampato min=-9.569264 max=-0.626674 mean=-2.675667

DEBUG EPOCA 9
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.735953 max=0.300016 mean=0.047425
beta*Delta Intoccato      min=-7.099256 max=2.894055 mean=0.457479
Delta Clampato [-1,1]     min=-0.735953 max=0.300016 mean=0.047425
beta*Delta Clampato       min=-7.099256 max=2.894055 mean=0.457479
L(v) + beta*Delta         min=-10.575355 max=-0.582044 mean=-3.018620
L(v) + beta*Delta Clampato min=-10.575355 max=-0.582044 mean=-3.018620
    Epoca 09/12  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00167  Loss(Valid)=0.01633  |  RETE F: Loss(Train)=0.00172  Loss(Valid)=0.00181

DEBUG EPOCA 10
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.671101 max=0.284201 mean=0.050103
beta*Delta Intoccato      min=-6.473674 max=2.741501 mean=0.483310
Delta Clampato [-1,1]     min=-0.671101 max=0.284201 mean=0.050103
beta*Delta Clampato       min=-6.473674 max=2.741501 mean=0.483310
L(v) + beta*Delta         min=-9.949773 max=-0.734598 mean=-2.992789
L(v) + beta*Delta Clampato min=-9.949773 max=-0.734598 mean=-2.992789

DEBUG EPOCA 11
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.670723 max=0.307083 mean=0.041527
beta*Delta Intoccato      min=-6.470020 max=2.962224 mean=0.400588
Delta Clampato [-1,1]     min=-0.670723 max=0.307083 mean=0.041527
beta*Delta Clampato       min=-6.470020 max=2.962224 mean=0.400588
L(v) + beta*Delta         min=-9.946119 max=-0.513875 mean=-3.075511
L(v) + beta*Delta Clampato min=-9.946119 max=-0.513875 mean=-3.075511

DEBUG EPOCA 12
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.708246 max=0.318659 mean=0.012818
beta*Delta Intoccato      min=-6.831982 max=3.073890 mean=0.123642
Delta Clampato [-1,1]     min=-0.708246 max=0.318659 mean=0.012818
beta*Delta Clampato       min=-6.831982 max=3.073890 mean=0.123642
L(v) + beta*Delta         min=-10.308081 max=-0.402208 mean=-3.352457
L(v) + beta*Delta Clampato min=-10.308081 max=-0.402208 mean=-3.352457
    Epoca 12/12  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00120  Loss(Valid)=0.01630  |  RETE F: Loss(Train)=0.00156  Loss(Valid)=0.00188
    Rollout su 48 strade sintetiche NUOVE (seme 1234+4441, celle identiche al training) + 2 registrate

[2b] DAgger: allineamento fra dati di addestramento e stati visitati
    Preparate 50 strade per l'aggregazione (20 s ciascuna), di cui 48 con strada VERA nota; etichette dell'esperto calcolate una volta
    Validazione ricalibrata su rollout closed-loop di 4 strade di validazione (la vecchia era sulla distribuzione passiva)
    giro 1/5  beta=1.00  |  RMS a_z visitata 3.600 contro 2.106 nel dataset  |  |xi_applicato - xi*| = 0.0000  |  campioni 566864 -> 653945

DEBUG EPOCA 1
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=0.100164 max=0.341997 mean=0.244501
beta*Delta Intoccato      min=0.966215 max=3.299022 mean=2.358538
Delta Clampato [-1,1]     min=0.100164 max=0.341997 mean=0.244501
beta*Delta Clampato       min=0.966215 max=3.299022 mean=2.358538
L(v) + beta*Delta         min=-2.509884 max=-0.177077 mean=-1.117560
L(v) + beta*Delta Clampato min=-2.509884 max=-0.177077 mean=-1.117560
      Epoca 01/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00496  Loss(Valid)=0.00636  |  RETE F: Loss(Train)=0.00180  Loss(Valid)=0.00193

DEBUG EPOCA 2
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=0.081985 max=0.346671 mean=0.243741
beta*Delta Intoccato      min=0.790855 max=3.344104 mean=2.351206
Delta Clampato [-1,1]     min=0.081985 max=0.346671 mean=0.243741
beta*Delta Clampato       min=0.790855 max=3.344104 mean=2.351206
L(v) + beta*Delta         min=-2.685244 max=-0.131995 mean=-1.124893
L(v) + beta*Delta Clampato min=-2.685244 max=-0.131995 mean=-1.124893
      Epoca 02/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00329  Loss(Valid)=0.00604  |  RETE F: Loss(Train)=0.00172  Loss(Valid)=0.00189
    giro 2/5  beta=0.40  |  RMS a_z visitata 3.571 contro 2.375 nel dataset  |  |xi_applicato - xi*| = 0.0170  |  campioni 566864 -> 741026

DEBUG EPOCA 1
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=0.014957 max=0.358902 mean=0.248533
beta*Delta Intoccato      min=0.144281 max=3.462093 mean=2.397439
Delta Clampato [-1,1]     min=0.014957 max=0.358902 mean=0.248533
beta*Delta Clampato       min=0.144281 max=3.462093 mean=2.397439
L(v) + beta*Delta         min=-3.331818 max=-0.014006 mean=-1.078659
L(v) + beta*Delta Clampato min=-3.331818 max=-0.014006 mean=-1.078659
      Epoca 01/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00272  Loss(Valid)=0.00672  |  RETE F: Loss(Train)=0.00179  Loss(Valid)=0.00183

DEBUG EPOCA 2
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.008079 max=0.365024 mean=0.229895
beta*Delta Intoccato      min=-0.077938 max=3.521143 mean=2.217645
Delta Clampato [-1,1]     min=-0.008079 max=0.365024 mean=0.229895
beta*Delta Clampato       min=-0.077938 max=3.521143 mean=2.217645
L(v) + beta*Delta         min=-3.554036 max=0.045044 mean=-1.258453
L(v) + beta*Delta Clampato min=-3.554036 max=0.045044 mean=-1.258453
      Epoca 02/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00219  Loss(Valid)=0.00658  |  RETE F: Loss(Train)=0.00173  Loss(Valid)=0.00187
    giro 3/5  beta=0.16  |  RMS a_z visitata 3.547 contro 2.557 nel dataset  |  |xi_applicato - xi*| = 0.0217  |  campioni 566864 -> 828107

DEBUG EPOCA 1
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.062804 max=0.372412 mean=0.234732
beta*Delta Intoccato      min=-0.605827 max=3.592411 mean=2.264305
Delta Clampato [-1,1]     min=-0.062804 max=0.372412 mean=0.234732
beta*Delta Clampato       min=-0.605827 max=3.592411 mean=2.264305
L(v) + beta*Delta         min=-4.081925 max=0.116312 mean=-1.211794
L(v) + beta*Delta Clampato min=-4.081925 max=0.116312 mean=-1.211794
      Epoca 01/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00177  Loss(Valid)=0.00762  |  RETE F: Loss(Train)=0.00177  Loss(Valid)=0.00181

DEBUG EPOCA 2
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=-0.033335 max=0.375592 mean=0.227626
beta*Delta Intoccato      min=-0.321556 max=3.623095 mean=2.195763
Delta Clampato [-1,1]     min=-0.033335 max=0.375592 mean=0.227626
beta*Delta Clampato       min=-0.321556 max=3.623095 mean=2.195763
L(v) + beta*Delta         min=-3.797655 max=0.146996 mean=-1.280336
L(v) + beta*Delta Clampato min=-3.797655 max=0.146996 mean=-1.280336
      Epoca 02/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00151  Loss(Valid)=0.00683  |  RETE F: Loss(Train)=0.00171  Loss(Valid)=0.00178
    giro 4/5  beta=0.06  |  RMS a_z visitata 3.534 contro 2.689 nel dataset  |  |xi_applicato - xi*| = 0.0216  |  campioni 566864 -> 915188

DEBUG EPOCA 1
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=0.004087 max=0.372282 mean=0.247131
beta*Delta Intoccato      min=0.039423 max=3.591165 mean=2.383912
Delta Clampato [-1,1]     min=0.004087 max=0.372282 mean=0.247131
beta*Delta Clampato       min=0.039423 max=3.591165 mean=2.383912
L(v) + beta*Delta         min=-3.436675 max=0.115066 mean=-1.092187
L(v) + beta*Delta Clampato min=-3.436675 max=0.115066 mean=-1.092187
      Epoca 01/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00127  Loss(Valid)=0.00698  |  RETE F: Loss(Train)=0.00172  Loss(Valid)=0.00188

DEBUG EPOCA 2
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=0.049937 max=0.377436 mean=0.258589
beta*Delta Intoccato      min=0.481713 max=3.640874 mean=2.494442
Delta Clampato [-1,1]     min=0.049937 max=0.377436 mean=0.258589
beta*Delta Clampato       min=0.481713 max=3.640874 mean=2.494442
L(v) + beta*Delta         min=-2.994385 max=0.164775 mean=-0.981657
L(v) + beta*Delta Clampato min=-2.994385 max=0.164775 mean=-0.981657
      Epoca 02/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00111  Loss(Valid)=0.00667  |  RETE F: Loss(Train)=0.00167  Loss(Valid)=0.00180
    giro 5/5  beta=0.03  |  RMS a_z visitata 3.527 contro 2.790 nel dataset  |  |xi_applicato - xi*| = 0.0203  |  campioni 566864 -> 1002269

DEBUG EPOCA 1
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=0.029630 max=0.371484 mean=0.262729
beta*Delta Intoccato      min=0.285817 max=3.583461 mean=2.534374
Delta Clampato [-1,1]     min=0.029630 max=0.371484 mean=0.262729
beta*Delta Clampato       min=0.285817 max=3.583461 mean=2.534374
L(v) + beta*Delta         min=-3.190282 max=0.107363 mean=-0.941724
L(v) + beta*Delta Clampato min=-3.190282 max=0.107363 mean=-0.941724
      Epoca 01/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00097  Loss(Valid)=0.00733  |  RETE F: Loss(Train)=0.00166  Loss(Valid)=0.00190

DEBUG EPOCA 2
Coefficiente Alpha (beta buffer) = 9.6463
L(v)                      min=-3.476099 max=-3.476099 mean=-3.476099
Delta Intoccato           min=0.028206 max=0.366044 mean=0.256250
beta*Delta Intoccato      min=0.272087 max=3.530990 mean=2.471874
Delta Clampato [-1,1]     min=0.028206 max=0.366044 mean=0.256250
beta*Delta Clampato       min=0.272087 max=3.530990 mean=2.471874
L(v) + beta*Delta         min=-3.204012 max=0.054892 mean=-1.004225
L(v) + beta*Delta Clampato min=-3.204012 max=0.054892 mean=-1.004225
      Epoca 02/2  (alpha=9.65)  |  RETE xi: Loss(Train)=0.00086  Loss(Valid)=0.00774  |  RETE F: Loss(Train)=0.00160  Loss(Valid)=0.00188

==========================================================================================
 VALIDAZIONE — confronto fra valori APPRESI e valori CALCOLATI, su dati mai visti
==========================================================================================
 xi predetto / F predetta : APPRESO dalle reti (solo a_z passata + v)
 xi* / F* di riferimento  : CALCOLATO per ottimizzazione (conosce la strada e lo stato vero)

 [A] REGISTRATA: thessaloniki_19_11_2023.pkl  |  2525 s  |  v 0-55 km/h  |  RMS a_z 0.32 m/s^2  |  rugosita' equivalente classe ISO B
   xi(t)    -> R^2: +0.2035 | RMSE: 0.0809 | MAE: 0.0311 | dev.std target: 0.0906
   forza(t) -> R^2: +0.7504 | RMSE: 93.6 N | MAE: 61.6 N
   picchi   -> |F| max: target 1421 N, ML 919 N (-35%) | margine ML sul limite 39%
                saturazione a ±1500 N: target 0.00%, ML 0.00% | sui picchi (|F*|>50% del max) il ML sta -401 N

 [B] SINTETICHE, celle classe x velocita' DISGIUNTE dal training
     valid  1 ISO C(x0.62) 20-98 km/h [rampa]; valid  2 ISO E(x0.62) 19-24 km/h [costante]; valid  3 ISO A(x1.44) 11-86 km/h [sweep]; valid  4 ISO D(x1.08) 53-119 km/h [urbano]
   xi(t)    -> R^2: +0.9227 | RMSE: 0.0342 | MAE: 0.0223 | dev.std target: 0.1228
   forza(t) -> R^2: +0.8174 | RMSE: 105.9 N | MAE: 60.1 N
   picchi   -> |F| max: target 1500 N, ML 1500 N (+0%) | margine ML sul limite 0%
                saturazione a ±1500 N: target 0.49%, ML 0.17% | sui picchi (|F*|>50% del max) il ML sta -374 N
     ^ [B] e' il test severo: rugosita' x velocita' mai viste insieme.

 |forza| max prevista dal ML: 919.1 N  (limite 1500 N, margine 39%)

 COMFORT in CLOSED-LOOP — RMS accelerazione cassa
   dati: thessaloniki_19_11_2023.pkl, primi 60 s  (v 0-15 km/h, rugosita' equivalente classe ISO B)
   passiva  0.166 m/s^2
   ML       0.042 m/s^2   (+74.7% vs passiva)
   ottimo   0.038 m/s^2   (+76.9% vs passiva)  <- il tetto raggiungibile con le etichette

--------------------------------------------------------------------------
 ETICHETTE OTTIME: xi* in funzione della RUGOSITA' (RMS a_z) e della velocita'   [strade sintetiche]
--------------------------------------------------------------------------
RMS a_z        | v [km/h]  | xi* medio  | |forza| [N]
0.12-0.67      | 41.4      | 0.114      | 134
0.67-2.15      | 51.2      | 0.325      | 100
2.15-3.11      | 38.7      | 0.336      | 181
3.11-7.95      | 80.3      | 0.321      | 178
 TENUTA DI STRADA su thessaloniki_19_11_2023.pkl (60 s) — RMS deflessione pneumatico: passiva 0.24 mm -> ML 0.08 mm   (limite ammesso 4.66 mm)

 BILANCIO ENERGETICO — integrali su 60 s di thessaloniki_19_11_2023.pkl
   IPOTESI  damper    : damper IDRAULICO: il suo calore NON e' recuperabile ed e' escluso dal recupero
   IPOTESI  attuatore : attuatore ideale in trazione (eta_attuatore = 100%): il costo e' un LIMITE INFERIORE
   CONVENZIONE: le voci misurate sono MODULI (>=0), il verso e' nel nome. Il bilancio ha segno
   dal punto di vista della BATTERIA: <0 esce, >0 rientra. I joule scalano con la durata: non sono una potenza.
   MECCANICA  damper dissipato in calore nell'olio — PERSO, fuori bilancio :     3.9 J
   MECCANICA  attuatore, lavoro fatto sul veicolo (spinge)                :     7.7 J
   MECCANICA  attuatore, lavoro ricevuto dal veicolo (frena)              :    12.5 J
   ELETTRICO  ADDEBITO  = lavoro in spinta / eta_attuatore                :    -7.7 J   (esce dalla batteria)
   ELETTRICO  ACCREDITO = eta*(ATTUATORE assorbita)                      :    +7.5 J   (rientra in batteria)
   ELETTRICO  NETTO     = accredito - addebito                            :    -0.2 J  -> COSTO NETTO (esce dalla batteria piu' di quanto rientra)
   [sensibilita'] con le perdite di trazione al 60% come il generatore, il netto sarebbe -5.3 J
   [sensibilita'] con un damper ELETTROMAGNETICO al posto dell'idraulico il netto sarebbe +2.2 J
################################################################################################
&num;  DIAGNOSTICA FISICA — xi deve seguire xi* ottimo: dipende dalla RUGOSITA' (da a_z) e dalla velocita'
&num;  2026-08-01 00:12   |   vincolo strutturale: ATTIVO alpha=9.65   |   metodo_freq: velocita
################################################################################################

================================================================================================
 1) MAPPA xi(v, RMS a_z) — OPEN LOOP
================================================================================================
 DATI      : banca di finestre di a_z generate col modello PASSIVO su profili ISO
             8608 classi B,C,D,E a 29/58/86 km/h, poi RISCALATE al livello di RMS
             indicato in riga. La FORMA spettrale resta realistica, cambia solo
             l'ampiezza: e' un esperimento controllato, una variabile per volta.
 xi        : APPRESO dalla rete (uscita, non etichetta). Media su tutte le finestre.
 NB        : e' OPEN LOOP — a_z e' imposta dall'esterno, non prodotta dal controllo.
             Per il comportamento reale vedi la mappa closed-loop al punto 2.
    Righe = livello di a_z, colonne = velocita'. Quindi:
      lungo una RIGA (a_z fissa, v che cresce) xi deve SCENDERE da xi_max a xi_min;
      lungo una COLONNA (v fissa, a_z che cresce) xi deve restare QUASI COSTANTE.

  RMS a_z \ v [km/h]      14     22     29     36     43     50     58     65     72     79     86     94    101    108
  ---------------------------------------------------------------------------------------------------------------------
     6.00 m/s^2        0.374  0.365  0.360  0.357  0.356  0.356  0.357  0.357  0.357  0.357  0.356  0.354  0.353  0.351
     3.88 m/s^2        0.355  0.348  0.344  0.343  0.344  0.346  0.348  0.350  0.351  0.351  0.350  0.349  0.347  0.346
     2.51 m/s^2        0.330  0.326  0.324  0.326  0.329  0.333  0.338  0.341  0.343  0.344  0.344  0.343  0.342  0.340
     1.63 m/s^2        0.293  0.292  0.295  0.300  0.306  0.314  0.320  0.326  0.329  0.331  0.331  0.331  0.329  0.327
     1.05 m/s^2        0.235  0.242  0.250  0.261  0.272  0.284  0.294  0.301  0.306  0.308  0.308  0.306  0.304  0.301
     0.68 m/s^2        0.135  0.145  0.155  0.166  0.178  0.189  0.199  0.206  0.210  0.212  0.212  0.210  0.207  0.204
     0.44 m/s^2        0.085  0.086  0.087  0.089  0.091  0.093  0.095  0.097  0.098  0.098  0.099  0.099  0.099  0.099
     0.29 m/s^2        0.082  0.082  0.082  0.083  0.083  0.083  0.083  0.084  0.084  0.084  0.084  0.084  0.084  0.084
     0.18 m/s^2        0.082  0.082  0.082  0.082  0.082  0.083  0.083  0.083  0.083  0.083  0.083  0.083  0.083  0.083
     0.12 m/s^2        0.082  0.082  0.082  0.082  0.083  0.083  0.083  0.083  0.083  0.083  0.083  0.083  0.083  0.083
     0.08 m/s^2        0.082  0.082  0.082  0.083  0.083  0.083  0.084  0.084  0.084  0.084  0.084  0.084  0.084  0.084
     0.05 m/s^2        0.082  0.082  0.083  0.083  0.083  0.084  0.084  0.084  0.084  0.084  0.084  0.084  0.084  0.084
  ---------------------------------------------------------------------------------------------------------------------
  BASELINE √2 (solo v)   0.683  0.662  0.629  0.580  0.514  0.434  0.349  0.270  0.207  0.162  0.131  0.111  0.099  0.092
  [range damper: xi_min=0.082  xi_max=0.716]

================================================================================================
 2) xi MEDIO IN CLOSED-LOOP su strade ISO 8608 (anello di retroazione CHIUSO)
================================================================================================
 DATI      : 5 profili ISO 8608 (classi A, B, C, D, E) x 5 velocita' costanti, 12 s ciascuno,
             generati come le strade di TRAINING (ostacoli, pendenze, spettro ISO).
             Ogni cella e' la MEDIA di 5 realizzazioni con semi diversi:
             con una sola strada per cella si legge il sorteggio degli ostacoli.
             Scartato il 20% iniziale (warm-up: la finestra parte piena di zeri).
 xi rete   : APPRESO — la rete legge la propria a_z controllata e comanda.
 xi* target: CALCOLATO su QUELLA strada per ottimizzazione vincolata (conosce z_r).
  classe \ v [km/h]         22       40       58       76       97
  ----------------------------------------------------------------
  A (liscia)           0.083    0.098    0.089    0.124    0.125
  B                    0.083    0.095    0.183    0.281    0.283
  C                    0.108    0.257    0.327    0.329    0.309
  D                    0.294    0.325    0.338    0.331    0.335
  E (sconnessa)        0.340    0.348    0.353    0.351    0.358
  ----------------------------------------------------------------

  TARGET xi* per classe (l'ottimo su QUELLA strada: cambia riga per riga):
    A:    0.082   0.101   0.094   0.134   0.159
    B:    0.083   0.123   0.247   0.195   0.255
    C:    0.100   0.227   0.300   0.330   0.322
    D:    0.212   0.325   0.339   0.333   0.348
    E:    0.345   0.348   0.363   0.359   0.368
  baseline √2 (solo v):    0.662   0.549   0.349   0.182   0.105

  RMS a_z closed-loop [m/s^2] (per capire quanto e' fuori distribuzione):
    A:     0.19    0.31    0.34    0.54    0.75
    B:     0.24    0.51    1.07    1.04    1.69
    C:     0.52    1.72    1.54    2.05    3.07
    D:     1.79    2.61    3.12    3.97    4.42
    E:     3.49    4.85    6.13    7.01    8.00

  Errore medio  |xi_rete - xi*| = 0.019
  Errore MASSIMO                = 0.086 (classe B, 76 km/h: xi=0.281 contro xi*=0.195)

  RUMORE DEL TARGET (da 5 realizzazioni della stessa cella):
    scarto tipo fra realizzazioni     : 0.035   (max 0.122)
    incertezza sulla MEDIA riportata  : 0.016   <- barra d'errore delle celle qui sopra
    celle con errore oltre 2 barre    : 5 su 25   (le altre sono compatibili col rumore)
    bias medio con segno              : +0.002   (rete piu RIGIDA del target)

  Escursione di xi* fra classe A ed E, a parita' di velocita' : 0.243
  Escursione prodotta dalla RETE                              : 0.247
  (se la prima e' grande e la seconda ~0, la rete sta ignorando la strada;
   se la prima e' ~0, e' il TARGET a non dipendere dalla strada)

================================================================================================
 3) DISTRIBUZIONE DATASET (TRAINING) — dove vivono i campioni nel piano (v, a_z)
================================================================================================
 DATI      : 50 tracce di TRAINING (2 reali registrate + 48 sintetiche ISO 8608), finestre da 2 s
 xi        : ETICHETTA calcolata (target), non uscita della rete
  campioni analizzati: 150000 (sottocampionati da 566864)
  v     [km/h] : min    7.2  mediana   37.7  max  107.7
  RMS a_z      : min  0.053  mediana  0.343  max 13.329
  xi target    : min  0.082  mediana  0.082  max  0.716

  corr(RMS a_z, xi)          = +0.630   <- il SEGNALE VERO (xi* dipende dalla rugosita')
  Spearman(RMS a_z, xi)      = +0.741
  corr(v, xi)                = +0.527   <- informazione COMPLEMENTARE: serve a convertire a_z in rugosita'
  corr(RMS a_z, v)           = +0.413   <- quanto a_z e v sono confuse fra loro
  corr parziale (a v fissa)  = +0.532   <- quanto conta a_z A PARITA' di velocita' (deve essere grande)
  copertura griglia (v x a_z)= 94%      <- alta = disegno fattoriale, bassa = diagonale

  Tabella incrociata: xi target medio per bin di velocita' e di RMS a_z
  RMS \ v [km/h]         14       25       37       57       89
  -------------------------------------------------------------
            7.38      0.216    0.248    0.329    0.324    0.350
            0.94      0.145    0.153    0.185    0.270    0.319
            0.36      0.085    0.090    0.094    0.119    0.135
            0.21      0.083    0.083    0.089    0.091    0.096
            0.11      0.082    0.082    0.083    0.084    0.087
  ('--' = cella VUOTA: combinazione mai vista in addestramento; se le celle piene
   stanno su una diagonale, velocita' e accelerazione NON sono separabili)

================================================================================================
 4) INDICI DI SANITA' FISICA   [target = xi* ottimo]
================================================================================================
  [    ] quota di xi spiegata da a_z  = 0.939   (ora NON deve essere 0: se lo e', la rete ignora la strada)
          variazione di xi lungo a_z  = 0.1177
          variazione di xi lungo v    = 0.0076
  [OK  ] errore closed-loop |xi - xi*| = 0.0188   (e' l'indicatore che conta ora)
  [OK  ] escursione con la strada: rete 0.247 contro target 0.243  (102% del necessario)
  [    ] frazione monotona in v       = 26.3%  (non piu' un requisito: la tenuta puo' chiedere di irrigidire)
  [    ] distanza dal baseline √2     = 0.2150   (se ~0 la correzione appresa non sta facendo nulla)

  Sensibilita' d(xi)/d(log10 RMS a_z), crossover r=√2 a 53 km/h:
        sotto il crossover : +0.1260
        sopra il crossover : +0.1209
        (sopra r=√2 una sensibilita' POSITIVA e' un anello di retroazione
         positivo, ma ora puo' essere legittimo: se xi* stesso sale con la
         rugosita', l'anello e' voluto. Va giudicato contro xi*, non da solo.)

  Figura diagnostica salvata: diagnostica_xi.png
  Report testuale salvato:    diagnostica_report.txt
    [ok] Figura salvata: risultati_sospensione.png
    [ok] Figura efficienza salvata: efficienza_sospensione.png

[3] Animazione: tutti i controllori sulla stessa strada demo...
    (la strada demo e' severa — equivale a una classe E ISO 8608 — quindi xi DEVE muoversi parecchio)
    Controllori simulati sulla stessa strada demo (8 s):
                          xi  comfort Wk  gomma[mm]  margine
      ML       0.097-0.390        1.86       4.56     0.98
      ottimo   0.082-0.426        1.81       4.61     0.99
      sqrt2    0.120-0.606        1.98       5.87     1.26
      passivo  0.307 (fisso)        1.98       4.50     0.96
      (margine > 1 = il vincolo di distacco ruota e' violato)
    scarto medio |xi_ML - xi*| sulla demo: 0.029
</pre>

---



<div align="center">
    <img src="img_Ottimo/efficienza_sospensione.png" style="width: 85%; max-width: 900px; height: auto; margin-bottom: 20px;">
    <br>
    <img src="img_Ottimo/risultati_sospensione.png" style="width: 85%; max-width: 900px; height: auto; margin-bottom: 20px;">
    <br>
    <img src="img_Ottimo/diagnostica_xi.png" style="width: 85%; max-width: 900px; height: auto; margin-bottom: 20px">
</div>

<div align="center">
    <video width="100%" controls style="max-width: 900px; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
        <source src="img_Ottimo/confronto_sospensione.mp4" type="video/mp4">
    </video>
</div>

---

## 6 — Analisi Finale del Modello post-processo

Dopo aver addestrato la rete con l'algoritmo DAgger, la validazione avviene confrontando i valori appresi (in base alla sola cinematica e accelerazione misurabile) con i valori calcolati dall'ottimizzatore (che agisce da oracolo, conoscendo l'esatto profilo stradale $z_r$). 

I test sono stati condotti su due domini: un tracciato reale registrato (Thessaloniki) e un set di strade sintetiche severissime, con combinazioni di classe ISO e velocità volutamente disgiunte da quelle viste in fase di training.



### 6.1 Accuratezza Predittiva

La tabella seguente riassume le metriche di regressione per il coefficiente di smorzamento $\xi$ e la forza attiva $F$:

| Dataset | $\xi$ ($R^2$) | $\xi$ (RMSE) | Forza ($R^2$) | Forza (RMSE) | Forza Max Target | Forza Max ML |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Reale (Thessaloniki)** | +0.2035 | 0.0809 | +0.7504 | 93.6 N | 1421 N | 919 N (-35%) |
| **Sintetico (Out-of-distribution)** | +0.9227 | 0.0342 | +0.8174 | 105.9 N | 1500 N | 1500 N (0%) |

**Analisi:** Sul dataset sintetico severo (il vero banco di prova per la generalizzazione), la rete dimostra un'eccezionale capacità di replicare la policy dell'esperto ($R^2$ di 0.92 per lo smorzamento). Sul tracciato reale a bassa velocità, la rete adotta un approccio più conservativo sui picchi di forza (mantenendo un margine del 39% sul limite fisico di 1500 N), evitando del tutto la saturazione.

---



### 6.2 Metriche di Ride Comfort e Tenuta di Strada (Closed-Loop)

Il test in anello chiuso sui primi 60 secondi del tracciato reale dimostra i reali benefici della transizione al Deep Learning rispetto alla configurazione puramente passiva. *Anello chiuso* qui significa che le due reti, già addestrate e con i pesi congelati, pilotano davvero la simulazione RK4 del quarter-car: la $a_z(t)$ letta a ogni passo è la conseguenza della loro decisione al passo precedente, non un dato prelevato da un file (vedi §4.1 per la distinzione fra questo caso e l'addestramento, che invece è open-loop). Nessun peso viene aggiornato in questa fase: è una misura, non un training.

*   **Comfort (RMS accelerazione di cassa $a_z$):**
    *   Sospensione Passiva: 0.166 m/s²
    *   **Rete Neurale (ML): 0.042 m/s² (+74.7% di miglioramento)**
    *   Limite Teorico (Esperto): 0.038 m/s² (+76.9%)
    *   *Nota: La rete satura quasi completamente l'efficienza teorica del sistema ottimale.*
*   **Road Holding (RMS deflessione pneumatico):**
    *   La deflessione passa da 0.24 mm (passiva) a **0.08 mm** (ML), rimanendo ampiamente al di sotto del limite di sicurezza ammesso (4.66 mm).

---



### 6.3 Bilancio Energetico

L'integrazione del sistema semi-attivo e attivo comporta flussi di energia. Valutando il sistema dal punto di vista della batteria (dove i valori negativi indicano energia consumata e i positivi energia recuperata), il bilancio sui 60 secondi di test è il seguente:

| Voce di Bilancio | Valore (Joule) | Descrizione |
| :--- | :--- | :--- |
| **Lavoro in Spinta (Attuatore)** | 7.7 J | Energia meccanica erogata al veicolo |
| **Lavoro in Freno (Attuatore)** | 12.5 J | Energia meccanica ricevuta dal veicolo |
| **Addebito Elettrico** | -7.7 J | Esce dalla batteria (assumendo $\eta = 100\%$) |
| **Accredito Elettrico** | +7.5 J | Rientra in batteria tramite recupero |
| **Costo Elettrico Netto** | **-0.2 J** | Bilancio quasi neutrale |

*Nota sulle sensibilità:* Il calore dissipato dal damper idraulico (3.9 J) è attualmente perso. Sostituendo il damper con uno elettromagnetico, il sistema diventerebbe energeticamente attivo, generando un *surplus* netto stimato di **+2.2 J**.

---



### 6.4 Diagnostica Fisica: Il superamento del limite cinematico

L'analisi della mappa $\xi$ in closed-loop certifica il successo del cambio di paradigma. Il modello precedente, basato sulla trasmissibilità e su un singolo tono, faceva dipendere la strategia quasi esclusivamente dalla velocità (crossover a $\sqrt{2}$). 

La diagnostica attuale dimostra che:

1.  **Dipendenza dalla rugosità:** La rete ha imparato che $\xi^*$ dipende fortemente dall'accelerazione RMS $a_z$ (correlazione di +0.630). Lungo i profili stradali, a parità di velocità, la rete esplora un'escursione di smorzamento di 0.247, che ricalca quasi perfettamente l'escursione necessaria calcolata dall'oracolo (0.243).
2.  **Errore minimo:** L'errore medio assoluto $|\xi_{rete} - \xi^*|$ in closed-loop è limitato a soli **0.019**.
3.  **Sicurezza (Vincoli strutturali):** Nella simulazione su strada demo classe E (estremamente severa), l'euristica basata su $\sqrt{2}$ fallisce, superando il limite di distacco ruota (margine 1.26). La rete neurale, invece, adatta rigidamente lo smorzamento e mantiene il contatto (margine 0.98), garantendo la sicurezza del veicolo.  

## 7 — Conclusioni e Sviluppi Futuri

Il presente lavoro ha affrontato la riprogettazione del sistema di controllo per le sospensioni ibride (attive/semi-attive) del veicolo Peugeot 207, superando i vincoli analitici tradizionali mediante un approccio guidato dai dati (*Data-Driven*).



### 7.1 Sintesi dei Risultati

*   **Superamento delle Euristiche Analitiche:** La transizione dal modello euristico basato sulla trasmissibilità mono-tono ($\sqrt{2}$) a una policy basata su Deep Learning ha eliminato le criticità legate alle *race condition* cinematiche e all'effetto "oracolo". La rete neurale ha dimostrato di saper interpretare correttamente la natura a banda larga (*broadband*) delle sollecitazioni stradali reali anche in assenza di informazioni sulla lunghezza d'onda del manto stradale preciso $\lambda_c(t)$.
*   **Robustezza e Generalizzazione con DAgger:** L'integrazione dell'algoritmo di *Imitation Learning* (DAgger) ha risolto il problema del *covariate shift*. La rete ha imparato a correggere autonomamente le deviazioni dagli stati ideali, garantendo stabilità ed elevata accuratezza predittiva ($R^2 > 0.92$ su profili stradali mai visti in addestramento).
*   **Prestazioni Dinamiche e Sicurezza:** Il controllo in anello chiuso ha portato a una riduzione del **74.7% dell'accelerazione verticale di cassa** ($a_z(t)$) rispetto alla configurazione passiva, saturando quasi integralmente le prestazioni del modello teorico ottimo (76.9%). Allo stesso tempo, la deflessione del pneumatico è stata ridotta a soli **0.08 mm**, garantendo un margine di tenuta di strada elevato anche su fondi ad alta severità (ISO E).
*   **Efficienza Energetica:** L'analisi del bilancio energetico ha mostrato un costo elettrico netto quasi nullo (-0.2 J su 60 s), assumendo erogazione ideale in scarica ($\eta=100\%$) e un recupero in frenata al 60% di efficienza (7.5 J recuperati sui 12.5 J di lavoro frenante disponibile).

---



### 7.2 Limiti Attuali dello Studio

Per completezza, i risultati vanno letti con questi limiti in mente:

*   **Un solo tracciato reale di validazione:** le metriche su dati reali (Thessaloniki) vengono da un'unica traccia mai vista, non da una media su più semi/tracciate indipendenti — non è disponibile una stima della varianza di quelle metriche, solo il valore puntuale.
*   **Quarter-car, non full-car:** il modello isola un solo angolo del veicolo (§1.3), quindi non cattura beccheggio, rollio né le interazioni fra i quattro angoli che un controllore reale dovrebbe gestire insieme.
*   **Dipendenza dai dati sintetici:** la maggioranza del dataset di training è generata secondo ISO 8608 e non da strade realmente percorse; per quanto l'augmentation sia fisicamente motivata (§2.5), resta un modello della realtà, non la realtà.
*   **Nessuna validazione SiL/HiL:** la latenza e il comportamento su hardware ECU reale (vibrazioni, temperatura, rumore elettrico) non sono stati misurati, solo stimati per ordine di grandezza (si veda il punto 2 sotto).

---



### 7.3 Prospettive e Sviluppi Futuri

1.  **Sensoristica e Stimatori di Stato:** Estendere l'architettura per stimare in tempo reale il parametro di lunghezza d'onda dinamico $\lambda_c(t)$ tramite filtri di Kalman non lineari (EKF/UKF) o reti ricorsive, compensando l'assenza di sensoristica per quella variabile.
2.  **Validazione Software-in-the-Loop (SiL) e Hardware-in-the-Loop (HiL):** Testare il modello appreso su piattaforme di simulazione ad alta fedeltà (es. CarSim) prima della migrazione su centralina di controllo ECU target per la valutazione dei tempi di latenza in ambiente *real-time*.
3.  **Co-Design Attuatore-Controllo:** Ottimizzare congiuntamente le mappe di rendimento elettromeccanico dell'attuatore e le uscite della rete neurale per massimizzare il recupero energetico senza penalizzare il *Ride Comfort*, in modo tale da poter effettivamente realizzare meccanismi rigenerativi che possono parzialmente sostenersi durante la marcia.

## Riferimenti Bibliografici

Elenco delle fonti primarie (algoritmi, standard, filtri) richiamate nel testo ed utilizzate per scrivere il codice:

**Controllo e vibrazioni**

* Karnopp, D., Crosby, M. J., Harwood, R. A. (1974). *Vibration Control Using Semi-Active Force Generators*. Journal of Engineering for Industry, ASME. — origine del controllo **Skyhook**.
* Kalman, R. E. (1960). *Contributions to the Theory of Optimal Control*. Boletín de la Sociedad Matemática Mexicana. — base teorica del regolatore **LQR**.
* Richalet, J., Rault, A., Testud, J. L., Papon, J. (1978). *Model Predictive Heuristic Control: Applications to Industrial Processes*. Automatica. — Cutler, C. R., Ramaker, B. L. (1980). *Dynamic Matrix Control*. AIChE. — origini industriali dell'**MPC**.
* Luenberger, D. G. (1964). *Observing the State of a Linear System*. IEEE Transactions on Military Electronics. — Luenberger, D. G. (1971). *An Introduction to Observers*. IEEE Transactions on Automatic Control. — **osservatore di Luenberger**.
* Rugh, W. J., Shamma, J. S. (2000). *Research on Gain Scheduling*. Automatica. — rassegna sul **gain scheduling**.
* Den Hartog, J. P. (1956). *Mechanical Vibrations* (4ª ed.). McGraw-Hill. — Rao, S. S. (2011). *Mechanical Vibrations* (5ª ed.). Pearson. — teoria classica della **trasmissibilità** e del punto fisso $r=\sqrt{2}$.

**Standard e normative**

* ISO 8608:2016. *Mechanical vibration — Road surface profiles — Reporting of measured data*. International Organization for Standardization. — classificazione A–H della rugosità stradale usata per generare le strade sintetiche.
* ISO 2631-1:1997. *Mechanical vibration and shock — Evaluation of human exposure to whole-body vibration*. International Organization for Standardization. — filtro di ponderazione $W_k$ usato nella funzione di costo per il comfort.

**Filtraggio e stima**

* Kálmán, R. E. (1960). *A New Approach to Linear Filtering and Prediction Problems*. Journal of Basic Engineering. — **filtro di Kalman**, usato per stimare $\dot z_s$.
* Hampel, F. R. (1974). *The Influence Curve and its Role in Robust Estimation*. Journal of the American Statistical Association. — **identificatore di Hampel**, usato per la rimozione robusta degli outlier nei segnali IMU.
* Butterworth, S. (1930). *On the Theory of Filter Amplifiers*. Experimental Wireless & the Wireless Engineer. — **filtro di Butterworth** usato per l'estrazione delle bande di frequenza.
* Runge, C. (1895). *Über die numerische Auflösung von Differentialgleichungen*. Mathematische Annalen. — Kutta, W. (1901). *Beitrag zur näherungsweisen Integration totaler Differentialgleichungen*. Zeitschrift für Mathematik und Physik. — metodo **Runge-Kutta (RK4)**.

**Ottimizzazione e apprendimento profondo**

* Kingma, D. P., Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR. — Loshchilov, I., Hutter, F. (2019). *Decoupled Weight Decay Regularization*. ICLR. — ottimizzatore **AdamW**.
* Hendrycks, D., Gimpel, K. (2016). *Gaussian Error Linear Units (GELUs)*. arXiv:1606.08415. — attivazione **GELU**.
* Ioffe, S., Szegedy, C. (2015). *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*. ICML. — **BatchNorm**.
* Srivastava, N. et al. (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. JMLR. — **Dropout**.
* Cortes, C., Vapnik, V. (1995). *Support-Vector Networks*. Machine Learning. — origine della **hinge loss** usata nel termine di penalità.
* Bai, S., Kolter, J. Z., Koltun, V. (2018). *An Empirical Evaluation of Generic Convolutional and Recurrent Networks for Sequence Modeling*. arXiv:1803.01271. — **TCN**.
* van den Oord, A. et al. (2016). *WaveNet: A Generative Model for Raw Audio*. arXiv:1609.03499. — convoluzioni causali dilatate alla base delle TCN.
* Caruana, R. (1997). *Multitask Learning*. Machine Learning, Springer. — Ruder, S. (2017). *An Overview of Multi-Task Learning in Deep Neural Networks*. arXiv:1706.05098. — già citati nel testo, framework della loss multi-task.

**Imitation learning**

* Ross, S., Gordon, G. J., Bagnell, J. A. (2011). *A Reduction of Imitation Learning and Structured Prediction to No-Regret Online Learning*. AISTATS. — algoritmo **DAgger**, già citato nel testo.

**Algoritmi discussi per confronto (§1.3)**

* Chen, T., Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD.
* MacQueen, J. (1967). *Some Methods for Classification and Analysis of Multivariate Observations*. Proc. 5th Berkeley Symposium. — Lloyd, S. (1982). *Least Squares Quantization in PCM*. IEEE Transactions on Information Theory. — algoritmo **K-Means**.
* Ester, M., Kriegel, H.-P., Sander, J., Xu, X. (1996). *A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise*. KDD. — **DBSCAN**.
* ONNX (Open Neural Network Exchange), Microsoft e Meta/Facebook (2017). Formato aperto per l'interscambio di modelli di deep learning, richiamato come possibile via di deployment su ECU.

---
